# 🎥 CCTV Footage Analysis — find what matters without watching hours of video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OfficialAshish/home-camera-analysis/blob/main/cctv-analysis.ipynb)

This notebook scans every recording in a Google Drive folder and produces a **reviewable, cited index of everything that happened**:

| What you get | Where |
|---|---|
| Every **motion event**: start/end clock time, the video file, and offset inside the video (HH:MM:SS) | `CCTV_Report.xlsx` → *Events* |
| **Snapshots** (annotated + raw), a contact sheet and a short **clip** for each event | `events/<camera>/<date>/<event-id>/` |
| **People & objects** tracked (YOLO26 + ByteTrack): confidence, how long they stayed, speed, restricted zones | *People & Objects* |
| **Faces**: detected, quality-scored, cropped and **grouped into identities across days** (P0001, P0002 …); known staff can be labelled | *Faces*, *Face Identities* |
| **Suspicion score 0–100** with plain-English reasons: after-hours activity, camera tampering, loitering, running, crowding, hidden face, hands up, restricted zone, weapon-like objects, AI scene match | *Events* (sortable) |
| **AI description** of each flagged event (Qwen3-VL vision-language model) | *Events* → *AI description* |
| **Daily highlight video** – every movement of the day from all cameras **in true time order** (time read from the file name or the DVR clock printed on the picture), original picture without added text, plus a timing table (reel time → camera, clock, video, offset) | `highlights/<year>/<MM-Mon>/<date>_highlights.mp4`, *Highlights* sheet |
| **Number plates** *(optional)* – plate detector + plate OCR, Indian formats, motion-deblur for fast vehicles, plate watchlist | *Number Plates* sheet |
| **Weekly workbooks** (`reports/2026/11-17-Oct-2026-W41.xlsx`) plus an all-time `CCTV_Report.xlsx` | `reports/` |
| **Unique faces** workbook: every distinct person with photo, first/last seen, days, cameras; **NEW faces highlighted** and alerted at the end of each run | `Unique_Faces.xlsx` |
| **Authorised-only cameras**: anyone who is not a known face there is flagged | *Events*, *Unauthorized alerts* |
| Activity **heatmaps**, timeline charts and an interactive **HTML dashboard** (Events / Faces / Highlights / Videos tabs) | output folder |
| **Search**: free text (*"man in a helmet"*) or **by photo** (find every visit of a person) | cells at the end |

**How to use:** `Runtime → Change runtime type → T4 GPU`, check the settings in **Step 3**, then `Runtime → Run all`.
Re-running is safe and incremental: finished videos **and duplicate copies of them** are skipped, only the affected days/weeks are rebuilt, and models are kept in Drive (`MODEL_CACHE_DIR`) so they download only once.

## Step 1 · GPU check & Google Drive

In [ ]:
#@title Check GPU and mount Google Drive
import subprocess, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "⚠️ No GPU – use Runtime › Change runtime type › T4 GPU (CPU works, but slowly)")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab – using local paths.")

## Step 2 · Install dependencies (~1–2 min)

In [ ]:
#@title Install packages
!pip -q install -U "ultralytics>=8.3.200" lap xlsxwriter "transformers>=4.57" accelerate sentencepiece protobuf rapidocr-onnxruntime fast-plate-ocr open-image-models psutil
!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true
import importlib, ultralytics, transformers, torch, cv2
print("ultralytics", ultralytics.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| opencv", cv2.__version__, "| CUDA", torch.cuda.is_available())

### Analysis code
The next cells write the `cctv_analysis` package to `/content` (same code as the GitHub repo, so the notebook is fully self-contained).
You don't need to read or edit them — collapse the section and move on.

In [ ]:
import os; os.makedirs('/content/cctv_analysis', exist_ok=True)

In [ ]:
%%writefile /content/cctv_analysis/__init__.py
"""CCTV footage analysis toolkit: motion events, people/object tracking, faces,
suspicious-activity scoring and reviewer-friendly Excel/HTML reports."""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/cctv_analysis/config.py
"""All tunable settings live here. Every field has a sensible default, so a run
only needs `input_dir` and `output_dir`."""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass, field, fields
from typing import Dict, List, Optional

VIDEO_EXTENSIONS = [
    ".mp4", ".avi", ".mkv", ".mov", ".m4v", ".dav", ".h264", ".264", ".h265",
    ".265", ".hevc", ".ts", ".mts", ".m2ts", ".flv", ".asf", ".wmv", ".3gp",
    ".mpg", ".mpeg", ".webm", ".vob", ".ps", ".mpeg4",
]

# Zero-shot prompts used by SigLIP-2 to tag keyframes. Prompts in the
# "suspicious" group raise the suspicion score.
DEFAULT_SUSPICIOUS_PROMPTS = [
    "a person hiding a product inside their clothes or pocket",
    "a person putting merchandise into a bag without paying",
    "a person reaching into the cash register or cash drawer",
    "a person taking money from a counter",
    "two people fighting or pushing each other",
    "a person threatening someone with a knife or gun",
    "a person wearing a mask or helmet covering their face indoors",
    "a person breaking or damaging property",
    "a person covering or tampering with the security camera",
    "a person climbing over the counter",
    "a person lying on the floor",
    "a person looking around nervously while touching products",
]
DEFAULT_NORMAL_PROMPTS = [
    "an empty shop with no people",
    "a customer browsing products on a shelf",
    "a customer paying at the counter",
    "a shopkeeper standing behind the counter",
    "people talking calmly in a shop",
    "a person cleaning the floor",
    "a delivery person carrying boxes",
    "a street outside a shop with vehicles passing",
]

DEFAULT_VLM_PROMPT = (
    "You are a security analyst reviewing CCTV footage from a retail shop. "
    "The image is a grid of frames from one motion event, in time order "
    "(left to right, top to bottom). Describe briefly what the people are "
    "doing. Then decide if anything looks suspicious (theft, concealing items, "
    "taking cash, violence, vandalism, tampering with the camera, entering a "
    "restricted area). Answer ONLY with JSON of the form "
    '{"description": "...", "suspicious": true/false, "category": "none|theft|'
    'cash_handling|violence|vandalism|tampering|intrusion|other", '
    '"confidence": 0.0-1.0, "reason": "..."}'
)


# Speed presets (field overrides). Used by the notebook, the CLI (--preset) and the desktop app.
PRESETS = {
    "fast":     dict(motion_fps=3, detect_fps=3, detector_model="yolo26n.pt", detector_fallbacks=["yolo11n.pt"],
                     use_pose=False, det_imgsz=640, detect_width=960, semantic_model="google/siglip2-base-patch16-224"),
    "balanced": dict(motion_fps=4, detect_fps=5, detector_model="yolo26s.pt", detector_fallbacks=["yolo11s.pt"],
                     use_pose=True, det_imgsz=960, detect_width=1280, semantic_model="google/siglip2-base-patch16-224"),
    "accurate": dict(motion_fps=5, detect_fps=8, detector_model="yolo26m.pt", detector_fallbacks=["yolo11m.pt"],
                     use_pose=True, pose_model="yolo26s-pose.pt", det_imgsz=1280, detect_width=1920,
                     semantic_model="google/siglip2-so400m-patch14-384", vlm_model="Qwen/Qwen3-VL-4B-Instruct"),
}


@dataclass
class Config:
    # ---------------------------------------------------------------- paths
    input_dir: str = "/content/drive/MyDrive/CCTV"
    output_dir: str = "/content/drive/MyDrive/CCTV_Analysis"
    work_dir: str = "/content/cctv_work"          # fast local scratch (DB, temp)
    recursive: bool = True
    extensions: List[str] = field(default_factory=lambda: list(VIDEO_EXTENSIONS))
    include_glob: str = ""                       # e.g. "*ch01*" to limit files
    exclude_glob: str = ""
    max_videos: int = 0                          # 0 = no limit
    only_newer_than_days: float = 0              # 0 = all files
    reprocess: bool = False                      # ignore the "already done" cache
    max_runtime_minutes: float = 0               # stop cleanly (and still build reports) after this; 0 = no limit
    known_faces_dir: str = ""                    # folder/<Name>/*.jpg (staff, owner)
    # Only analyse sub-folders whose name matches one of these (any depth), e.g.
    # ["6-oct", "12oct"]; date-like keywords also match "2026-10-06", "06Oct26", "Oct 6"...
    # Empty = every video under input_dir.
    folder_keywords: List[str] = field(default_factory=list)
    dedupe: bool = True                          # skip identical copies (content hash), even renamed/moved
    model_cache_dir: str = "/content/drive/MyDrive/CCTV_models"   # models saved here once, reused next runs

    # ------------------------------------------------------------ time info
    timezone: str = "Asia/Kolkata"
    # Order in which the recording start time is resolved (see timeline.py):
    # filename | osd (burnt-in DVR clock read by OCR) | metadata (container creation_time) | mtime (approx)
    time_sources: List[str] = field(
        default_factory=lambda: ["filename", "osd", "metadata", "mtime"])
    ocr_engine: str = "auto"                     # auto | rapidocr | tesseract | vlm | none  (reads the DVR clock)
    osd_date_order: str = "auto"                 # auto | ymd | dmy | mdy  (how the DVR prints dates)
    osd_samples: int = 3                         # clock readings per video that must agree
    osd_verify_events: bool = True               # re-read the clock at every event start (fixes gaps inside a file)
    osd_event_tolerance: float = 5.0             # seconds; a larger disagreement uses the event's own clock reading
    use_path_date_hint: bool = True              # a date in the folder/file name ('31Oct26') validates/fixes times
    # Extra regexes with named groups Y m d H M S (optional) tried before built-ins.
    filename_time_patterns: List[str] = field(default_factory=list)
    # Manual overrides: {"file name or relative path": "2026-10-11 09:30:00"}
    time_overrides: Dict[str, str] = field(default_factory=dict)
    camera_name_source: str = "auto"             # auto | folder | filename | none

    # ------------------------------------------------------------ shop rules
    open_hour: str = ""                          # "" = open 24h (no after-hours flag); e.g. "09:00"
    close_hour: str = ""
    closed_weekdays: List[int] = field(default_factory=list)  # 0=Mon .. 6=Sun
    # Restricted zones per camera, normalised polygon coords (0..1).
    # {"*": {"cash_counter": [[0.6,0.5],[1,0.5],[1,1],[0.6,1]]}, "ch02": {...}}
    zones: Dict[str, Dict[str, List[List[float]]]] = field(default_factory=dict)
    # Cameras where only known (enrolled) people are allowed, e.g. ["warehouse"] or ["*"].
    # Anyone else (unknown face or no face visible) is flagged "unauthorized_person".
    authorized_only_cameras: List[str] = field(default_factory=list)
    # Areas to ignore for motion (timestamp overlay, swaying plants, TV screens).
    ignore_regions: Dict[str, List[List[List[float]]]] = field(default_factory=dict)
    auto_mask_osd: bool = True                   # auto-detect burnt-in clock text

    # ---------------------------------------------------------- motion pass
    motion_fps: float = 4.0
    motion_width: int = 384
    motion_sensitivity: float = 0.5              # 0 (least) .. 1 (most sensitive)
    min_motion_area: float = 0.0015              # fraction of frame
    global_change_ratio: float = 0.65            # > this = lighting change, ignore
    event_pre_roll: float = 2.0                  # seconds added before motion
    event_post_roll: float = 3.0
    event_merge_gap: float = 6.0                 # merge events closer than this
    event_min_duration: float = 1.0
    event_max_duration: float = 600.0            # split very long events
    detect_tamper: bool = True

    # -------------------------------------------------------- detection pass
    device: str = "auto"                         # auto | cuda | cpu
    hw_decode: str = "auto"                      # auto | cuda | none  (NVDEC video decoding on Colab GPUs)
    detector_model: str = "yolo26s.pt"           # falls back to yolo11s.pt
    detector_fallbacks: List[str] = field(default_factory=lambda: ["yolo11s.pt", "yolov8s.pt"])
    pose_model: str = "yolo26n-pose.pt"
    use_pose: bool = True
    tracker: str = "bytetrack.yaml"
    detect_fps: float = 6.0
    detect_width: int = 1280
    det_conf: float = 0.30
    det_imgsz: int = 960
    # COCO classes that matter in a shop (None = everything).
    keep_classes: Optional[List[str]] = field(default_factory=lambda: [
        "person", "bicycle", "car", "motorcycle", "bus", "truck", "dog", "cat",
        "backpack", "umbrella", "handbag", "suitcase", "bottle", "cup", "knife",
        "scissors", "baseball bat", "cell phone", "laptop", "book", "chair",
    ])
    require_person_for_event: bool = False       # drop motion with no person/vehicle/animal

    # ----------------------------------------------------------------- faces
    use_faces: bool = True                       # detect + save face crops
    face_identities: bool = True                 # group faces into persistent identities (unique faces, P0001…)
    face_watchlist_dir: str = ""                 # folder/<Name>/*.jpg – people who raise a "watchlist_face" alert
    face_score_threshold: float = 0.75
    face_min_size: int = 20                      # px, in detection-resolution frame
    face_match_threshold: float = 0.42           # SFace cosine similarity
    known_face_threshold: float = 0.45
    faces_per_track: int = 3
    face_min_quality: float = 0.30               # below this a face is kept as evidence but not used for identity

    # --------------------------------------------------- number plates (ANPR)
    use_plates: bool = False                     # read vehicle number plates (slower; off by default)
    plate_enhance: bool = False                  # super-resolve / sharpen / multi-frame vote hard plates (slowest)
    plate_vlm_read: bool = False                 # ask the VLM when OCR cannot read a plate
    plate_detector: str = "auto"                 # auto | yolo-plate (open-image-models ONNX) | classic (OpenCV)
    plate_ocr: str = "auto"                      # auto | fast-plate-ocr | generic (ocr_engine)
    plate_region: str = "IN"                     # IN (Indian formats, incl. BH series) | any
    plate_min_vehicle_px: int = 48               # skip vehicles smaller than this (box height, detection frame)
    plate_reads_per_track: int = 6               # plate crops read per vehicle track
    plate_watchlist: List[str] = field(default_factory=list)   # plates that raise a "watchlist_plate" alert
    plate_watchlist_max_edits: int = 1           # tolerated OCR differences when matching the watchlist

    # ------------------------------------------------------ footfall counting
    # Virtual counting lines per camera, normalised coords: {"*": {"entrance": [[0.1,0.6],[0.9,0.6]]}}
    # People crossing a line are counted in (left→right of the line direction) / out.
    count_lines: Dict[str, Dict[str, List[List[float]]]] = field(default_factory=dict)

    # ------------------------------------------------- suspicious heuristics
    loiter_seconds: float = 90.0
    crowd_size: int = 4
    running_speed: float = 1.6                   # body-heights per second
    weapon_classes: List[str] = field(default_factory=lambda: ["knife", "scissors", "baseball bat"])
    hidden_face_seconds: float = 8.0
    suspicion_alert_threshold: float = 40.0      # score >= this = "REVIEW"

    # ------------------------------------------------------ semantic models
    use_semantic: bool = True
    semantic_model: str = "google/siglip2-base-patch16-224"
    suspicious_prompts: List[str] = field(default_factory=lambda: list(DEFAULT_SUSPICIOUS_PROMPTS))
    normal_prompts: List[str] = field(default_factory=lambda: list(DEFAULT_NORMAL_PROMPTS))
    semantic_alert_score: float = 0.10           # SigLIP sigmoid; also needs lift >= 0.6

    use_vlm: bool = True                         # AI text description of events (slow on CPU)
    vlm_provider: str = "transformers"           # transformers (local HF model) | openai (any OpenAI-compatible server)
    vlm_model: str = "Qwen/Qwen3-VL-2B-Instruct" # HF repo id, or the model name the server knows (e.g. "qwen3-vl:2b")
    vlm_base_url: str = ""                       # openai provider: e.g. http://127.0.0.1:11434/v1 (Ollama), :1234/v1 (LM Studio)
    vlm_api_key_env: str = "CCTV_VLM_API_KEY"    # name of the environment variable holding the API key (never stored)
    vlm_timeout: float = 120.0                   # seconds per request (openai provider)
    vlm_scope: str = "flagged"                   # flagged | people | all
    vlm_max_events: int = 300
    vlm_prompt: str = DEFAULT_VLM_PROMPT
    vlm_max_new_tokens: int = 180

    # --------------------------------------------------------------- outputs
    snapshots_per_event: int = 6
    save_raw_snapshots: bool = True
    snapshot_jpeg_quality: int = 88
    save_clips: bool = True
    clip_scope: str = "people"                   # all | people | flagged
    clip_max_seconds: float = 180.0
    clip_width: int = 960
    clip_overlay: bool = False                   # burn an event label into clips (off = untouched picture)
    excel_thumbnails: bool = True
    excel_max_thumbnails: int = 3000
    make_html: bool = True
    master_report: bool = True                   # all-time CCTV_Report.xlsx (in addition to weekly files)
    weekly_reports: bool = True                  # reports/<year>/11-17-Oct-2026-W41.xlsx
    week_start: str = "sunday"                   # sunday (US week numbers, W41 = 11-17 Oct 2026) | monday (ISO)
    make_highlight_reel: bool = True             # one reel per day with every movement of all cameras
    reel_scope: str = "all"                      # all | people | flagged
    reel_max_event_seconds: float = 0            # 0 = whole event; else cap each event's part of the reel
    reel_overlay: bool = False                   # burn clock/camera text into reels (off = original picture only)
    reel_skip_blank: bool = True                 # leave black/blank, nothing-detected motion out of reels
    reel_width: int = 1280
    reel_fps: int = 15
    drive_links: bool = True                     # resolve Google Drive file links in Colab

    def validate(self) -> List[str]:
        """Human-readable problems with this configuration (empty list = valid)."""
        from .ocr import ENGINES
        from .timeline import TIME_SOURCES
        p = []
        bad = [s for s in self.time_sources if s not in TIME_SOURCES]
        if bad:
            p.append(f"time_sources: unknown {bad}; valid: {list(TIME_SOURCES)}")
        choices = {"ocr_engine": ENGINES, "osd_date_order": ("auto", "ymd", "dmy", "mdy"),
                   "vlm_provider": ("transformers", "openai"), "vlm_scope": ("flagged", "people", "all"),
                   "clip_scope": ("all", "people", "flagged"), "reel_scope": ("all", "people", "flagged"),
                   "plate_detector": ("auto", "yolo-plate", "classic"),
                   "plate_ocr": ("auto", "fast-plate-ocr", "generic"), "plate_region": ("IN", "any"),
                   "week_start": ("sunday", "monday"), "camera_name_source": ("auto", "folder", "filename", "none"),
                   "hw_decode": ("auto", "cuda", "none")}
        for k, ok in choices.items():
            if getattr(self, k) not in ok:
                p.append(f"{k}: {getattr(self, k)!r} is not one of {list(ok)}")
        if self.use_vlm and self.vlm_provider == "openai" and not self.vlm_base_url.strip():
            p.append("vlm_base_url: NOT_CONFIGURED – required when vlm_provider='openai' (e.g. http://127.0.0.1:11434/v1)")
        for k in ("open_hour", "close_hour"):
            v = (getattr(self, k) or "").strip()
            if v and not re.fullmatch(r"([01]?\d|2[0-3]):[0-5]\d", v):
                p.append(f"{k}: {v!r} must look like 09:00")
        if bool((self.open_hour or "").strip()) != bool((self.close_hour or "").strip()):
            p.append("open_hour/close_hour: set both or neither")
        for k in ("motion_fps", "detect_fps", "reel_fps"):
            if not getattr(self, k) or getattr(self, k) <= 0:
                p.append(f"{k}: must be > 0")
        if not 0 <= self.motion_sensitivity <= 1:
            p.append("motion_sensitivity: must be between 0 and 1")
        return p

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        names = {f.name for f in fields(cls)}
        unknown = set(d) - names
        if unknown:
            raise ValueError(f"Unknown config keys: {sorted(unknown)}")
        return cls(**d)

    @classmethod
    def from_json(cls, path: str) -> "Config":
        with open(path, "r", encoding="utf-8") as fh:
            return cls.from_dict(json.load(fh))

    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as fh:
            json.dump(self.to_dict(), fh, indent=2)

In [ ]:
%%writefile /content/cctv_analysis/modelcache.py
"""Keeps every model (YOLO weights, face ONNX, Hugging Face models) in one
persistent folder – on Colab that is your Google Drive – so each model is
downloaded only once and reused by all later runs."""
from __future__ import annotations

import logging
import os
import shutil

log = logging.getLogger("cctv")
_ROOT = {"dir": os.path.expanduser("~/.cache/cctv_analysis")}


def init(cache_dir: str) -> str:
    """Use cache_dir if its parent exists (e.g. Drive mounted), else a local cache."""
    d = cache_dir
    if d:
        parent = os.path.dirname(os.path.abspath(d))
        if os.path.isdir(parent) or os.path.isdir(d):
            try:
                os.makedirs(d, exist_ok=True)
                _ROOT["dir"] = d
            except OSError as e:
                log.warning("model cache %s unusable (%s) – using local cache", d, e)
        else:
            log.warning("model cache parent %s not found – using local cache %s", parent, _ROOT["dir"])
    os.makedirs(_ROOT["dir"], exist_ok=True)
    # HF env vars help when transformers has not been imported yet; we also pass cache_dir explicitly.
    os.environ.setdefault("HF_HUB_CACHE", hf_dir())
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    log.info("Model cache: %s", _ROOT["dir"])
    return _ROOT["dir"]


def root() -> str:
    os.makedirs(_ROOT["dir"], exist_ok=True)
    return _ROOT["dir"]


def sub(name: str) -> str:
    d = os.path.join(root(), name)
    os.makedirs(d, exist_ok=True)
    return d


def hf_dir() -> str:
    return sub("huggingface")


def yolo_weights(name: str) -> str:
    """Return a local path to YOLO weights, downloading into the cache only once."""
    if not name or os.path.isabs(name) or os.sep in name:
        return name
    dst = os.path.join(sub("yolo"), name)
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000:
        return dst
    from ultralytics import YOLO
    YOLO(name)                                   # downloads into the working directory
    for cand in (name, os.path.join(os.getcwd(), name)):
        if os.path.exists(cand):
            try:
                shutil.copyfile(cand, dst + ".part")
                os.replace(dst + ".part", dst)
                log.info("cached %s -> %s", name, dst)
                return dst
            except OSError as e:
                log.warning("could not cache %s: %s", name, e)
                return cand
    return name

In [ ]:
%%writefile /content/cctv_analysis/video_io.py
"""Video discovery, probing, fast frame reading (ffmpeg pipe with OpenCV
fallback), recording-time inference, clip export and Google-Drive links."""
from __future__ import annotations

import fnmatch
import json
import os
import re
import shutil
import subprocess
import time
from typing import Iterator, List, Optional, Tuple

import cv2
import numpy as np

FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
HW_DECODE = "auto"          # auto | cuda | none   (set from Config.hw_decode)
_HW_STATE: dict = {}              # path-independent cache: {"cuda": True/False}


def _cuda_present() -> bool:
    return shutil.which("nvidia-smi") is not None or os.path.exists("/dev/nvidia0")


def _hw_args() -> List[str]:
    if HW_DECODE == "none" or not FFMPEG:
        return []
    if HW_DECODE == "auto" and not _cuda_present():
        return []
    if _HW_STATE.get("cuda") is False:
        return []
    return ["-hwaccel", "cuda"]

# --------------------------------------------------------------------- files

_MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}


def natural_key(s: str):
    """'clip2' < 'clip10' (numbers compared as numbers)."""
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", s)]


def _norm(t: str) -> str:
    return re.sub(r"[^a-z0-9]", "", t.lower())


def day_month(text: str):
    """(day, month) from things like '6-oct', '06Oct26', 'Oct 6', '2026-10-06', '06-10-2026', '20261006'."""
    t = text.lower()
    m = re.search(r"(?<!\d)(\d{1,2})\s*[-_. ]?\s*(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*", t)
    if m:
        return int(m.group(1)), _MONTHS[m.group(2)]
    m = re.search(r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s*[-_. ]?\s*(\d{1,2})(?!\d)", t)
    if m:
        return int(m.group(2)), _MONTHS[m.group(1)]
    m = re.search(r"(?<!\d)(20\d{2})[-_.]?([01]\d)[-_.]?([0-3]\d)(?!\d)", t)
    if m:
        return int(m.group(3)), int(m.group(2))
    m = re.search(r"(?<!\d)([0-3]?\d)[-_.]([01]?\d)[-_.](20\d{2}|\d{2})(?!\d)", t)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None


def folder_matches(name: str, keyword: str) -> bool:
    kw = keyword.strip()
    if not kw:
        return False
    n, k = _norm(name), _norm(kw)
    if k and k in n:
        i = n.find(k)
        # avoid '6oct' matching '16oct'
        if not (k[0].isdigit() and i > 0 and n[i - 1].isdigit()):
            return True
    dk, dn = day_month(kw), day_month(name)
    return bool(dk and dn and dk == dn)


def find_videos(cfg) -> List[str]:
    import logging
    log = logging.getLogger("cctv")
    exts = {e.lower() for e in cfg.extensions}
    out = []
    if os.path.isfile(cfg.input_dir):
        return [cfg.input_dir]
    if not os.path.isdir(cfg.input_dir):
        raise FileNotFoundError(f"input folder not found: {cfg.input_dir}")
    kws = [k for k in (cfg.folder_keywords or []) if k and k.strip()]
    cutoff = time.time() - cfg.only_newer_than_days * 86400 if cfg.only_newer_than_days else None
    out_abs = os.path.abspath(cfg.output_dir)
    seen_dirs = set()
    for root, dirs, files in os.walk(cfg.input_dir):
        dirs[:] = sorted((d for d in dirs if os.path.abspath(os.path.join(root, d)) != out_abs
                         and not d.startswith(".")), key=natural_key)
        rel_parts = [] if os.path.abspath(root) == os.path.abspath(cfg.input_dir) else \
            os.path.relpath(root, cfg.input_dir).split(os.sep)
        if rel_parts:
            seen_dirs.add(rel_parts[-1])
        if kws and not any(folder_matches(part, k) for part in rel_parts for k in kws):
            if not cfg.recursive:
                break
            continue
        for f in sorted(files, key=natural_key):
            if os.path.splitext(f)[1].lower() not in exts:
                continue
            p = os.path.join(root, f)
            if cfg.include_glob and not fnmatch.fnmatch(f, cfg.include_glob):
                continue
            if cfg.exclude_glob and fnmatch.fnmatch(f, cfg.exclude_glob):
                continue
            try:
                st = os.stat(p)
            except OSError:
                continue
            if st.st_size < 10_000:
                continue
            if cutoff and st.st_mtime < cutoff:
                continue
            out.append(p)
        if not cfg.recursive:
            break
    if kws and not out:
        log.warning("No sub-folder matched %s. Folders found: %s", kws, ", ".join(sorted(seen_dirs)[:60]))
    if cfg.max_videos:
        out = out[: cfg.max_videos]
    return out


def quick_hash(path: str, chunk: int = 2 << 20) -> str:
    """Content fingerprint of size + first/middle/last 2 MB (fast even on Drive);
    identical copies under other names/folders get the same value."""
    import hashlib
    size = os.path.getsize(path)
    h = hashlib.sha1(str(size).encode())
    with open(path, "rb") as fh:
        for off in sorted({0, max(0, size // 2 - chunk // 2), max(0, size - chunk)}):
            fh.seek(off)
            h.update(fh.read(chunk))
    return h.hexdigest()


def file_fingerprint(path: str) -> str:
    st = os.stat(path)
    return f"{st.st_size}-{int(st.st_mtime)}"

# ------------------------------------------------------------------- probing

# Raw elementary streams have no index or timestamps: ffmpeg cannot seek in them before decoding.
RAW_FORMATS = ("h264", "hevc")
# original path -> lossless MP4 remux of it (same packets, same timing; cctv_analysis.ingest). Frame
# reading decodes the proxy, which can seek; hashes and evidence cuts always use the original.
_PROXY: dict = {}


def register_proxy(original: str, proxy: str) -> None:
    if proxy and os.path.isfile(proxy):
        _PROXY[os.path.abspath(original)] = proxy


def decode_path(path: str) -> str:
    return _PROXY.get(os.path.abspath(path), path)


def is_raw(path: str) -> bool:
    a = _input_args(path)
    return bool(a) and a[1] in RAW_FORMATS


def seek_args(path: str, t: float, copy: bool = False, fps: float = 0.0):
    """(arguments before -i, arguments after -i) that start reading at `t` seconds.
    Containers seek on input (fast, and exact with decoding); raw streams cannot, so they seek on
    output (decode and discard up to t – exact, slower); a stream copy of a raw stream also needs
    generated timestamps (-fflags +genpts -r <fps>) or nothing could be cut."""
    t = max(0.0, float(t))
    if not is_raw(path):
        return (["-ss", f"{t:.3f}"] if t > 0 else []), []
    pre = ["-fflags", "+genpts", "-r", f"{fps or probe(path)['fps'] or 25:g}"] if copy else []
    return pre, (["-ss", f"{t:.3f}"] if t > 0 else [])


def _input_args(path: str) -> List[str]:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dav":
        return ["-f", "dhav"]          # Dahua container
    if ext in (".h264", ".264"):
        return ["-f", "h264"]
    if ext in (".h265", ".265", ".hevc"):
        return ["-f", "hevc"]
    return []


def _raw_timing(path: str):
    """(fps, frames) of a raw H.264/H.265 stream: frames = video packets (demux only, no
    decoding); fps from the timestamps the decoder gives the first frames (the stream's own VUI
    timing, else ffmpeg's 25 fps default – the same timing every later decode of the file uses)."""
    if not (FFPROBE and FFMPEG):
        return 0.0, 0
    try:
        r = subprocess.run([FFPROBE, "-v", "error", *_input_args(path), "-count_packets", "-select_streams", "v:0",
                            "-show_entries", "stream=nb_read_packets", "-of", "csv=p=0", path],
                           capture_output=True, text=True, timeout=1800)
        n = int((r.stdout.strip().splitlines() or ["0"])[0].strip(",") or 0)
        r = subprocess.run([FFMPEG, "-nostdin", "-v", "info", *_input_args(path), "-i", path, "-map", "0:v:0",
                            "-frames:v", "16", "-vf", "showinfo", "-f", "null", "-"],
                           capture_output=True, text=True, timeout=120)
        pts = [float(x) for x in re.findall(r"pts_time:([0-9.]+)", r.stderr)]
        steps = sorted(b - a for a, b in zip(pts, pts[1:]) if b > a)
        if not steps or not n:
            return 0.0, n
        fps = 1.0 / steps[len(steps) // 2]
        return (round(fps, 3) if 0 < fps < 241 else 0.0), n
    except (OSError, ValueError, subprocess.TimeoutExpired):
        return 0.0, 0


def probe(path: str) -> dict:
    info = {"duration": 0.0, "fps": 0.0, "width": 0, "height": 0, "codec": "",
            "creation_time": None, "nb_frames": 0, "format": ""}
    if FFPROBE:
        for extra in (_input_args(path), []):
            try:
                r = subprocess.run(
                    [FFPROBE, "-v", "error", *extra, "-print_format", "json",
                     "-show_format", "-show_streams", path],
                    capture_output=True, text=True, timeout=120)
                j = json.loads(r.stdout or "{}")
            except Exception:
                j = {}
            vs = [s for s in j.get("streams", []) if s.get("codec_type") == "video"]
            if not vs:
                continue
            s = vs[0]
            fmt = j.get("format", {})
            info["codec"] = s.get("codec_name", "")
            info["width"] = int(s.get("width") or 0)
            info["height"] = int(s.get("height") or 0)
            info["format"] = fmt.get("format_name", "")
            for key in ("avg_frame_rate", "r_frame_rate"):
                try:
                    n, d = s.get(key, "0/1").split("/")
                    v = float(n) / float(d) if float(d) else 0
                    if 0 < v < 241:
                        info["fps"] = v
                        break
                except Exception:
                    pass
            for src in (fmt, s):
                try:
                    if float(src.get("duration") or 0) > 0:
                        info["duration"] = float(src["duration"])
                        break
                except Exception:
                    pass
            info["nb_frames"] = int(s.get("nb_frames") or 0)
            tags = {**fmt.get("tags", {}), **s.get("tags", {})}
            info["creation_time"] = tags.get("creation_time")
            break
    if info["width"] and not info["duration"] and _input_args(path) and _input_args(path)[1] in ("h264", "hevc"):
        # Raw elementary streams (.264/.265 DVR dumps) have no container duration and their
        # probed frame rate is the demuxer's default; time them as the decoder will.
        fps, n = _raw_timing(path)
        if fps and n:
            info["fps"], info["nb_frames"], info["duration"] = fps, n, n / fps
    if not info["width"] or not info["duration"]:
        cap = cv2.VideoCapture(path)
        if cap.isOpened():
            info["width"] = info["width"] or int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            info["height"] = info["height"] or int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 0
            n = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
            info["fps"] = info["fps"] or (fps if 0 < fps < 241 else 0)
            if not info["duration"] and fps and n > 0:
                info["duration"] = n / fps
        cap.release()
    if not info["duration"] and FFMPEG:
        # last resort: decode-count (slow but reliable for raw streams)
        try:
            r = subprocess.run([FFMPEG, "-v", "error", *_input_args(path), "-i", path,
                                "-map", "0:v:0", "-c", "copy", "-f", "null", "-"],
                               capture_output=True, text=True, timeout=1800)
            m = re.findall(r"time=(\d+):(\d+):([\d.]+)", r.stderr)
            if m:
                h, mi, s = m[-1]
                info["duration"] = int(h) * 3600 + int(mi) * 60 + float(s)
        except Exception:
            pass
    return info

# ------------------------------------------------------------- frame reading

def iter_frames(path: str, fps: float, width: int, start: float = 0.0,
                duration: Optional[float] = None, gray: bool = False,
                src_size: Tuple[int, int] = (0, 0)) -> Iterator[Tuple[float, np.ndarray]]:
    """Yield (timestamp_seconds, frame) sampled at `fps`, resized to `width`."""
    sw, sh = src_size
    if width and sw and width >= sw:
        width = sw
    produced = False
    if FFMPEG and sw and sh:
        w = int(width or sw) // 2 * 2
        h = int(round(sh * w / sw)) // 2 * 2
        ch = 1 if gray else 3
        hw = _hw_args()
        cmd = [FFMPEG, "-nostdin", "-v", "error", *hw, "-threads", "0"]
        src = decode_path(path)
        pre, post = seek_args(src, start)
        cmd += [*pre, *_input_args(src), "-i", src, *post]
        if duration:
            cmd += ["-t", f"{duration:.3f}"]
        cmd += ["-map", "0:v:0", "-an", "-sn", "-vf",
                f"fps={fps},scale={w}:{h}:flags=area", "-pix_fmt",
                "gray" if gray else "bgr24", "-f", "rawvideo", "pipe:1"]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                                bufsize=w * h * ch * 4)
        nbytes = w * h * ch
        i = 0
        try:
            while True:
                buf = proc.stdout.read(nbytes)
                if len(buf) < nbytes:
                    if i == 0 and hw:
                        proc.kill()
                        prev = _HW_STATE.get("cuda")
                        _HW_STATE["cuda"] = False      # retry on CPU
                        got = 0
                        for item in iter_frames(path, fps, width, start, duration, gray, src_size):
                            got += 1
                            yield item
                        if not got:                    # empty range, not a GPU problem
                            if prev is None:
                                _HW_STATE.pop("cuda", None)
                            else:
                                _HW_STATE["cuda"] = prev
                        return
                    break
                if i == 0 and hw:
                    _HW_STATE["cuda"] = True
                frame = np.frombuffer(buf, np.uint8).reshape((h, w) if gray else (h, w, 3))
                produced = True
                yield start + i / fps, frame
                i += 1
        finally:
            proc.stdout.close()
            proc.kill()
            proc.wait()
    if produced:
        return
    # ---- OpenCV fallback
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return
    nfps = cap.get(cv2.CAP_PROP_FPS) or 25
    if not (0 < nfps < 241):
        nfps = 25
    if start > 0:
        cap.set(cv2.CAP_PROP_POS_MSEC, start * 1000)
    step = max(1, int(round(nfps / fps)))
    idx = 0
    end = start + duration if duration else None
    while True:
        ok = cap.grab()
        if not ok:
            break
        t = start + idx / nfps
        if end and t > end:
            break
        if idx % step == 0:
            ok, frame = cap.retrieve()
            if not ok:
                break
            if width and frame.shape[1] != width:
                h = int(round(frame.shape[0] * width / frame.shape[1]))
                frame = cv2.resize(frame, (width, h), interpolation=cv2.INTER_AREA)
            if gray:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            yield t, frame
        idx += 1
    cap.release()


def read_frame_at(path: str, t: float, width: int = 0, src_size=(0, 0)) -> Optional[np.ndarray]:
    """The frame shown at time t (first frame with pts >= t), resized to `width` (0 = full size).
    Accurate seek + '-frames:v 1' – an fps filter here would return the *last* frame of the
    first output interval (~0.5 s late), which mis-crops moving objects."""
    sw, sh = src_size
    if FFMPEG and sw and sh:
        w = int(min(width, sw) if width else sw) // 2 * 2
        h = int(round(sh * w / sw)) // 2 * 2
        src = decode_path(path)
        pre, post = seek_args(src, t)
        cmd = [FFMPEG, "-nostdin", "-v", "error", *pre, *_input_args(src), "-i", src, *post,
               "-map", "0:v:0", "-an", "-sn", "-frames:v", "1", "-vf", f"scale={w}:{h}:flags=area",
               "-pix_fmt", "bgr24", "-f", "rawvideo", "pipe:1"]
        try:
            r = subprocess.run(cmd, capture_output=True, timeout=120)
            if len(r.stdout) == w * h * 3:
                return np.frombuffer(r.stdout, np.uint8).reshape((h, w, 3)).copy()
        except Exception:
            pass
    for _, f in iter_frames(path, 1.0, width, start=max(0.0, t), duration=1.0, src_size=src_size):
        return f.copy()
    return None


def read_frames(path: str, t: float, n: int, src_size=(0, 0)) -> List[np.ndarray]:
    """`n` consecutive decoded frames at full size, the first being the frame shown at time t
    (no fps filter: every frame of the recording, none duplicated or dropped)."""
    sw, sh = src_size
    if not (FFMPEG and sw and sh) or n < 1:
        return []
    w, h = sw // 2 * 2, sh // 2 * 2
    src = decode_path(path)
    pre, post = seek_args(src, t)
    cmd = [FFMPEG, "-nostdin", "-v", "error", *pre, *_input_args(src), "-i", src, *post,
           "-map", "0:v:0", "-an", "-sn", "-frames:v", str(int(n)), "-vf", f"scale={w}:{h}:flags=area",
           "-pix_fmt", "bgr24", "-f", "rawvideo", "pipe:1"]
    try:
        r = subprocess.run(cmd, capture_output=True, timeout=300)
    except (OSError, subprocess.TimeoutExpired):
        return []
    size = w * h * 3
    return [np.frombuffer(r.stdout[i:i + size], np.uint8).reshape((h, w, 3)).copy()
            for i in range(0, len(r.stdout) - size + 1, size)]

# ------------------------------------------------------------- cameras

def _device_serial(base: str) -> str:
    """Camera serial prefix of cloud exports, e.g. 'G225087WSFW' in 'G225087WSFW-cloud-18056409.mp4'."""
    m = re.match(r"([A-Z0-9]{8,20})[-_]", base)
    if m and re.search(r"[A-Z]", m.group(1)) and re.search(r"\d", m.group(1)):
        return m.group(1)
    return ""


def camera_name(path: str, cfg) -> str:
    """Camera label: 'ch01'-style token in the file name, else the first sub-folder (unless that
    folder is a date such as '31Oct26New' or '2026-10-11'), else a device serial prefix, else 'cam'."""
    if cfg.camera_name_source == "none":
        return ""
    base = os.path.basename(path)
    m = re.search(r"(?i)(?:^|[^a-z])((?:ch|cam|camera|channel|d)[ _-]?\d{1,3})(?![0-9])", base)
    if cfg.camera_name_source in ("auto", "filename") and m:
        return re.sub(r"[ _-]", "", m.group(1)).lower()
    if cfg.camera_name_source in ("auto", "folder"):
        parent = os.path.dirname(os.path.abspath(path))
        if os.path.abspath(parent) != os.path.abspath(cfg.input_dir):
            rel = os.path.relpath(parent, cfg.input_dir)
            first = rel.split(os.sep)[0]
            if first and first != ".." and not re.fullmatch(r"[\d\-_.]+", first) and not day_month(first):
                return first
    if cfg.camera_name_source in ("auto", "filename"):
        serial = _device_serial(base)
        if serial:
            return serial
    return "cam"

# ------------------------------------------------------------- encoding

_ENCODER: dict = {}


def h264_args(crf: int = 23) -> List[str]:
    """Video encoder arguments for the available ffmpeg build: libx264 (GPL builds), else
    libopenh264 (LGPL builds, Cisco's BSD codec), else MPEG-4 Part 2 (plays in VLC, not browsers)."""
    if "args" not in _ENCODER:
        enc = ""
        try:
            r = subprocess.run([FFMPEG, "-hide_banner", "-encoders"], capture_output=True, text=True, timeout=30)
            enc = r.stdout
        except Exception:
            pass
        if " libx264 " in enc:
            _ENCODER["name"], _ENCODER["args"] = "libx264", ["-c:v", "libx264", "-preset", "veryfast"]
        elif " libopenh264 " in enc:
            _ENCODER["name"], _ENCODER["args"] = "libopenh264", ["-c:v", "libopenh264", "-allow_skip_frames", "1"]
        else:
            _ENCODER["name"], _ENCODER["args"] = "mpeg4", ["-c:v", "mpeg4"]
    args = list(_ENCODER["args"])
    if _ENCODER["name"] == "libx264":
        args += ["-crf", str(crf)]
    elif _ENCODER["name"] == "libopenh264":
        args += ["-b:v", "3M"]
    else:
        args += ["-q:v", "4"]
    return args + ["-pix_fmt", "yuv420p"]

# ---------------------------------------------------------------- clip export

def export_clip(path: str, start: float, duration: float, out_path: str,
                width: int = 960, label: str = "") -> bool:
    if not FFMPEG:
        return False
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    vf = f"scale='min({width},iw)':-2"
    if label:
        safe = re.sub(r"[^A-Za-z0-9 _:./-]", "", label).replace(":", r"\:")
        vf += (f",drawtext=text='{safe}':x=10:y=h-th-10:fontsize=18:fontcolor=white:"
               "box=1:boxcolor=black@0.5")
    src = decode_path(path)
    pre, post = seek_args(src, start)
    base = [FFMPEG, "-nostdin", "-y", "-v", "error", *pre, *_input_args(src), "-i", src, *post,
            "-t", f"{duration:.3f}", "-map", "0:v:0", "-an"]
    tail = [*h264_args(27), "-movflags", "+faststart", out_path]
    for filt in (vf, f"scale='min({width},iw)':-2"):     # retry without drawtext
        try:
            r = subprocess.run(base + ["-vf", filt] + tail, capture_output=True, timeout=900)
            if r.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
                return True
        except Exception:
            pass
    return False

# --------------------------------------------------------------- drive links

def drive_link(path: str) -> str:
    """Google-Drive web link for a file on a Colab-mounted Drive ('' if unknown)."""
    try:
        fid = os.getxattr(path, "user.drive.id").decode()
        if fid:
            return f"https://drive.google.com/file/d/{fid}/view"
    except Exception:
        pass
    return ""


def fmt_offset(sec: float) -> str:
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

In [ ]:
%%writefile /content/cctv_analysis/ocr.py
"""Text recognition providers used for the DVR clock (OSD) and number plates.

One interface, several engines – the first available one is used for ``auto``:

* ``rapidocr``  – RapidOCR (PaddleOCR models on ONNX Runtime, Apache-2.0); pip-installable,
                  works offline, CPU friendly. Recommended default.
* ``tesseract`` – the Tesseract command-line program, if installed.
* ``vlm``       – the configured vision-language model (slow, used only when asked).
* ``none``      – disabled.

Every engine returns a list of ``(text, confidence 0..1)`` lines.
"""
from __future__ import annotations

import logging
import shutil
import subprocess
from typing import Callable, List, Optional, Tuple

import cv2
import numpy as np

log = logging.getLogger("cctv")
ENGINES = ("auto", "rapidocr", "tesseract", "vlm", "none")
Lines = List[Tuple[str, float]]


class OCREngine:
    name = "none"

    def read(self, img_bgr: np.ndarray) -> Lines:  # pragma: no cover - interface
        return []

    def __bool__(self):
        return self.name != "none"


class NullOCR(OCREngine):
    name = "none"


class RapidOCREngine(OCREngine):
    name = "rapidocr"

    def __init__(self):
        from rapidocr_onnxruntime import RapidOCR
        self._r = RapidOCR()

    def read(self, img_bgr):
        res, _ = self._r(img_bgr)
        return [(str(x[1]), float(x[2])) for x in res or []]


class TesseractEngine(OCREngine):
    name = "tesseract"

    def __init__(self, binary: Optional[str] = None, psm: int = 6):
        self.bin = binary or shutil.which("tesseract")
        if not self.bin:
            raise RuntimeError("tesseract program not found on PATH")
        self.psm = psm

    def read(self, img_bgr):
        ok, buf = cv2.imencode(".png", img_bgr)
        if not ok:
            return []
        r = subprocess.run([self.bin, "stdin", "stdout", "--psm", str(self.psm)], input=buf.tobytes(),
                           capture_output=True, timeout=60)
        txt = r.stdout.decode("utf-8", "ignore")
        # tesseract's CLI gives no per-line confidence; 0.6 marks it as "unscored"
        return [(ln.strip(), 0.6) for ln in txt.splitlines() if ln.strip()]


class VLMOCR(OCREngine):
    name = "vlm"

    def __init__(self, vlm_getter: Callable):
        self._get = vlm_getter

    def read(self, img_bgr):
        vlm = self._get()
        if vlm is None:
            return []
        txt = vlm.ask(img_bgr, "Transcribe all text visible in this image exactly, one line per text item. "
                               "Reply with the text only.", 60)
        return [(ln.strip(), 0.5) for ln in txt.splitlines() if ln.strip()]


def _try(factory):
    try:
        return factory()
    except Exception as e:
        log.debug("OCR engine unavailable: %s", e)
        return None


def available_engines() -> dict:
    """{engine: True/False} for the system check (does not load the VLM)."""
    out = {}
    try:
        import importlib.util
        out["rapidocr"] = importlib.util.find_spec("rapidocr_onnxruntime") is not None
    except Exception:
        out["rapidocr"] = False
    out["tesseract"] = shutil.which("tesseract") is not None
    return out


_CACHE: dict = {}


def get_ocr(engine: str = "auto", vlm_getter: Optional[Callable] = None) -> OCREngine:
    """Return a ready engine (cached per process). Unknown names raise ValueError."""
    if engine not in ENGINES:
        raise ValueError(f"ocr_engine must be one of {ENGINES}, got {engine!r}")
    if engine == "none":
        return NullOCR()
    if engine == "vlm":
        return VLMOCR(vlm_getter) if vlm_getter else NullOCR()
    order = ["rapidocr", "tesseract"] if engine == "auto" else [engine]
    for name in order:
        if name in _CACHE:
            return _CACHE[name]
        eng = _try(RapidOCREngine if name == "rapidocr" else TesseractEngine)
        if eng is not None:
            _CACHE[name] = eng
            log.info("OCR engine: %s", name)
            return eng
    if engine == "auto" and vlm_getter is not None:
        return VLMOCR(vlm_getter)
    log.warning("No OCR engine available (install rapidocr-onnxruntime or tesseract) – "
                "the DVR clock cannot be read from frames")
    return NullOCR()

In [ ]:
%%writefile /content/cctv_analysis/timeline.py
"""Recording-time resolution – the single place that decides *when* a frame was recorded.

Sources, tried in ``Config.time_sources`` order (default filename → osd → metadata → mtime):

* ``filename`` – time encoded in the file name / path (Hikvision, Dahua, NVR exports …)
* ``osd``      – the DVR's burnt-in clock, read by OCR at several offsets; the readings must
                 agree with each other (clock(t) − t = constant) and with the folder date, so a
                 single misread digit cannot shift a whole video
* ``metadata`` – container ``creation_time``; rejected when it contradicts the folder date
                 (cloud/phone exports stamp the *download* time here)
* ``mtime``    – file modified time minus duration; always marked approximate

A date in the folder or file name (``31Oct26New``, ``2026-10-31``, ``31-10-2026``) is used as a
plausibility check and, for approximate sources, as the day.
"""
from __future__ import annotations

import datetime as dt
import logging
import os
import re
from typing import List, Optional, Tuple
from zoneinfo import ZoneInfo

from .video_io import read_frame_at

log = logging.getLogger("cctv")

TIME_SOURCES = ("filename", "osd", "metadata", "mtime")
# labels written to videos.time_source that count as exact recording times
EXACT_PREFIXES = ("override", "filename", "osd")

_MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}


def is_exact(time_source: Optional[str]) -> bool:
    return bool(time_source) and time_source.startswith(EXACT_PREFIXES)


def tz(name: str):
    try:
        return ZoneInfo(name)
    except Exception:
        return dt.timezone.utc

# ------------------------------------------------------------------ file names

_BUILTIN_PATTERNS = [
    # 20261011143000, 2026-10-11_14-30-00, 2026.10.11 14.30.00, 20261011T143000
    r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)[-_ T.]?(?P<H>[0-2]\d)[-_.:hH]?(?P<M>[0-5]\d)[-_.:mM]?(?P<S>[0-5]\d)",
    # 11-10-2026_14-30-00 (day first)
    r"(?P<d>[0-3]\d)[-_.](?P<m>[01]\d)[-_.](?P<Y>20\d{2})[-_ T.]?(?P<H>[0-2]\d)[-_.:]?(?P<M>[0-5]\d)[-_.:]?(?P<S>[0-5]\d)",
    # date only + hour-minute
    r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)[-_ T.]?(?P<H>[0-2]\d)[-_.:]?(?P<M>[0-5]\d)(?!\d)",
]
_DATE_ONLY = r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)(?!\d)"
_TIME_ONLY = r"(?<!\d)(?P<H>[0-2]\d)[.:_-](?P<M>[0-5]\d)[.:_-](?P<S>[0-5]\d)"   # Dahua 14.30.00-14.45.00


def _mk(g: dict, tzinfo) -> Optional[dt.datetime]:
    try:
        d = dt.datetime(int(g["Y"]), int(g["m"]), int(g["d"]), int(g.get("H") or 0),
                        int(g.get("M") or 0), int(g.get("S") or 0), tzinfo=tzinfo)
        if 2000 <= d.year <= 2100:
            return d
    except (ValueError, TypeError, KeyError):
        pass
    return None


def _rel(path: str, root: str) -> str:
    try:
        return os.path.relpath(path, root) if root and os.path.isdir(root) else path
    except ValueError:
        return path


def time_from_name(path: str, root: str, patterns: List[str], tzinfo) -> Optional[dt.datetime]:
    name = os.path.basename(path)
    rel = _rel(path, root)
    for pat in list(patterns) + _BUILTIN_PATTERNS:
        for text in (name, rel):
            m = re.search(pat, text)
            if m:
                d = _mk(m.groupdict(), tzinfo)
                if d:
                    return d
    # date somewhere in path + time in file name (Dahua / many NVR exports)
    md = re.search(_DATE_ONLY, rel)
    mt = re.search(_TIME_ONLY, name)
    if md and mt:
        return _mk({**md.groupdict(), **mt.groupdict()}, tzinfo)
    return None


def date_from_text(text: str) -> Optional[dt.date]:
    """A full calendar date in a folder/file name: 31Oct26, 31-Oct-2026, Oct31_2026, 2026-10-31,
    20261031, 31-10-2026, 31.10.26. Returns None when no unambiguous full date is present."""
    t = text.lower()
    cands = []
    for m in re.finditer(r"(?<!\d)([0-3]?\d)\s*[-_. ]?\s*(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"
                         r"\s*[-_. ]?\s*(20\d{2}|\d{2})(?!\d)", t):
        cands.append((m.group(3), _MONTHS[m.group(2)], m.group(1)))
    for m in re.finditer(r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s*[-_. ]?\s*([0-3]?\d)"
                         r"\s*[-_.,' ]+\s*(20\d{2})(?!\d)", t):
        cands.append((m.group(3), _MONTHS[m.group(1)], m.group(2)))
    for m in re.finditer(r"(?<!\d)(20\d{2})[-_.]?([01]\d)[-_.]?([0-3]\d)(?!\d)", t):
        cands.append((m.group(1), int(m.group(2)), m.group(3)))
    for m in re.finditer(r"(?<!\d)([0-3]?\d)[-_.]([01]?\d)[-_.](20\d{2}|\d{2})(?!\d)", t):
        cands.append((m.group(3), int(m.group(2)), m.group(1)))
    for y, mo, d in cands:
        y = int(y)
        y = y + 2000 if y < 100 else y
        try:
            return dt.date(y, int(mo), int(d))
        except ValueError:
            continue
    return None


def path_date_hint(path: str, root: str) -> Optional[dt.date]:
    """Date written in the file name or the nearest folder that carries one."""
    rel = _rel(path, root)
    parts = rel.replace("\\", "/").split("/")
    for part in [parts[-1]] + parts[-2::-1]:
        d = date_from_text(os.path.splitext(part)[0] if part == parts[-1] else part)
        if d:
            return d
    return None

# ------------------------------------------------------------ clock-text parsing

_DIGIT_FIX = str.maketrans({"O": "0", "o": "0", "D": "0", "Q": "0", "l": "1", "I": "1", "|": "1", "i": "1",
                            "Z": "2", "z": "2", "S": "5", "s": "5", "B": "8", "g": "9", "q": "9"})
_DIGIT_FIX_CHARS = set("OoDQlI|iZzSsBgq")
_MON_RE = "jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec"


def _fix_digits(text: str) -> str:
    """Repair OCR letter/digit confusions only where a letter sits inside a number
    ('2O26', '05:3B', 'O5:38') – month and weekday names ('31Oct', 'Sat') are kept."""
    out = list(text)
    n = len(text)
    for i, c in enumerate(text):
        if c not in _DIGIT_FIX_CHARS:
            continue
        prev = text[i - 1] if i else ""
        nxt = text[i + 1] if i + 1 < n else ""
        num_prev, num_next = prev.isdigit(), nxt.isdigit()
        sep_prev, sep_next = prev in ":-/." or prev == "", nxt in ":-/." or nxt == ""
        if (num_prev and (num_next or sep_next)) or (num_next and (num_prev or sep_prev or prev == " ")):
            out[i] = c.translate(_DIGIT_FIX)
    return "".join(out)


def _time_part(s: str):
    # no "not preceded by a digit" guard: OCR often glues the date to the time ('2026-10-3105:38:27')
    m = re.search(r"([0-2]?\d)\s*[:.]\s*([0-5]\d)\s*[:.]\s*([0-5]\d)(?!\d)(?:\s*([ap])\.?m)?", s, re.I)
    if not m:
        return None
    h, mi, se = int(m.group(1)), int(m.group(2)), int(m.group(3))
    ap = (m.group(4) or "").lower()
    if ap == "p" and h < 12:
        h += 12
    elif ap == "a" and h == 12:
        h = 0
    if h > 23:
        return None
    return h, mi, se, m.start(), m.end()


def parse_clock_text(text: str, order: str = "auto", hint: Optional[dt.date] = None) -> Optional[dt.datetime]:
    """Parse a DVR clock such as '2026-10-31 05:38:27', '2026-10-3105:38:27' (OCR dropped the
    space), '31/10/2026 Sat 05:38:27 PM', '10-31-2026 05:38:27', '31 Oct 2026 05:38:27'.
    ``order`` = auto | ymd | dmy | mdy; for ambiguous day/month ``auto`` prefers the reading that
    matches ``hint`` (folder date), else day-first. Returns a naive datetime or None."""
    if not text:
        return None
    s = _fix_digits(text)
    tp = _time_part(s)
    if not tp:
        return None
    H, M, S, ts, te = tp
    date_txt = s[:ts] + " " + s[te:]
    cands = []
    m = re.search(r"(?<!\d)(20\d{2})\s*[-/.年]\s*(\d{1,2})\s*[-/.月]\s*(\d{1,2})", date_txt)
    if m and order in ("auto", "ymd"):
        cands.append((int(m.group(1)), int(m.group(2)), int(m.group(3))))
    m = re.search(r"(?<!\d)(\d{1,2})\s*[-/.]\s*(\d{1,2})\s*[-/.]\s*(20\d{2}|\d{2})(?!\d)", date_txt)
    if m:
        a, b, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        y = y + 2000 if y < 100 else y
        if order in ("auto", "dmy"):
            cands.append((y, b, a))
        if order in ("auto", "mdy"):
            cands.append((y, a, b))
    m = re.search(rf"(?<!\d)(\d{{1,2}})\s*[-/. ]?\s*({_MON_RE})[a-z]*\s*[-/., ]?\s*(20\d{{2}})", date_txt, re.I)
    if m:
        cands.append((int(m.group(3)), _MONTHS[m.group(2).lower()], int(m.group(1))))
    m = re.search(rf"({_MON_RE})[a-z]*\s*[-/. ]?\s*(\d{{1,2}})\s*[-/., ]+\s*(20\d{{2}})", date_txt, re.I)
    if m:
        cands.append((int(m.group(3)), _MONTHS[m.group(1).lower()], int(m.group(2))))
    valid = []
    for y, mo, d in cands:
        try:
            valid.append(dt.datetime(y, mo, d, H, M, S))
        except ValueError:
            continue
    if not valid:
        return None
    if hint and len(valid) > 1:
        valid.sort(key=lambda x: abs((x.date() - hint).days))
    return valid[0]

# ---------------------------------------------------------------- OSD reading

def osd_bands(frame) -> list:
    """Regions where DVRs print the clock: top and bottom strips, upscaled for small text."""
    import cv2
    h, w = frame.shape[:2]
    out = []
    for y0, y1 in ((0, int(h * 0.35)), (int(h * 0.65), h)):
        band = frame[y0:y1]
        if band.size == 0:
            continue
        if w < 1200:
            sc = 1200 / w
            band = cv2.resize(band, None, fx=sc, fy=sc, interpolation=cv2.INTER_CUBIC)
        out.append(band)
    return out


def read_clock(frame, ocr, order: str = "auto", hint: Optional[dt.date] = None) -> Optional[dt.datetime]:
    """OCR the clock printed on one frame (naive datetime) or None."""
    if frame is None or not ocr:
        return None
    for band in osd_bands(frame):
        try:
            lines = ocr.read(band)
        except Exception as e:
            log.debug("OCR failed: %s", e)
            continue
        texts = [t for t, _ in lines]
        for cand in texts + [" ".join(texts)]:
            d = parse_clock_text(cand, order, hint)
            if d and 2005 <= d.year <= 2100:
                return d
    return None


def _plausible(d: dt.datetime, hint: Optional[dt.date]) -> bool:
    # a DVR whose clock was never set prints 2000-01-01 etc.; a folder date must match within a day
    if d.year < 2005:
        return False
    return hint is None or abs((d.date() - hint).days) <= 1


def osd_start_time(path: str, meta: dict, ocr, order: str = "auto", hint: Optional[dt.date] = None,
                   samples: int = 3) -> Tuple[Optional[dt.datetime], str]:
    """Video start from the burnt-in clock; start = clock(t) - t.

    Readings at 0.5 s and 1.5 s near the start, the middle and the end are grouped by agreement;
    the biggest group wins and ties go to the group with the earliest reading, so a file stitched
    from several recordings still starts at its first frame. One group = 'consensus'; several =
    'discontinuous' (events then use their own clock reading).
    Returns (naive start, detail) - detail is 'consensus', 'single', 'discontinuous' or a reason."""
    dur = float(meta.get("duration") or 0)
    if not ocr or dur <= 0:
        return None, "no-ocr" if not ocr else "no-duration"
    offs = [min(0.5, dur / 4)]
    if dur > 3:
        offs.append(1.5)
    if samples >= 3 and dur > 8:
        offs.append(dur / 2)
    if samples >= 2 and dur > 4:
        offs.append(dur - 1.5)
    reads: List[tuple] = []
    for n, t in enumerate(offs):
        if n == 2 and not reads:
            break                     # no clock in the first two frames: footage without an OSD, stop here
        f = read_frame_at(path, t, 1280, (meta.get("width") or 0, meta.get("height") or 0))
        d = read_clock(f, ocr, order, hint)
        if d and _plausible(d, hint):
            # the clock shows the current whole second; mid-second is the unbiased estimate
            reads.append((t, d + dt.timedelta(seconds=0.5 - t)))
    if not reads:
        return None, "unreadable"
    if len(reads) == 1:
        return _round(reads[0][1]), "single"

    def close(a, b):
        return abs((a - b).total_seconds()) <= 2.5

    # Group agreeing readings. The biggest group wins; on a tie the group holding the earliest
    # reading wins - a file stitched from several recordings starts at its first frame.
    groups: List[list] = []
    for t, s in reads:
        for g in groups:
            if close(s, g[0][1]):
                g.append((t, s))
                break
        else:
            groups.append([(t, s)])
    groups.sort(key=lambda g: (-len(g), g[0][0]))
    if len(groups[0]) >= 2:
        return _round(groups[0][0][1]), "consensus" if len(groups) == 1 else "discontinuous"
    return _round(reads[0][1]), "discontinuous"


def _round(d: dt.datetime) -> dt.datetime:
    return (d + dt.timedelta(microseconds=500_000)).replace(microsecond=0)

# ----------------------------------------------------------------- resolution

def resolve_start_time(path: str, meta: dict, cfg, ocr=None) -> Tuple[Optional[dt.datetime], str]:
    """Return (aware datetime of the first frame, source label)."""
    zone = tz(cfg.timezone)
    base = os.path.basename(path)
    rel = _rel(path, cfg.input_dir)
    for key in (rel, base):
        if key in cfg.time_overrides:
            return dt.datetime.fromisoformat(cfg.time_overrides[key]).replace(tzinfo=zone), "override"
    hint = path_date_hint(path, cfg.input_dir) if cfg.use_path_date_hint else None
    for src in cfg.time_sources:
        if src == "filename":
            d = time_from_name(path, cfg.input_dir, cfg.filename_time_patterns, zone)
            if d:
                return d, "filename"
        elif src == "osd" and ocr:
            d, detail = osd_start_time(path, meta, ocr, cfg.osd_date_order, hint, cfg.osd_samples)
            if d:
                return d.replace(tzinfo=zone), f"osd({ocr.name},{detail})"
        elif src == "metadata" and meta.get("creation_time"):
            try:
                d = dt.datetime.fromisoformat(meta["creation_time"].replace("Z", "+00:00"))
                if d.tzinfo is None:
                    d = d.replace(tzinfo=dt.timezone.utc)
                d = d.astimezone(zone)
                if 2000 <= d.year <= 2100 and d.year != 1970:
                    # exact day: cloud/phone exports stamp the download time, often the next day
                    if hint and d.date() != hint:
                        log.info("  metadata time %s contradicts folder date %s – ignored", d, hint)
                    else:
                        return d, "metadata"
            except Exception:
                pass
        elif src == "mtime":
            st = os.stat(path)
            d = dt.datetime.fromtimestamp(st.st_mtime, zone) - dt.timedelta(seconds=meta.get("duration") or 0)
            if hint and d.date() != hint:
                return d.replace(year=hint.year, month=hint.month, day=hint.day), "folder-date+file-modified(approx)"
            return d, "file-modified(approx)"
    return None, "unknown"


def event_clock(path: str, meta: dict, start_dt: Optional[dt.datetime], tsrc: str, s: float, cfg, ocr):
    """(base, source) for one event starting at offset `s`: base + offset = clock time.

    Only when the video time came from the DVR clock: the clock is read again at the event start
    (files stitched from several recordings jump in time) and a disagreement larger than
    ``osd_event_tolerance`` means the event's own reading is the truth. Footage whose clock could
    not be read is not re-read per event (each failed read costs seconds)."""
    if not cfg.osd_verify_events or not tsrc.startswith("osd") or not ocr:
        return start_dt, tsrc
    f = read_frame_at(path, s + 0.1, 1280, (meta["width"], meta["height"]))
    hint = start_dt.date() if (start_dt and is_exact(tsrc)) else \
        (path_date_hint(path, cfg.input_dir) if cfg.use_path_date_hint else None)
    d = read_clock(f, ocr, cfg.osd_date_order, hint)
    if d is None or (hint and abs((d.date() - hint).days) > 1):
        return start_dt, tsrc
    zone = start_dt.tzinfo if start_dt else tz(cfg.timezone)
    clock = d.replace(tzinfo=zone) + dt.timedelta(seconds=0.5)
    if start_dt is not None and is_exact(tsrc):
        diff = (clock - (start_dt + dt.timedelta(seconds=s + 0.1))).total_seconds()
        if abs(diff) <= cfg.osd_event_tolerance:
            return start_dt, tsrc
        log.info("  clock at offset %.0fs reads %s (%.0fs from the file start time) – using the event's own clock",
                 s, d, diff)
    base = clock - dt.timedelta(seconds=s + 0.1)
    return base.replace(microsecond=0), f"osd({ocr.name},event)"

In [ ]:
%%writefile /content/cctv_analysis/motion.py
"""Pass 1: cheap full-video motion scan.

* MOG2 background subtraction on a small grey stream (default 384px @ 4 fps)
* automatic masking of the burnt-in DVR clock (OSD) + user ignore regions
* rejection of global lighting changes (lights on/off, IR switch, auto exposure)
* camera tamper detection (covered lens, blackout, defocus, moved camera)
* per-video activity heatmap
* motion intervals -> padded, merged, length-limited events
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import cv2
import numpy as np

from .video_io import iter_frames


@dataclass
class MotionResult:
    events: List[dict] = field(default_factory=list)
    times: np.ndarray = None
    ratios: np.ndarray = None
    heatmap: Optional[np.ndarray] = None
    tamper: List[dict] = field(default_factory=list)
    lighting_changes: List[float] = field(default_factory=list)
    osd_mask: Optional[np.ndarray] = None
    preview: Optional[np.ndarray] = None
    frames: int = 0


def poly_mask(shape: Tuple[int, int], polys, value=255) -> np.ndarray:
    h, w = shape
    m = np.zeros((h, w), np.uint8)
    for poly in polys or []:
        pts = np.array([[int(x * w), int(y * h)] for x, y in poly], np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(m, [pts], value)
    return m


def detect_osd_mask(path: str, meta: dict, width: int, seconds: float = 90.0) -> Optional[np.ndarray]:
    """Find the DVR clock: small text-like regions near the frame border that change
    roughly once per second in an otherwise static picture."""
    frames = [f for _, f in iter_frames(path, 2.0, width, 0, min(seconds, meta.get("duration") or seconds),
                                        gray=True, src_size=(meta["width"], meta["height"]))]
    if len(frames) < 8:
        return None
    h, w = frames[0].shape
    freq = np.zeros((h, w), np.float32)
    for a, b in zip(frames, frames[1:]):
        freq += (cv2.absdiff(a, b) > 30)
    freq /= len(frames) - 1
    cand = (freq > 0.25).astype(np.uint8) * 255
    band = np.zeros_like(cand)
    band[: int(h * 0.18)] = 255
    band[int(h * 0.85):] = 255
    cand = cv2.bitwise_and(cand, band)
    cand = cv2.dilate(cand, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 7)))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(cand)
    mask = np.zeros((h, w), np.uint8)
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if bh < h * 0.12 and bw < w * 0.75 and area > 20:
            cv2.rectangle(mask, (max(0, x - 4), max(0, y - 4)), (min(w, x + bw + 4), min(h, y + bh + 4)), 255, -1)
    return mask if mask.any() else None


def _intervals(times, active, gap):
    out = []
    s = prev = None
    for t, a in zip(times, active):
        if not a:
            continue
        if s is None:
            s = prev = t
        elif t - prev > gap:
            out.append([s, prev])
            s = prev = t
        else:
            prev = t
    if s is not None:
        out.append([s, prev])
    return out


def scan_motion(path: str, meta: dict, cfg, camera: str = "") -> MotionResult:
    res = MotionResult()
    W, H = meta["width"], meta["height"]
    width = min(cfg.motion_width, W) if W else cfg.motion_width
    sens = float(np.clip(cfg.motion_sensitivity, 0, 1))
    var_thr = 12 + (1 - sens) * 44
    diff_thr = int(14 + (1 - sens) * 26)
    min_area = cfg.min_motion_area * (1.6 - sens)
    fps = cfg.motion_fps

    mog = cv2.createBackgroundSubtractorMOG2(history=int(fps * 120), varThreshold=var_thr,
                                             detectShadows=True)
    # only *mild* darkening counts as shadow; dark clothing must stay foreground
    mog.setShadowThreshold(0.82)
    kernel_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_d = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    ign = None
    times, ratios, bright, glob = [], [], [], []
    heat = None
    prev = None
    for i, (t, g) in enumerate(iter_frames(path, fps, width, gray=True, src_size=(W, H))):
        if ign is None:
            h, w = g.shape
            polys = list(cfg.ignore_regions.get("*", [])) + list(cfg.ignore_regions.get(camera, []))
            ign = poly_mask((h, w), polys)
            if cfg.auto_mask_osd:
                osd = detect_osd_mask(path, meta, width)
                if osd is not None and osd.shape == ign.shape:
                    ign = cv2.bitwise_or(ign, osd)
                    res.osd_mask = osd
            keep = cv2.bitwise_not(ign)
            heat = np.zeros((h, w), np.float32)
            valid_px = max(1, int(np.count_nonzero(keep)))
        gb = cv2.GaussianBlur(g, (5, 5), 0)
        lr = 1.0 if i == 0 else -1
        fg = mog.apply(gb, learningRate=lr)
        fg = np.where(fg > 200, 255, 0).astype(np.uint8)          # drop shadows (127)
        if prev is not None:                                        # confirm with frame diff
            d = cv2.absdiff(gb, prev)
            dm = (d > diff_thr).astype(np.uint8) * 255
            dm = cv2.dilate(dm, kernel_d, iterations=2)
            fg = cv2.bitwise_and(fg, dm)
        prev = gb
        fg = cv2.bitwise_and(fg, keep)
        raw_ratio = np.count_nonzero(fg) / valid_px
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel_o)
        fg = cv2.dilate(fg, kernel_d)
        n, _, stats, _ = cv2.connectedComponentsWithStats(fg)
        blob_min = valid_px * min_area * 0.25
        area = sum(int(stats[k, 4]) for k in range(1, n) if stats[k, 4] >= blob_min)
        ratio = area / valid_px
        is_global = raw_ratio > cfg.global_change_ratio
        if i < 3:
            ratio = 0.0
        if is_global:
            # lighting/exposure jump: rebuild the background from this frame
            if not res.lighting_changes or t - res.lighting_changes[-1] > 1.5:
                res.lighting_changes.append(float(t))
            mog.apply(gb, learningRate=1.0)
            ratio = 0.0
        else:
            heat += (fg > 0)
        times.append(t)
        ratios.append(ratio)
        glob.append(is_global)
        if cfg.detect_tamper and i % max(1, int(fps)) == 0:
            bright.append((t, float(g.mean()), float(cv2.Laplacian(g, cv2.CV_32F).var()), float(g.std())))
        if res.preview is None and i == int(fps * 2):
            res.preview = g.copy()
    res.frames = len(times)
    res.times = np.asarray(times, np.float32)
    res.ratios = np.asarray(ratios, np.float32)
    if heat is not None and heat.max() > 0:
        res.heatmap = heat / heat.max()
    if not times:
        return res

    # ---------------- events
    active = res.ratios >= min_area
    # require 2 active samples within a short window to suppress single-frame noise
    sm = np.convolve(active.astype(np.float32), np.ones(3), mode="same")
    active = active & (sm >= 2)
    ivs = _intervals(res.times, active, cfg.event_merge_gap)
    dur = float(meta.get("duration") or res.times[-1])
    for s, e in ivs:
        if e - s + 1.0 / fps < cfg.event_min_duration:
            continue
        s2, e2 = max(0.0, s - cfg.event_pre_roll), min(dur, e + cfg.event_post_roll)
        # split very long events
        cur = s2
        while cur < e2:
            nxt = min(e2, cur + cfg.event_max_duration)
            sel = (res.times >= cur) & (res.times <= nxt)
            r = res.ratios[sel]
            res.events.append({"start": float(cur), "end": float(nxt),
                               "motion_peak": float(r.max()) if r.size else 0.0,
                               "motion_mean": float(r.mean()) if r.size else 0.0,
                               "motion_active_s": float(np.count_nonzero(active[sel]) / fps)})
            cur = nxt
    # merge padded overlaps
    merged: List[dict] = []
    for ev in res.events:
        if merged and ev["start"] <= merged[-1]["end"] and \
                ev["end"] - merged[-1]["start"] <= cfg.event_max_duration:
            m = merged[-1]
            m["end"] = max(m["end"], ev["end"])
            m["motion_peak"] = max(m["motion_peak"], ev["motion_peak"])
            m["motion_mean"] = (m["motion_mean"] + ev["motion_mean"]) / 2
            m["motion_active_s"] += ev["motion_active_s"]
        else:
            merged.append(ev)
    res.events = merged

    # ---------------- tamper detection
    if cfg.detect_tamper and len(bright) >= 10:
        arr = np.array(bright)
        base_b = np.median(arr[:, 1])
        base_s = np.median(arr[:, 2])
        base_d = np.median(arr[:, 3])
        flags = []
        for t, b, s, d in arr:
            kind = None
            if b < 12 and base_b > 35:
                kind = "blackout/covered"
            elif s < 0.15 * base_s and d < 0.5 * base_d and d < 20 and base_d > 15:
                kind = "lens covered/obstructed"
            elif s < 0.08 * base_s and base_s > 50:
                kind = "defocus/blur"
            flags.append((t, kind))
        cur = None
        for t, kind in flags:
            if kind and cur and cur["kind"] == kind and t - cur["end"] <= 3:
                cur["end"] = float(t)
            elif kind:
                if cur:
                    res.tamper.append(cur)
                cur = {"start": float(t), "end": float(t), "kind": kind}
        if cur:
            res.tamper.append(cur)
        res.tamper = [x for x in res.tamper
                      if x["end"] - x["start"] >= (10 if x["kind"] == "defocus/blur" else 3)]
        # sustained global change = camera moved / scene changed
        g = np.array(glob)
        run = 0
        for t, gg in zip(res.times, g):
            run = run + 1 if gg else 0
            if run == int(fps * 4):
                res.tamper.append({"start": float(t - 4), "end": float(t), "kind": "scene change / camera moved"})
    return res

In [ ]:
%%writefile /content/cctv_analysis/detector.py
"""Pass 2 models: Ultralytics YOLO (detection + multi-object tracking) and an
optional pose model used for posture cues (crouching, hands raised, face
visibility)."""
from __future__ import annotations

import logging
from typing import List, Optional

import numpy as np

log = logging.getLogger("cctv")

COCO_KPT = {"nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4, "l_sh": 5, "r_sh": 6,
            "l_el": 7, "r_el": 8, "l_wr": 9, "r_wr": 10, "l_hip": 11, "r_hip": 12,
            "l_kn": 13, "r_kn": 14, "l_an": 15, "r_an": 16}


def resolve_device(pref: str = "auto") -> str:
    if pref and pref != "auto":
        return pref
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda:0"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


class _DropNoise(logging.Filter):
    NOISE = ("'half' is deprecated", "Use 'quantize' instead")

    def filter(self, record):
        msg = record.getMessage()
        return not any(n in msg for n in self.NOISE)


def _quiet_ultralytics():
    lg = logging.getLogger("ultralytics")
    if not any(isinstance(f, _DropNoise) for f in lg.filters):
        lg.addFilter(_DropNoise())
        for h in lg.handlers:
            h.addFilter(_DropNoise())


def load_yolo(candidates: List[str]):
    from ultralytics import YOLO

    from . import modelcache
    _quiet_ultralytics()
    last = None
    for name in candidates:
        if not name:
            continue
        try:
            m = YOLO(modelcache.yolo_weights(name))
            log.info("Loaded %s", name)
            return m, name
        except Exception as e:  # weights missing / unsupported by installed version
            last = e
            log.warning("Could not load %s (%s); trying fallback", name, e)
    raise RuntimeError(f"No YOLO model could be loaded: {last}")


def iou(a, b) -> float:
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


class Detector:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = resolve_device(cfg.device)
        self.half = self.device.startswith("cuda")
        self.model, self.model_name = load_yolo([cfg.detector_model, *cfg.detector_fallbacks])
        names = self.model.names
        self.names = names if isinstance(names, dict) else dict(enumerate(names))
        if cfg.keep_classes:
            wanted = set(cfg.keep_classes)
            self.class_ids = [i for i, n in self.names.items() if n in wanted]
        else:
            self.class_ids = None
        self.pose = None
        self.pose_name = ""
        if cfg.use_pose and cfg.pose_model:
            try:
                self.pose, self.pose_name = load_yolo([cfg.pose_model, "yolo11n-pose.pt", "yolov8n-pose.pt"])
            except Exception as e:
                log.warning("Pose model disabled: %s", e)
        self._fresh = True
        self._extra = {}
        if self.half:                      # newer ultralytics: quantize=16 replaces the deprecated half=True
            try:
                from ultralytics.cfg import DEFAULT_CFG_DICT
                self._extra = {"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}
            except Exception:
                self._extra = {"half": True}

    def reset(self):
        """Start a new, independent tracking session (one per event)."""
        self._fresh = True

    def __call__(self, frame: np.ndarray) -> List[dict]:
        cfg = self.cfg
        r = self.model.track(frame, persist=not self._fresh, tracker=cfg.tracker, conf=cfg.det_conf,
                             imgsz=cfg.det_imgsz, classes=self.class_ids, device=self.device,
                             verbose=False, **self._extra)[0]
        self._fresh = False
        dets = []
        b = r.boxes
        if b is not None and len(b):
            xyxy = b.xyxy.cpu().numpy()
            conf = b.conf.cpu().numpy()
            cls = b.cls.cpu().numpy().astype(int)
            ids = b.id.cpu().numpy().astype(int) if b.id is not None else [None] * len(cls)
            for k in range(len(cls)):
                dets.append({"cls": self.names.get(int(cls[k]), str(cls[k])), "conf": float(conf[k]),
                             "xyxy": [float(v) for v in xyxy[k]],
                             "track": int(ids[k]) if ids[k] is not None else None, "kpts": None})
        if self.pose is not None and any(d["cls"] == "person" for d in dets):
            try:
                p = self.pose.predict(frame, conf=max(0.25, cfg.det_conf - 0.05), imgsz=cfg.det_imgsz,
                                      device=self.device, verbose=False, **self._extra)[0]
                if p.keypoints is not None and p.boxes is not None and len(p.boxes):
                    pb = p.boxes.xyxy.cpu().numpy()
                    kp = p.keypoints.data.cpu().numpy()          # (n,17,3)
                    for d in dets:
                        if d["cls"] != "person":
                            continue
                        best, bi = 0.0, -1
                        for j in range(len(pb)):
                            v = iou(d["xyxy"], pb[j])
                            if v > best:
                                best, bi = v, j
                        if best > 0.5:
                            d["kpts"] = kp[bi]
            except Exception as e:
                log.debug("pose failed: %s", e)
        return dets


# ------------------------------------------------------------ pose heuristics

def posture(kpts: Optional[np.ndarray], conf_thr: float = 0.4) -> dict:
    """Simple, explainable posture cues from COCO keypoints."""
    out = {"face_visible_pose": False, "crouching": False, "hands_up": False}
    if kpts is None:
        return out
    k = kpts

    def ok(n):
        return k[COCO_KPT[n], 2] >= conf_thr

    def p(n):
        return k[COCO_KPT[n], :2]

    out["face_visible_pose"] = ok("nose") and ok("l_eye") and ok("r_eye")
    if ok("l_sh") and ok("r_sh") and ok("l_hip") and ok("r_hip"):
        sh = (p("l_sh") + p("r_sh")) / 2
        hip = (p("l_hip") + p("r_hip")) / 2
        torso = np.linalg.norm(sh - hip) + 1e-6
        if ok("l_kn") and ok("r_kn"):
            kn = (p("l_kn") + p("r_kn")) / 2
            # knees close to hip height -> crouching / squatting / bending low
            if (kn[1] - hip[1]) < 0.35 * torso:
                out["crouching"] = True
        for w in ("l_wr", "r_wr"):
            if ok(w):
                wr = p(w)
                if wr[1] < sh[1] - 0.25 * torso:
                    out["hands_up"] = True
        if out["hands_up"]:
            both = ok("l_wr") and ok("r_wr") and p("l_wr")[1] < sh[1] and p("r_wr")[1] < sh[1]
            out["hands_up"] = bool(both)
    return out

In [ ]:
%%writefile /content/cctv_analysis/faces.py
"""Face detection (OpenCV YuNet), quality scoring, recognition embeddings
(OpenCV SFace, Apache-2.0) and a persistent cross-video identity database
with optional known-person enrolment (staff / owner)."""
from __future__ import annotations

import json
import logging
import os
import urllib.request
from typing import List, Optional

import cv2
import numpy as np

log = logging.getLogger("cctv")

CACHE = os.path.expanduser("~/.cache/cctv_analysis")
MODELS = {
    "yunet": ["https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
              "https://huggingface.co/opencv/face_detection_yunet/resolve/main/face_detection_yunet_2023mar.onnx"],
    "sface": ["https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
              "https://huggingface.co/opencv/face_recognition_sface/resolve/main/face_recognition_sface_2021dec.onnx"],
}


def fetch(key: str) -> str:
    from . import modelcache
    cache = modelcache.sub("opencv")
    dst = os.path.join(cache, os.path.basename(MODELS[key][0]))
    old = os.path.join(CACHE, os.path.basename(MODELS[key][0]))
    if not os.path.exists(dst) and os.path.exists(old) and os.path.getsize(old) > 100_000:
        import shutil
        shutil.copyfile(old, dst)
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000:
        return dst
    for url in MODELS[key]:
        try:
            urllib.request.urlretrieve(url, dst + ".part")
            if os.path.getsize(dst + ".part") > 100_000:
                os.replace(dst + ".part", dst)
                return dst
        except Exception as e:
            log.warning("download failed %s: %s", url, e)
    raise RuntimeError(f"could not download {key} model")


def sharpness(img: np.ndarray) -> float:
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


class FaceEngine:
    def __init__(self, cfg):
        self.cfg = cfg
        self.det = cv2.FaceDetectorYN.create(fetch("yunet"), "", (320, 320), cfg.face_score_threshold, 0.3, 50)
        self.rec = cv2.FaceRecognizerSF.create(fetch("sface"), "")

    def detect_in_people(self, frame: np.ndarray, persons: List[dict]) -> List[dict]:
        """Search faces in the upper part of each person box (much better than the
        full frame for small CCTV faces). Returns faces tagged with track id."""
        H, W = frame.shape[:2]
        out = []
        for p in persons:
            x1, y1, x2, y2 = p["xyxy"]
            bw, bh = x2 - x1, y2 - y1
            if bw < 12 or bh < 24:
                continue
            cx1 = int(max(0, x1 - 0.1 * bw))
            cx2 = int(min(W, x2 + 0.1 * bw))
            # full body (tall box): face is in the top half; close-up / partial body: search almost all of it
            frac = 0.5 if bh / max(bw, 1) > 2.0 else 0.9
            cy1 = int(max(0, y1 - 0.08 * bh))
            cy2 = int(min(H, y1 + frac * bh))
            crop = frame[cy1:cy2, cx1:cx2]
            if crop.size == 0:
                continue
            scale = 1.0
            if crop.shape[1] < 200:
                scale = 200.0 / crop.shape[1]
                crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
            self.det.setInputSize((crop.shape[1], crop.shape[0]))
            _, faces = self.det.detect(crop)
            if faces is None:
                continue
            for f in faces:
                f = f.copy()
                f[:14] /= scale
                # layout: x, y, w, h, 5 landmarks (x, y), score -> offset positions, not sizes
                f[[0, 4, 6, 8, 10, 12]] += cx1
                f[[1, 5, 7, 9, 11, 13]] += cy1
                fw, fh = f[2], f[3]
                if min(fw, fh) < self.cfg.face_min_size:
                    continue
                out.append({"track": p.get("track"), "row": f, "score": float(f[14]),
                            "box": [float(f[0]), float(f[1]), float(f[0] + fw), float(f[1] + fh)]})
                break  # one face per person
        return out

    def detect_full(self, frame: np.ndarray) -> List[dict]:
        self.det.setInputSize((frame.shape[1], frame.shape[0]))
        _, faces = self.det.detect(frame)
        res = []
        for f in faces if faces is not None else []:
            res.append({"track": None, "row": f, "score": float(f[14]),
                        "box": [float(f[0]), float(f[1]), float(f[0] + f[2]), float(f[1] + f[3])]})
        return res

    def quality(self, frame: np.ndarray, face: dict) -> float:
        f = face["row"]
        x, y, w, h = [int(v) for v in f[:4]]
        crop = frame[max(0, y):y + h, max(0, x):x + w]
        if crop.size == 0:
            return 0.0
        # frontal-ness from landmarks: nose centred between the eyes
        le, re_, nose = f[4:6], f[6:8], f[8:10]
        eye_d = abs(re_[0] - le[0]) + 1e-6
        frontal = 1.0 - min(1.0, abs((nose[0] - (le[0] + re_[0]) / 2) / eye_d) * 1.5)
        size = min(1.0, min(w, h) / 80.0)
        sharp = min(1.0, sharpness(crop) / 150.0)
        return float(face["score"] * (0.4 * size + 0.3 * sharp + 0.3 * max(0, frontal)))

    def embed(self, frame: np.ndarray, face: dict) -> Optional[np.ndarray]:
        try:
            aligned = self.rec.alignCrop(frame, face["row"])
            e = self.rec.feature(aligned).flatten().astype(np.float32)
            return e / (np.linalg.norm(e) + 1e-9)
        except Exception:
            return None

    def crop(self, frame: np.ndarray, box, pad: float = 0.35) -> np.ndarray:
        H, W = frame.shape[:2]
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        return frame[int(max(0, y1 - pad * h)):int(min(H, y2 + pad * h)),
                     int(max(0, x1 - pad * w)):int(min(W, x2 + pad * w))].copy()


def enroll_folder(engine: FaceEngine, folder: str, what: str = "faces") -> List[dict]:
    """folder/<Name>/*.jpg (or folder/<Name>.jpg) -> [{"name", "emb" (unit vector), "n"}]."""
    if not folder:
        return []
    if not os.path.isdir(folder):
        log.info("%s folder not found (%s) – continuing without it", what, folder)
        return []
    people = []
    for name in sorted(os.listdir(folder)):
        d = os.path.join(folder, name)
        files = [os.path.join(d, f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else [d]
        label = name if os.path.isdir(d) else os.path.splitext(name)[0]
        embs = []
        for f in files:
            try:
                img = cv2.imread(f)
            except Exception:
                img = None
            if img is None:
                log.warning("face image unreadable, skipped: %s", f)
                continue
            faces = sorted(engine.detect_full(img), key=lambda x: -x["score"] * (x["box"][2] - x["box"][0]))
            if faces:
                e = engine.embed(img, faces[0])
                if e is not None:
                    embs.append(e)
        if embs:
            c = np.mean(embs, 0)
            people.append({"name": label, "emb": c / np.linalg.norm(c), "n": len(embs)})
    log.info("%s: enrolled %d people", what, len(people))
    return people


class Watchlist:
    """People of interest (face_watchlist_dir). A match raises the 'watchlist_face' flag."""

    def __init__(self, engine: FaceEngine, folder: str, threshold: float):
        self.thr = threshold
        self.people = enroll_folder(engine, folder, "Face watchlist")

    def match(self, emb: np.ndarray):
        best = ("", -1.0)
        for p in self.people:
            s = float(np.dot(emb, p["emb"]))
            if s >= self.thr and s > best[1]:
                best = (p["name"], s)
        return best


class IdentityDB:
    """Greedy online clustering of face embeddings, persisted between runs so that
    'P0007' means the same person across days. Known people come from
    known_faces_dir/<Name>/*.jpg and are matched first."""

    def __init__(self, path: str, match_thr: float, known_thr: float):
        self.path = path
        self.thr = match_thr
        self.known_thr = known_thr
        self.ids: List[dict] = []
        self.emb: List[np.ndarray] = []
        self.known: List[dict] = []
        if os.path.exists(path + ".json") and os.path.exists(path + ".npy"):
            with open(path + ".json") as fh:
                self.ids = json.load(fh)
            arr = np.load(path + ".npy")
            self.emb = [a for a in arr]
            if len(self.emb) != len(self.ids):
                self.ids, self.emb = [], []

    def enroll_known(self, engine: FaceEngine, folder: str) -> int:
        self.known = enroll_folder(engine, folder, "Known-faces")
        return len(self.known)

    run_id = ""
    last_new = False

    def assign(self, emb: np.ndarray, when: str, sample: str = "", event_id: str = "",
               camera: str = "") -> tuple:
        """Return (identity_id, similarity, known_name). `self.last_new` tells whether
        the identity was created by this call (a face never seen before)."""
        self.last_new = False
        known_name, known_sim = "", -1.0
        for k in self.known:
            s = float(np.dot(emb, k["emb"]))
            if s >= self.known_thr and (not known_name or s > known_sim):
                known_name, known_sim = k["name"], s
        best, bi = -1.0, -1
        if self.emb:
            sims = np.stack(self.emb) @ emb
            bi = int(np.argmax(sims))
            best = float(sims[bi])
        if bi >= 0 and best >= self.thr:
            rec = self.ids[bi]
            n = rec["count"]
            c = self.emb[bi] * n + emb
            self.emb[bi] = c / np.linalg.norm(c)
            rec["count"] = n + 1
            ev = rec.setdefault("events", [])
            if event_id and event_id not in ev:
                ev.append(event_id)
            rec["last_seen"] = max(rec["last_seen"], when)
            rec["first_seen"] = min(rec["first_seen"], when)
            if known_name and not rec.get("name"):
                rec["name"] = known_name
            cams = rec.setdefault("cameras", [])
            if camera and camera not in cams:
                cams.append(camera)
            return rec["id"], best, rec.get("name", "") or known_name
        new_id = f"P{len(self.ids) + 1:04d}"
        self.ids.append({"id": new_id, "count": 1, "first_seen": when, "last_seen": when,
                         "sample": sample, "name": known_name, "events": [event_id] if event_id else [],
                         "first_run": self.run_id, "first_event": event_id, "first_camera": camera})
        self.last_new = True
        self.emb.append(emb)
        return new_id, 1.0, known_name

    def forget_events(self, event_ids) -> None:
        """Drop references to events that were deleted (video re-analysed)."""
        gone = set(event_ids)
        if not gone:
            return
        for r in self.ids:
            if r.get("events"):
                r["events"] = [e for e in r["events"] if e not in gone]

    def set_sample(self, pid: str, path: str):
        for r in self.ids:
            if r["id"] == pid and not r.get("sample"):
                r["sample"] = path

    def save(self):
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
        with open(self.path + ".json", "w") as fh:
            json.dump(self.ids, fh, indent=1)
        np.save(self.path + ".npy", np.stack(self.emb) if self.emb else np.zeros((0, 128), np.float32))


def match_identities(engine: FaceEngine, img: np.ndarray, identities_path: str, threshold: float):
    """Search by photo: [(identity id, similarity)] of stored identities matching the largest face
    in `img`, best first. Returns None when the photo has no detectable face."""
    faces = sorted(engine.detect_full(img), key=lambda f: -(f["box"][2] - f["box"][0]))
    if not faces:
        return None
    q = engine.embed(img, faces[0])
    if q is None or not (os.path.exists(identities_path + ".json") and os.path.exists(identities_path + ".npy")):
        return []
    with open(identities_path + ".json") as fh:
        ids = json.load(fh)
    emb = np.load(identities_path + ".npy")
    if len(ids) != len(emb) or not len(emb):
        return []
    sims = emb @ q
    return [(ids[i]["id"], float(sims[i])) for i in np.argsort(-sims) if sims[i] >= threshold]

In [ ]:
%%writefile /content/cctv_analysis/resources.py
"""CPU and RAM protection for 8 GB laptops without a GPU.

``governor()`` returns the process-wide :class:`MemoryGovernor`. Code that processes items in
batches (SigLIP keyframe encoding, appearance vectors, multi-frame stacking) asks it for the
batch size: below ``high`` (85 % of RAM in use) the nominal size is used; above it the size is
halved for every 5 percentage points over the limit (down to 1), garbage is collected, and above
``critical`` (93 %) the caller is made to wait (up to ``max_wait`` seconds) for memory to be
released before continuing. Every throttling decision is counted and logged once per minute so
a run's log shows when and how often it happened.

Memory readings come from ``psutil.virtual_memory().percent`` (system-wide, the figure Task
Manager shows). Tests inject a reader.
"""
from __future__ import annotations

import gc
import logging
import threading
import time
from typing import Callable, Optional

log = logging.getLogger("cctv")


def _psutil_percent() -> float:
    try:
        import psutil
        return float(psutil.virtual_memory().percent)
    except Exception:                       # no psutil or no permission: never throttle on a guess
        return 0.0


class MemoryGovernor:
    def __init__(self, high: float = 85.0, critical: float = 93.0, max_wait: float = 30.0,
                 reader: Optional[Callable[[], float]] = None, sleep: Callable[[float], None] = time.sleep):
        self.high, self.critical, self.max_wait = high, critical, max_wait
        self.reader = reader or _psutil_percent
        self.sleep = sleep
        self.throttled = 0
        self.waited = 0.0
        self._last_log = 0.0
        self._lock = threading.Lock()

    def percent(self) -> float:
        return self.reader()

    def batch_size(self, nominal: int) -> int:
        """Batch size to use now (nominal when memory is fine)."""
        nominal = max(1, int(nominal))
        used = self.percent()
        if used < self.high:
            return nominal
        steps = 1 + int((used - self.high) // 5)
        size = max(1, nominal >> steps)
        gc.collect()
        with self._lock:
            self.throttled += 1
            now = time.monotonic()
            if now - self._last_log > 60:
                self._last_log = now
                log.warning("memory %.0f%% in use (limit %.0f%%): batch %d -> %d (throttled %d time(s))",
                            used, self.high, nominal, size, self.throttled)
        if used >= self.critical:
            self.wait_for_memory()
        return size

    def wait_for_memory(self) -> bool:
        """Block while memory is critical; True if it dropped below the limit in time."""
        t0 = time.monotonic()
        while self.percent() >= self.critical:
            if time.monotonic() - t0 >= self.max_wait:
                log.warning("memory still above %.0f%% after %.0f s – continuing with batch size 1",
                            self.critical, self.max_wait)
                return False
            gc.collect()
            self.sleep(1.0)
            with self._lock:
                self.waited += 1.0
        return True

    def status(self) -> dict:
        return {"percent": round(self.percent(), 1), "high": self.high, "critical": self.critical,
                "throttled": self.throttled, "waited_seconds": self.waited}


_GOV = {"g": MemoryGovernor()}


def governor() -> MemoryGovernor:
    return _GOV["g"]


def set_governor(g: MemoryGovernor) -> MemoryGovernor:
    _GOV["g"] = g
    return g

In [ ]:
%%writefile /content/cctv_analysis/semantic.py
"""SigLIP-2 zero-shot scene tagging of keyframes + an embedding index that
powers free-text search ("person in red shirt near counter")."""
from __future__ import annotations

import logging
import os
from typing import List, Tuple

import numpy as np

log = logging.getLogger("cctv")


class Semantic:
    def __init__(self, cfg, device: str):
        import torch
        from transformers import AutoModel, AutoProcessor
        self.torch = torch
        self.device = device if device.startswith("cuda") else "cpu"
        dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
        from . import modelcache
        cd = modelcache.hf_dir()
        self.model = AutoModel.from_pretrained(cfg.semantic_model, dtype=dtype, cache_dir=cd).to(self.device).eval()
        self.proc = AutoProcessor.from_pretrained(cfg.semantic_model, cache_dir=cd)
        self.dtype = dtype
        self.sus = list(cfg.suspicious_prompts)
        self.norm = list(cfg.normal_prompts)
        self.prompts = self.sus + self.norm
        self.text_emb = self.encode_text([f"this is a photo of {x}." for x in self.prompts])
        self.scale = float(self.model.logit_scale.exp().item())
        self.bias = float(self.model.logit_bias.item()) if hasattr(self.model, "logit_bias") else 0.0

    def encode_text(self, texts: List[str]) -> np.ndarray:
        with self.torch.no_grad():
            t = self.proc(text=[x.lower() for x in texts], padding="max_length", max_length=64,
                          truncation=True, return_tensors="pt").to(self.device)
            e = self.model.get_text_features(**t)
            if not isinstance(e, self.torch.Tensor):
                e = getattr(e, "pooler_output", e[0])
            e = e.float()
            e = e / e.norm(dim=-1, keepdim=True)
        return e.cpu().numpy()

    def encode_images(self, images_bgr: List[np.ndarray], batch: int = 16) -> np.ndarray:
        from PIL import Image

        from .resources import governor
        outs = []
        i = 0
        while i < len(images_bgr):
            n = governor().batch_size(batch)           # smaller batches when RAM is above 85 %
            ims = [Image.fromarray(im[:, :, ::-1]) for im in images_bgr[i:i + n]]
            i += n
            with self.torch.no_grad():
                x = self.proc(images=ims, return_tensors="pt").to(self.device)
                x = {k: (v.to(self.dtype) if v.dtype.is_floating_point else v) for k, v in x.items()}
                e = self.model.get_image_features(**x)
                if not isinstance(e, self.torch.Tensor):
                    e = getattr(e, "pooler_output", e[0])
                e = e.float()
                e = e / e.norm(dim=-1, keepdim=True)
            outs.append(e.cpu().numpy())
        return np.concatenate(outs) if outs else np.zeros((0, self.text_emb.shape[1]), np.float32)

    def classify(self, img_emb: np.ndarray) -> List[dict]:
        """SigLIP's native per-prompt sigmoid scores (well calibrated: ~0 when no
        prompt fits). 'lift' compares the best suspicious vs best normal prompt."""
        res = []
        logits = img_emb @ self.text_emb.T * self.scale + self.bias
        sig = 1 / (1 + np.exp(-logits))
        ns = len(self.sus)
        for row in sig:
            top = int(np.argmax(row))
            ts = int(np.argmax(row[:ns]))
            tn = ns + int(np.argmax(row[ns:])) if len(row) > ns else None
            best_n = float(row[tn]) if tn is not None else 0.0
            lift = float(row[ts] / (row[ts] + best_n + 1e-9))
            res.append({"top_label": self.prompts[top], "top_score": float(row[top]),
                        "top_suspicious": self.sus[ts], "suspicious_score": float(row[ts]),
                        "top_normal": self.prompts[tn] if tn is not None else "", "normal_score": best_n,
                        "lift": lift})
        return res


def backfill_index(out_dir: str, cfg, device: str = "cpu") -> int:
    """Embed the saved keyframes (raw snapshots) of events that have no entry in
    ``semantic_index.npz`` yet – events analysed while scene search was off – so that turning
    it on later does not require analysing the videos again. Returns the number of keyframes added."""
    import json
    import sqlite3

    import cv2

    from .storage import resolve
    path = os.path.join(out_dir, "semantic_index.npz")
    keys: List[str] = []
    embs: List[np.ndarray] = []
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        keys, embs = [str(k) for k in z["keys"]], list(z["emb"])
    have = {k.rpartition("|")[0] for k in keys}
    db = sqlite3.connect(f"file:{os.path.join(out_dir, 'analysis.db')}?mode=ro", uri=True)
    try:
        rows = db.execute("SELECT event_id, snapshots FROM events").fetchall()
    finally:
        db.close()
    todo = []
    for eid, snaps in rows:
        if eid in have:
            continue
        for sn in json.loads(snaps or "[]"):
            p = resolve(out_dir, sn.get("raw") or sn.get("annotated"))
            if p:
                todo.append((f"{eid}|{float(sn.get('t_off') or 0):.2f}", p))
    if not todo:
        return 0
    sem = Semantic(cfg, device)
    added = 0
    for i in range(0, len(todo), 64):
        part = [(k, cv2.imread(p)) for k, p in todo[i:i + 64]]
        part = [(k, im) for k, im in part if im is not None]
        if not part:
            continue
        for (k, _), e in zip(part, sem.encode_images([im for _, im in part])):
            keys.append(k)
            embs.append(e.astype(np.float16))
            added += 1
    np.savez_compressed(path + ".tmp.npz", emb=np.stack(embs).astype(np.float16), keys=np.array(keys, dtype=object))
    os.replace(path + ".tmp.npz", path)
    log.info("scene search: %d keyframe(s) of earlier events added to the index", added)
    return added


def search_index(sem: Semantic, index_path: str, query: str, k: int = 20) -> List[Tuple[str, float]]:
    """Rank stored keyframes by similarity to a free-text query."""
    if not os.path.exists(index_path):
        return []
    z = np.load(index_path, allow_pickle=True)
    emb, keys = z["emb"].astype(np.float32), z["keys"]
    q = sem.encode_text([f"this is a photo of {query}."])[0]
    s = emb @ q
    order = np.argsort(-s)[:k]
    return [(str(keys[i]), float(s[i])) for i in order]

In [ ]:
%%writefile /content/cctv_analysis/vlm.py
"""Vision-language model (VLM) providers: event descriptions + suspicious verdicts,
and image reading used for plates/clock when OCR is not enough.

* ``transformers`` – a local Hugging Face model (default Qwen3-VL-2B-Instruct).
* ``openai``       – any OpenAI-compatible ``/v1/chat/completions`` server running a vision
                     model: Ollama, LM Studio, vLLM, llama.cpp server, a hosted API …

Wire format (openai provider): OpenAI Chat Completions with an image content part
``{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,…"}}`` – see
https://platform.openai.com/docs/api-reference/chat/create . Ollama accepts this object form
(ollama/openai/openai.go, case "image_url", which takes either a {"url"} object or a string).
The API key is read from the environment variable named by ``Config.vlm_api_key_env``; it is
never written to config files or logs.
"""
from __future__ import annotations

import base64
import json
import logging
import os
import re
from typing import List

import numpy as np

log = logging.getLogger("cctv")


class VLMBase:
    name = "base"

    def __init__(self, cfg):
        self.cfg = cfg

    def ask(self, img_bgr: np.ndarray, prompt: str, max_new_tokens: int = 160) -> str:  # pragma: no cover
        raise NotImplementedError

    def describe_event(self, sheet_bgr: np.ndarray, context: str = "") -> dict:
        prompt = self.cfg.vlm_prompt + (f"\nContext from detectors: {context}" if context else "")
        return parse_verdict(self.ask(sheet_bgr, prompt, self.cfg.vlm_max_new_tokens))

    def read_plate(self, crop_bgr: np.ndarray) -> str:
        txt = self.ask(crop_bgr, "This is a vehicle number plate. Reply ONLY with the plate characters "
                                 "(letters and digits, no spaces), or NONE if unreadable.", 24)
        t = re.sub(r"[^A-Z0-9]", "", txt.upper())
        return "" if t in ("", "NONE") else t


def _resize(img_bgr: np.ndarray, max_side: int) -> np.ndarray:
    import cv2
    h, w = img_bgr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    return cv2.resize(img_bgr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img_bgr


class TransformersVLM(VLMBase):
    name = "transformers"

    def __init__(self, cfg, device: str):
        super().__init__(cfg)
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.torch = torch
        cuda = device.startswith("cuda")
        if cuda:
            if torch.cuda.is_bf16_supported():
                dtype = torch.bfloat16
            else:
                # T4 & older: fp16 can overflow in Qwen-VL; small models fit comfortably in fp32
                small = any(t in cfg.vlm_model for t in ("256M", "500M", "0.5B", "1B", "2B"))
                dtype = torch.float32 if small else torch.float16
        else:
            dtype = torch.float32
        from . import modelcache
        cd = modelcache.hf_dir()
        self.model = AutoModelForImageTextToText.from_pretrained(
            cfg.vlm_model, dtype=dtype, device_map="auto" if cuda else None, cache_dir=cd).eval()
        if not cuda:
            self.model.to("cpu")
        self.proc = AutoProcessor.from_pretrained(cfg.vlm_model, cache_dir=cd)

    def ask(self, img_bgr, prompt, max_new_tokens=160):
        from PIL import Image
        im = Image.fromarray(_resize(img_bgr, 1280)[:, :, ::-1])
        messages = [{"role": "user", "content": [{"type": "image", "image": im},
                                                 {"type": "text", "text": prompt}]}]
        inputs = self.proc.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(self.model.device)
        with self.torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return self.proc.batch_decode(gen, skip_special_tokens=True)[0].strip()


class OpenAICompatVLM(VLMBase):
    """Any server implementing OpenAI's POST {base_url}/chat/completions with image inputs."""
    name = "openai"

    def __init__(self, cfg, session=None):
        super().__init__(cfg)
        if not (cfg.vlm_base_url or "").strip():
            raise RuntimeError("NOT_CONFIGURED: vlm_base_url is empty (e.g. http://127.0.0.1:11434/v1 for Ollama)")
        import requests
        self.http = session or requests.Session()
        self.url = cfg.vlm_base_url.rstrip("/") + "/chat/completions"
        self.key = os.environ.get(cfg.vlm_api_key_env or "", "")

    def payload(self, img_bgr: np.ndarray, prompt: str, max_new_tokens: int) -> dict:
        import cv2
        ok, buf = cv2.imencode(".jpg", _resize(img_bgr, 1280), [cv2.IMWRITE_JPEG_QUALITY, 88])
        if not ok:
            raise ValueError("could not encode image")
        b64 = base64.b64encode(buf.tobytes()).decode()
        return {"model": self.cfg.vlm_model, "max_tokens": int(max_new_tokens), "temperature": 0,
                "messages": [{"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b64}}]}]}

    def ask(self, img_bgr, prompt, max_new_tokens=160):
        headers = {"Content-Type": "application/json"}
        if self.key:
            headers["Authorization"] = "Bearer " + self.key
        r = self.http.post(self.url, json=self.payload(img_bgr, prompt, max_new_tokens), headers=headers,
                           timeout=self.cfg.vlm_timeout)
        if r.status_code != 200:
            raise RuntimeError(f"VLM server returned HTTP {r.status_code}: {r.text[:200]}")
        j = r.json()
        try:
            content = j["choices"][0]["message"]["content"]
        except (KeyError, IndexError, TypeError):
            raise RuntimeError(f"unexpected VLM response shape: {str(j)[:200]}")
        if isinstance(content, list):          # some servers return content parts
            content = "".join(p.get("text", "") for p in content if isinstance(p, dict))
        return (content or "").strip()


def list_server_models(base_url: str, api_key: str = "", timeout: float = 10.0) -> List[str]:
    """GET {base_url}/models (OpenAI 'List models'); used by the app's 'Test connection'."""
    import requests
    h = {"Authorization": "Bearer " + api_key} if api_key else {}
    r = requests.get(base_url.rstrip("/") + "/models", headers=h, timeout=timeout)
    r.raise_for_status()
    return sorted(m.get("id", "") for m in r.json().get("data", []) if isinstance(m, dict))


def make_vlm(cfg, device: str) -> VLMBase:
    if cfg.vlm_provider == "openai":
        return OpenAICompatVLM(cfg)
    if cfg.vlm_provider == "transformers":
        return TransformersVLM(cfg, device)
    raise ValueError(f"unknown vlm_provider {cfg.vlm_provider!r}")


def parse_verdict(txt: str) -> dict:
    out: dict = {"description": txt.strip(), "suspicious": None, "category": "", "confidence": None,
                 "reason": "", "raw": txt}
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        raw = m.group(0)
        for cand in (raw, raw.replace("'", '"'), re.sub(r",\s*}", "}", raw)):
            try:
                j = json.loads(cand)
                out.update({k: j.get(k, out[k]) for k in ("description", "suspicious", "category",
                                                          "confidence", "reason")})
                break
            except Exception:
                continue
    if isinstance(out["suspicious"], str):
        out["suspicious"] = out["suspicious"].strip().lower() in ("true", "yes", "1")
    if out["suspicious"] is None:
        low = txt.lower()
        out["suspicious"] = bool(re.search(r"\b(suspicious|theft|steal|stealing|fight|weapon|conceal)", low)) \
            and not re.search(r"not suspicious|nothing suspicious|no suspicious", low)
    try:
        out["confidence"] = float(out["confidence"]) if out["confidence"] is not None else None
    except Exception:
        out["confidence"] = None
    return out

In [ ]:
%%writefile /content/cctv_analysis/plates.py
"""Vehicle number-plate reading (ANPR) for recorded footage. Off unless ``Config.use_plates``.

Per vehicle track the pipeline keeps the best few observations (large, sharp, confident);
this module re-reads those frames at full resolution and:

1. localises the plate inside the vehicle box – ``yolo-plate``: open-image-models YOLOv9
   plate detector (ONNX, MIT); ``classic``: OpenCV morphology (no download needed);
2. reads it – ``fast-plate-ocr`` (ONNX, MIT, plate-specialised, per-character confidence) or the
   generic OCR engine (RapidOCR / Tesseract);
3. with ``plate_enhance`` (fast vehicles): motion deblurring – the track gives the direction of
   motion, so the blur is modelled as a linear PSF along it; Wiener deconvolution is run for a
   range of blur lengths and the reconstruction the plate OCR is most confident in (and that
   fits the plate format) is kept. Plus a multi-frame fusion (crops from several frames
   aligned with ECC and median-stacked) that is deblurred the same way;
4. with ``plate_vlm_read``: asks the vision-language model when OCR confidence stays low;
5. corrects letter/digit confusions against the plate format (Indian formats incl. BH series)
   and votes across all reads, character by character.
"""
from __future__ import annotations

import logging
import re
from collections import defaultdict
from typing import Callable, List, Optional, Tuple, Union

import cv2
import numpy as np

log = logging.getLogger("cctv")

PLATE_DET_MODEL = "yolo-v9-t-384-license-plate-end2end"
PLATE_OCR_MODEL = "cct-s-v2-global-model"

IN_STATES = {"AN", "AP", "AR", "AS", "BR", "CG", "CH", "DD", "DL", "DN", "GA", "GJ", "HP", "HR", "JH", "JK",
             "KA", "KL", "LA", "LD", "MH", "ML", "MN", "MP", "MZ", "NL", "OD", "OR", "PB", "PY", "RJ", "SK",
             "TN", "TR", "TS", "UK", "UA", "UP", "WB"}
TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "B": "8", "G": "6",
            "A": "4", "T": "7"}
TO_LETTER = {v: k for k, v in [("O", "0"), ("I", "1"), ("Z", "2"), ("S", "5"), ("B", "8"), ("G", "6"),
                               ("A", "4"), ("T", "7")]}


# ---------------------------------------------------------------- formats

def normalize(text: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", (text or "").upper())


def _fit(text: str, classes: str) -> Optional[Tuple[str, int]]:
    """Map text onto a class string ('L' letter / 'D' digit); returns (fixed, substitutions)."""
    if len(text) != len(classes):
        return None
    out, cost = [], 0
    for ch, cl in zip(text, classes):
        if cl == "D":
            if ch.isdigit():
                out.append(ch)
            elif ch in TO_DIGIT:
                out.append(TO_DIGIT[ch])
                cost += 1
            else:
                return None
        else:
            if ch.isalpha():
                out.append(ch)
            elif ch in TO_LETTER:
                out.append(TO_LETTER[ch])
                cost += 1
            else:
                return None
    return "".join(out), cost


_LETTER_ALTS = {"0": "ODQ", "1": "IL", "2": "Z", "4": "A", "5": "S", "6": "G", "7": "T", "8": "B",
                "O": "D", "D": "O", "I": "L", "L": "I", "U": "W", "V": "W", "H": "MN", "M": "HN", "N": "MH",
                "W": "M"}


def _state_fix(two: str) -> str:
    """A valid state code reachable by substituting OCR look-alikes (0L -> DL, NH -> MH is not tried)."""
    opts = [[c] + list(_LETTER_ALTS.get(c, "")) for c in two]
    for a in opts[0]:
        for b in opts[1]:
            if a + b in IN_STATES and a.isalpha() and b.isalpha():
                return a + b
    return ""


def _in_templates(n: int):
    """State (2 letters) + RTO (2 digits, 1 for some older plates) + series (1-3 letters; none
    only on old two-digit-RTO plates) + 4 digits; or Bharat series 22BH1234AB."""
    for d in (2, 1):
        for s in (2, 1, 3, 0):
            if s == 0 and d == 1:
                continue
            if 2 + d + s + 4 == n:
                yield "LL" + "D" * d + "L" * s + "DDDD"
    if n in (9, 10):                 # Bharat series: 22BH1234AB
        yield "DDLLDDDD" + "L" * (n - 8)


def plate_fit(text: str, region: str = "IN") -> Tuple[str, float]:
    """(corrected text, fit 0..1): 1.0 = matches the regional format as read; each letter/digit
    substitution needed costs a quarter; more than one substitution = not a plate (0.3) – a
    truncated read must never be 'corrected' into a plausible but wrong plate."""
    t = normalize(text)
    if not t:
        return "", 0.0
    if region != "IN":
        return t, 1.0
    best = None
    for tpl in _in_templates(len(t)):
        r = _fit(t, tpl)
        if r is None:
            continue
        fixed, cost = r
        bh = tpl.startswith("DDLL")
        if bh and fixed[2:4] != "BH":
            continue
        if not bh and fixed[:2] not in IN_STATES:
            state = _state_fix(t[:2])
            if not state:
                continue
            fixed, cost = state + fixed[2:], cost + 1
        if best is None or cost < best[1]:
            best = (fixed, cost)
    if best is None or best[1] > 1:
        return t, 0.3
    return best[0], 1.0 - 0.25 * best[1]


def format_plate(text: str, region: str = "IN") -> Tuple[str, bool]:
    """(best corrected text, matches the regional format with at most one correction)."""
    t, fit = plate_fit(text, region)
    return t, fit > 0.5


def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def is_reportable(text: str, confidence: float, valid_format: bool) -> bool:
    """Shown in event summaries / matched against the watchlist. Partial reads (vehicle cut by
    the frame edge, 'MH12') stay in the plates table as evidence but are not reported."""
    return bool(text) and (valid_format or (len(text) >= 6 and confidence >= 0.7))


def within_edits(a: str, b: str, k: int) -> bool:
    """edit_distance(a, b) <= k, computed only inside the diagonal band and stopped as soon as
    every cell exceeds k (fast for the common 'no match' case)."""
    if abs(len(a) - len(b)) > k:
        return False
    if k == 0:
        return a == b
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [k + 1] * len(b)
        lo, hi = max(1, i - k), min(len(b), i + k)
        for j in range(lo, hi + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != b[j - 1]))
        if min(cur[max(0, lo - 1):hi + 1]) > k:
            return False
        prev = cur
    return prev[-1] <= k


def watchlist_match(text: str, watchlist: List[str], max_edits: int = 1) -> str:
    t = normalize(text)
    if not t:
        return ""
    best = ("", max_edits + 1)
    for w in watchlist or []:
        n = normalize(w)
        if not n:
            continue
        d = edit_distance(t, n)
        if d < best[1]:
            best = (n, d)
    return best[0] if best[1] <= max_edits else ""


def vote(reads: List[Tuple[str, float]], region: str = "IN") -> Tuple[str, float, bool]:
    """Fuse several (text, confidence) reads. Same-length candidates vote per character, weighted
    by confidence; the most supported length wins. Returns (text, confidence 0..1, valid format)."""
    fmt = [(plate_fit(t, region), float(c)) for t, c in reads if normalize(t)]
    if not fmt:
        return "", 0.0, False
    # weight = confidence x format fit: garbage from blurred frames and corrected reads count less
    reads = [(f, c * fit) for (f, fit), c in fmt]
    by_len = defaultdict(list)
    for t, c in reads:
        by_len[len(t)].append((t, c))
    L, group = max(by_len.items(), key=lambda kv: (sum(c for _, c in kv[1]), len(kv[1])))
    chars, agree = [], []
    for i in range(L):
        w: dict = defaultdict(float)
        for t, c in group:
            w[t[i]] += max(c, 0.05)
        ch, s = max(w.items(), key=lambda kv: kv[1])
        chars.append(ch)
        agree.append(s / sum(w.values()))
    fused, valid = format_plate("".join(chars), region)
    mean_conf = min(1.0, sum(c for _, c in group) / len(group))
    support = len(group) / len(reads)
    conf = float(min(agree) * (0.5 + 0.5 * mean_conf) * (0.6 + 0.4 * support))
    return fused, round(conf, 3), valid


# ---------------------------------------------------------- enhancement

def sharpness(img: np.ndarray) -> float:
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def motion_psf(length: int, angle: float) -> np.ndarray:
    """Normalised linear motion-blur kernel of `length` px along `angle` (radians, image coords)."""
    size = length + 2
    k = np.zeros((size, size), np.float32)
    c = (size - 1) / 2
    for i in range(length):
        t = i - (length - 1) / 2
        k[int(round(c + t * np.sin(angle))), int(round(c + t * np.cos(angle)))] = 1
    return k / k.sum()


def wiener_deconvolve(img: np.ndarray, psf: np.ndarray, k: float) -> np.ndarray:
    pad = psf.shape[0] * 2
    out = np.empty(img.shape, np.float32)
    chans = img.shape[2] if img.ndim == 3 else 1
    for ch in range(chans):
        x = img[:, :, ch] if img.ndim == 3 else img
        f = cv2.copyMakeBorder(x, pad, pad, pad, pad, cv2.BORDER_REFLECT).astype(np.float32) / 255
        h = np.zeros_like(f)
        h[:psf.shape[0], :psf.shape[1]] = psf
        h = np.roll(h, (-(psf.shape[0] // 2), -(psf.shape[1] // 2)), (0, 1))
        H = np.fft.fft2(h)
        r = np.real(np.fft.ifft2(np.conj(H) / (np.abs(H) ** 2 + k) * np.fft.fft2(f)))[pad:-pad, pad:-pad]
        if img.ndim == 3:
            out[:, :, ch] = r
        else:
            out = r
    return np.clip(out * 255, 0, 255).astype(np.uint8)


class Enhancer:
    """Blind motion deblurring scored by the OCR, and multi-frame aligned fusion."""
    method = "deblur"
    LENGTHS = tuple(range(3, 27))         # blur length in px at 2x upscale (every length: the true one must be tried)
    NOISE = (0.002, 0.006, 0.02)

    def __init__(self, region: str = "IN"):
        self.region = region

    def score(self, text: str, conf: float) -> float:
        f, fit = plate_fit(text, self.region)
        return conf * fit * min(1.0, len(f) / 10) if f else 0.0

    def best_read(self, crop: np.ndarray, reader, angle: float = 0.0, inner=None, second=None):
        """(text, conf, image) of the most convincing deblurred reconstruction, or None.
        `crop` should include a margin around the plate; `inner` = plate rect (x1, y1, x2, y2) in it.
        The plate reader scores every blur length; the 3 best reconstructions are also read by
        `second` (general OCR), which is more tolerant of deconvolution ringing (measured: 7/12
        exact reads vs 0/12 with the plate reader alone on H.264-like compressed blurred video)."""
        up = cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        cands = []
        tried = set()

        def attempt(L, k):
            if (L, k) in tried or L < self.LENGTHS[0] or L > self.LENGTHS[-1]:
                return
            tried.add((L, k))
            img = wiener_deconvolve(up, motion_psf(L, angle), k)
            if inner is not None:
                x1, y1, x2, y2 = (2 * v for v in inner)
                img = img[y1:y2, x1:x2]
            txt, c = reader.read(img)
            if txt:
                cands.append((self.score(txt, c), txt, c, img, L))

        # coarse pass over blur lengths, then every length/noise level around the two best
        for L in self.LENGTHS[::3]:
            attempt(L, self.NOISE[1])
        top = sorted({c[4] for c in sorted(cands, key=lambda r: -r[0])[:2]})
        for L0 in top:
            for L in range(L0 - 2, L0 + 3):
                for k in self.NOISE:
                    attempt(L, k)
        cands = [c[:4] for c in cands]
        if not cands:
            return None
        cands.sort(key=lambda r: -r[0])
        best = cands[0]
        if second is not None:
            for _, _, _, img in cands[:3]:
                txt, c = second.read(img)
                sc = self.score(txt, c)
                if txt and sc > best[0]:
                    best = (sc, txt, c, img)
        return best[1:]

    @staticmethod
    def fuse(crops: List[np.ndarray]) -> Optional[np.ndarray]:
        """Align plate crops from several frames to the sharpest one (ECC, affine) and take the
        per-pixel median: noise and single-frame artefacts average out."""
        crops = [c for c in crops if c is not None and c.size and min(c.shape[:2]) >= 6]
        if len(crops) < 2:
            return None
        crops.sort(key=sharpness, reverse=True)
        h, w = crops[0].shape[:2]
        ref = cv2.cvtColor(crops[0], cv2.COLOR_BGR2GRAY).astype(np.float32)
        stack = [crops[0].astype(np.float32)]
        for c in crops[1:]:
            c = cv2.resize(c, (w, h), interpolation=cv2.INTER_CUBIC)
            g = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY).astype(np.float32)
            warp = np.eye(2, 3, dtype=np.float32)
            try:
                cc, warp = cv2.findTransformECC(ref, g, warp, cv2.MOTION_AFFINE,
                                                (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 60, 1e-4), None, 3)
            except cv2.error:
                continue
            if cc < 0.5:
                continue
            stack.append(cv2.warpAffine(c, warp, (w, h), flags=cv2.INTER_CUBIC + cv2.WARP_INVERSE_MAP,
                                        borderMode=cv2.BORDER_REPLICATE).astype(np.float32))
        if len(stack) < 2:
            return None
        return np.clip(np.median(np.stack(stack), axis=0), 0, 255).astype(np.uint8)


# ------------------------------------------------------------- localisers

class YoloPlateDetector:
    name = "yolo-plate"

    def __init__(self):
        import pathlib

        from open_image_models.detection.core.hub import download_model
        from open_image_models.detection.factory import DETECTION_MODELS, create_detector

        from . import modelcache
        p = download_model(PLATE_DET_MODEL, save_directory=pathlib.Path(modelcache.sub("plates")) / PLATE_DET_MODEL)
        spec = DETECTION_MODELS[PLATE_DET_MODEL]
        self.det = create_detector(str(p), backend=spec.backend, class_labels=spec.class_labels, conf_thresh=0.25)

    def find(self, img: np.ndarray) -> List[Tuple[List[int], float]]:
        out = []
        for r in self.det.predict(img):
            b = r.bounding_box
            out.append(([int(b.x1), int(b.y1), int(b.x2), int(b.y2)], float(r.confidence)))
        return sorted(out, key=lambda x: -x[1])


class ClassicPlateDetector:
    """Bright/dark text-dense rectangles with a plate-like aspect ratio (no model download)."""
    name = "classic"

    def find(self, img: np.ndarray) -> List[Tuple[List[int], float]]:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        H, W = g.shape
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (max(9, W // 25), max(3, H // 40)))
        grad = cv2.morphologyEx(g, cv2.MORPH_GRADIENT, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
        _, th = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cl = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k)
        cnts, _ = cv2.findContours(cl, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        out = []
        for c in cnts:
            x, y, w, h = cv2.boundingRect(c)
            ar = w / max(1, h)
            if 2.0 <= ar <= 6.5 and w >= W * 0.08 and h >= 8 and w < W * 0.9:
                fill = cv2.contourArea(c) / float(w * h)
                out.append(([x, y, x + w, y + h], float(min(0.6, fill * 0.6))))
        return sorted(out, key=lambda x: -x[1])[:3]


# ------------------------------------------------------------------ OCR

class FastPlateOCR:
    name = "fast-plate-ocr"

    def __init__(self):
        import pathlib

        from fast_plate_ocr import LicensePlateRecognizer
        from fast_plate_ocr.inference.hub import download_model

        from . import modelcache
        m, c = download_model(PLATE_OCR_MODEL, save_directory=pathlib.Path(modelcache.sub("plates")) / PLATE_OCR_MODEL)
        self.rec = LicensePlateRecognizer(onnx_model_path=str(m), plate_config_path=str(c), device="cpu")

    def read(self, crop: np.ndarray) -> Tuple[str, float]:
        p = self.rec.run(crop, return_confidence=True)
        if not p:
            return "", 0.0
        p = p[0]
        probs = getattr(p, "char_probs", None)
        probs = np.asarray(probs if probs is not None else [], dtype=np.float32)
        return normalize(p.plate), float(probs.mean()) if probs.size else 0.0


class GenericPlateOCR:
    name = "generic"

    def __init__(self, ocr):
        self.ocr = ocr

    def read(self, crop: np.ndarray) -> Tuple[str, float]:
        if not self.ocr:
            return "", 0.0
        big = cv2.resize(crop, None, fx=max(1.0, 64 / max(1, crop.shape[0])), fy=max(1.0, 64 / max(1, crop.shape[0])),
                         interpolation=cv2.INTER_CUBIC)
        lines = self.ocr.read(big)
        if not lines:
            return "", 0.0
        text = "".join(normalize(t) for t, _ in lines)
        return text, float(np.mean([c for _, c in lines]))


# --------------------------------------------------------------- reader

class PlateReader:
    def __init__(self, cfg, ocr_getter: Callable, vlm_getter: Optional[Callable] = None,
                 on_vlm_error: Optional[Callable] = None):
        self.cfg = cfg
        self.vlm_getter = vlm_getter
        self.on_vlm_error = on_vlm_error
        self.detector: Optional[Union[YoloPlateDetector, ClassicPlateDetector]] = None
        if cfg.plate_detector in ("auto", "yolo-plate"):
            try:
                self.detector = YoloPlateDetector()
            except Exception as e:
                if cfg.plate_detector == "yolo-plate":
                    raise
                log.warning("plate detector model unavailable (%s) – using the OpenCV plate finder", e)
        if self.detector is None:
            self.detector = ClassicPlateDetector()
        self.find = self.detector.find
        self.reader: Optional[Union[FastPlateOCR, GenericPlateOCR]] = None
        if cfg.plate_ocr in ("auto", "fast-plate-ocr"):
            try:
                self.reader = FastPlateOCR()
            except Exception as e:
                if cfg.plate_ocr == "fast-plate-ocr":
                    raise
                log.warning("plate OCR model unavailable (%s) – using generic OCR", e)
        generic = GenericPlateOCR(ocr_getter())
        if self.reader is None:
            self.reader = generic
        self.read = self.reader.read
        self.second = generic if (generic.ocr and self.reader is not generic) else None
        self.enhancer = Enhancer(cfg.plate_region) if cfg.plate_enhance else None
        log.info("Plate reading: detector=%s, ocr=%s, enhance=%s, vlm=%s", self.detector.name, self.reader.name,
                 self.enhancer.method if self.enhancer else "off", bool(cfg.plate_vlm_read))

    def locate(self, frame: np.ndarray, vbox):
        """Best plate inside a vehicle box (full-resolution pixel coords):
        (crop, plate box in the frame, detector confidence, (context crop, plate rect inside it))."""
        H, W = frame.shape[:2]
        x1, y1, x2, y2 = [int(v) for v in vbox]
        px, py = int(0.06 * (x2 - x1)), int(0.06 * (y2 - y1))
        x1, y1, x2, y2 = max(0, x1 - px), max(0, y1 - py), min(W, x2 + px), min(H, y2 + py)
        roi = frame[y1:y2, x1:x2]
        if roi.size == 0:
            return None
        found = self.find(roi)
        if not found:
            return None
        (bx1, by1, bx2, by2), conf = found[0]
        bw, bh = bx2 - bx1, by2 - by1
        pad_x, pad_y = int(0.08 * bw), int(0.15 * bh)
        cx1, cy1 = max(0, bx1 - pad_x), max(0, by1 - pad_y)
        cx2, cy2 = min(roi.shape[1], bx2 + pad_x), min(roi.shape[0], by2 + pad_y)
        crop = roi[cy1:cy2, cx1:cx2].copy()
        if crop.size == 0:
            return None
        # wider context for deblurring: motion blur smears characters beyond the plate box
        fx1, fy1, fx2, fy2 = x1 + cx1, y1 + cy1, x1 + cx2, y1 + cy2
        mx, my = max(16, int(0.4 * bw)), max(10, int(0.6 * bh))
        kx1, ky1, kx2, ky2 = max(0, fx1 - mx), max(0, fy1 - my), min(W, fx2 + mx), min(H, fy2 + my)
        context = (frame[ky1:ky2, kx1:kx2].copy(), (fx1 - kx1, fy1 - ky1, fx2 - kx1, fy2 - ky1))
        return crop, [x1 + bx1, y1 + by1, x1 + bx2, y1 + by2], conf, context

    def _settled(self, reads) -> bool:
        """Three confident, format-perfect, identical reads: more deblurring cannot change the vote."""
        good = [r[0] for r in reads if r[1] >= 0.7 and plate_fit(r[0], self.cfg.plate_region)[1] == 1.0]
        return any(good.count(t) >= 3 for t in set(good))

    def read_track(self, samples: List[tuple]) -> Optional[dict]:
        """samples: (t_offset, full-res frame, vehicle box in that frame[, motion angle in radians]).
        Returns the fused result (or None when no plate was found in any sample)."""
        cfg = self.cfg
        crops, reads = [], []
        enhanced = None
        for smp in samples:
            t, frame, vbox = smp[:3]
            angle = smp[3] if len(smp) > 3 else 0.0
            loc = self.locate(frame, vbox)
            if loc is None:
                continue
            crop, box, dconf, (context, inner) = loc
            crops.append((t, crop, box, angle))
            txt, c = self.read(crop)
            if txt:
                reads.append((txt, c, "frame", t))
            if self.enhancer is not None and self.enhancer.score(txt, c) < 0.9 and not self._settled(reads):
                r = self.enhancer.best_read(context, self.reader, angle, inner, self.second)
                if r:
                    reads.append((r[0], r[1], "deblur", t))
                    if enhanced is None or self.enhancer.score(r[0], r[1]) > enhanced[0]:
                        enhanced = (self.enhancer.score(r[0], r[1]), r[2])
        if not crops:
            return None
        best_crop = max(crops, key=lambda x: sharpness(x[1]) * x[1].shape[1])
        if self.enhancer is not None:
            fused = Enhancer.fuse([c for _, c, _, _ in crops])
            if fused is not None:
                txt, c = self.read(fused)
                if txt:
                    reads.append((txt, c, "multi-frame", best_crop[0]))
                r = self.enhancer.best_read(fused, self.reader, float(np.median([a for *_, a in crops])),
                                            second=self.second)
                if r:
                    reads.append((r[0], r[1], "multi-frame+deblur", best_crop[0]))
                    if enhanced is None or self.enhancer.score(r[0], r[1]) > enhanced[0]:
                        enhanced = (self.enhancer.score(r[0], r[1]), r[2])
        enhanced = enhanced[1] if enhanced else None
        text, conf, valid = vote([(r[0], r[1]) for r in reads], cfg.plate_region)
        method = "ocr"
        if cfg.plate_vlm_read and (not text or conf < 0.6) and self.vlm_getter is not None:
            vlm = self.vlm_getter()
            if vlm is not None:
                try:
                    vt = vlm.read_plate(enhanced if enhanced is not None else best_crop[1])
                    if vt:
                        reads.append((vt, 0.7, "vlm", best_crop[0]))
                        text, conf, valid = vote([(r[0], r[1]) for r in reads], cfg.plate_region)
                        method = "ocr+vlm"
                except Exception as e:
                    log.warning("  VLM plate read failed: %s", e)
                    if self.on_vlm_error:
                        self.on_vlm_error(e)
        out = {"text": text, "confidence": conf if text else 0.0, "valid_format": valid if text else False,
               "t_off": best_crop[0], "crop": best_crop[1], "enhanced": enhanced, "box": best_crop[2],
               "reads": [{"text": r[0], "conf": round(r[1], 3), "source": r[2], "t_off": round(r[3], 2)} for r in reads],
               "method": (method + ("+deblur" if self.enhancer is not None else "")) if text else "unreadable"}
        return out

In [ ]:
%%writefile /content/cctv_analysis/rules.py
"""Explainable suspicion scoring. Every flag carries a weight and a human
reason, so a reviewer can see *why* an event was ranked high."""
from __future__ import annotations

import datetime as dt
from typing import Dict, List, Optional

import numpy as np

# event levels, most to least suspicious (the pipeline assigns them from the score)
LEVELS = ("HIGH", "REVIEW", "LOW", "NORMAL")

WEIGHTS = {
    "watchlist_face": 50,
    "watchlist_plate": 50,
    "after_hours": 30,
    "camera_tamper": 45,
    "unauthorized_person": 35,
    "weapon_like_object": 40,
    "restricted_zone": 25,
    "loitering": 15,
    "running": 12,
    "crowd": 10,
    "face_hidden": 18,
    "hands_up": 20,
    "crouching": 8,
    "lights_after_hours": 20,
    "semantic_match": 15,
    "vlm_suspicious": 30,
    "unknown_person_after_hours": 10,
    "repeat_visitor": 5,
    "long_event": 5,
}


def point_in_poly(x: float, y: float, poly) -> bool:
    inside = False
    n = len(poly)
    j = n - 1
    for i in range(n):
        xi, yi = poly[i]
        xj, yj = poly[j]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi:
            inside = not inside
        j = i
    return inside


def zones_for(cfg, camera: str) -> Dict[str, list]:
    z = dict(cfg.zones.get("*", {}))
    z.update(cfg.zones.get(camera, {}))
    return z


def lines_for(cfg, camera: str) -> Dict[str, list]:
    z = dict(cfg.count_lines.get("*", {}))
    z.update(cfg.count_lines.get(camera, {}))
    return z


def _side(p, a, b) -> float:
    return (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])


def _segments_cross(p, q, a, b) -> bool:
    d1, d2 = _side(p, a, b), _side(q, a, b)
    d3, d4 = _side(a, p, q), _side(b, p, q)
    return d1 * d2 < 0 and d3 * d4 < 0


def count_crossings(points, lines: Dict[str, list]) -> Dict[str, Dict[str, int]]:
    """Counting lines crossed by a trajectory of (x, y) points (normalised, foot position).
    'in' = from the right-hand side of the line a→b to its left-hand side (image coordinates,
    y down: for a line drawn left→right, 'in' means moving upwards/away from the camera)."""
    out = {}
    for name, (a, b) in lines.items():
        n_in = n_out = 0
        for p, q in zip(points, points[1:]):
            if _segments_cross(p, q, a, b):
                if _side(p, a, b) > 0:
                    n_in += 1
                else:
                    n_out += 1
        if n_in or n_out:
            out[name] = {"in": n_in, "out": n_out}
    return out


def _hm(s: str) -> dt.time:
    h, m = s.split(":")
    return dt.time(int(h), int(m))


def hours_enabled(cfg) -> bool:
    return bool((cfg.open_hour or "").strip() and (cfg.close_hour or "").strip())


def is_after_hours(when: Optional[dt.datetime], cfg) -> bool:
    if when is None:
        return False
    if not hours_enabled(cfg):            # no shop hours given -> every hour is normal
        return when.weekday() in (cfg.closed_weekdays or [])
    if when.weekday() in (cfg.closed_weekdays or []):
        return True
    o, c = _hm(cfg.open_hour), _hm(cfg.close_hour)
    t = when.time()
    if o <= c:
        return not (o <= t < c)
    return c <= t < o           # shop open across midnight


def score_event(ev: dict, tracks: List[dict], cfg, vlm: Optional[dict] = None,
                semantic: Optional[List[dict]] = None, tamper: Optional[List[dict]] = None) -> tuple:
    flags = []

    def add(key, reason, scale=1.0):
        flags.append({"flag": key, "points": round(WEIGHTS[key] * scale, 1), "reason": reason})

    persons = [t for t in tracks if t["cls"] == "person" and t["duration"] >= 0.5]
    start_dt = ev.get("start_dt")
    cam = ev.get("camera", "")
    auth = cfg.authorized_only_cameras or []
    if persons and ("*" in auth or cam in auth):
        strangers = [t for t in persons if not t.get("known_name")]
        if strangers:
            ids = sorted({t["identity"] for t in strangers if t.get("identity")})
            nof = sum(1 for t in strangers if not t.get("identity"))
            add("unauthorized_person", f"authorised-only camera '{cam}': {len(strangers)} person(s) not in known faces"
                + (f" (unknown {', '.join(ids)})" if ids else "") + (f"; {nof} without a visible face" if nof else ""))
    if is_after_hours(start_dt, cfg) and (persons or ev.get("motion_peak", 0) > 0.01):
        add("after_hours", f"activity at {start_dt:%H:%M} outside shop hours "
            f"{cfg.open_hour or '-'}-{cfg.close_hour or '-'}"
            + (" with people" if persons else " (no person detected)"), 1.0 if persons else 0.5)
    for tp in tamper or []:
        if tp["start"] <= ev["end"] and tp["end"] >= ev["start"]:
            add("camera_tamper", f"{tp['kind']} for {tp['end'] - tp['start']:.0f}s")
            break
    wl = sorted({t["watchlist"] for t in persons if t.get("watchlist")})
    if wl:
        add("watchlist_face", "person on the face watchlist: " + ", ".join(wl))
    wp = [p for p in ev.get("plates") or [] if p.get("watchlist")]
    if wp:
        add("watchlist_plate", "watchlist plate: " + ", ".join(f"{p['text']} (≈{p['watchlist']})" for p in wp))
    weapons = sorted({t["cls"] for t in tracks if t["cls"] in cfg.weapon_classes and t["max_conf"] >= 0.4})
    if weapons:
        add("weapon_like_object", "detected: " + ", ".join(weapons))
    zone_hits: Dict[str, float] = {}
    for t in persons:
        for z, secs in t.get("zones", {}).items():
            zone_hits[z] = zone_hits.get(z, 0) + secs
    zone_hits = {z: s for z, s in zone_hits.items() if s >= 1.0}
    if zone_hits:
        add("restricted_zone", "; ".join(f"{z}: {s:.0f}s" for z, s in zone_hits.items()))
    loiter = [t for t in persons if t["duration"] >= cfg.loiter_seconds]
    if loiter:
        add("loitering", f"{len(loiter)} person(s) stayed ≥{cfg.loiter_seconds:.0f}s "
                         f"(max {max(t['duration'] for t in loiter):.0f}s)")
    runners = [t for t in persons if t.get("max_speed", 0) >= cfg.running_speed]
    if runners:
        add("running", f"{len(runners)} fast-moving person(s), {max(t['max_speed'] for t in runners):.1f} body-heights/s")
    if ev.get("max_persons", 0) >= cfg.crowd_size:
        add("crowd", f"{ev['max_persons']} people at once")
    hidden = [t for t in persons if t.get("hidden_face_s", 0) >= cfg.hidden_face_seconds]
    if hidden:
        add("face_hidden", f"{len(hidden)} person(s) facing camera but no face detected "
                           f"(possible mask/helmet/cover) ≥{cfg.hidden_face_seconds:.0f}s")
    if any(t.get("hands_up_s", 0) >= 1.5 for t in persons):
        add("hands_up", "both hands raised above shoulders (possible threat / hold-up)")
    if any(t.get("crouch_s", 0) >= 3 for t in persons):
        add("crouching", "person crouching/bending low for a while")
    if ev.get("lighting_changes") and is_after_hours(start_dt, cfg):
        add("lights_after_hours", f"{ev['lighting_changes']} sudden lighting change(s) after hours (lights switched on/off?)")
    if semantic:
        best = max(semantic, key=lambda s: s["suspicious_score"])
        if best["suspicious_score"] >= cfg.semantic_alert_score and best["lift"] >= 0.6:
            add("semantic_match", f"scene resembles: '{best['top_suspicious']}' ({best['suspicious_score']:.2f})")
    if vlm and vlm.get("suspicious"):
        c = vlm.get("confidence")
        c = 0.7 if c is None else max(0.3, min(1.0, float(c)))
        add("vlm_suspicious", f"AI description: {vlm.get('category') or ''} – {vlm.get('reason') or vlm.get('description', '')[:120]}", c)
    if is_after_hours(start_dt, cfg) and any(t.get("identity") and not t.get("known_name") for t in persons):
        add("unknown_person_after_hours", "unrecognised face after hours")
    if any(t.get("identity_visits", 0) >= 5 and not t.get("known_name") for t in persons):  # distinct events
        add("repeat_visitor", "same unknown face seen in ≥5 events")
    if ev["end"] - ev["start"] >= 300 and persons:
        add("long_event", f"event lasted {(ev['end'] - ev['start']) / 60:.0f} min")
    staff_only = persons and all(t.get("known_name") for t in persons if t.get("identity"))
    score = sum(f["points"] for f in flags)
    if staff_only and any(t.get("identity") for t in persons) and not weapons and \
            not any(f["flag"] in ("unauthorized_person", "watchlist_face", "watchlist_plate") for f in flags):
        score *= 0.5
        flags.append({"flag": "known_people_only", "points": 0, "reason": "only known/staff faces – score halved"})
    return float(min(100.0, score)), flags


def summarize_track(t: dict, fps: float, frame_h: int) -> dict:
    """Turn raw per-frame observations into track statistics."""
    obs = t["obs"]
    times = np.array([o[0] for o in obs])
    out = {k: t[k] for k in ("track", "cls")}
    out["first"] = float(times.min())
    out["last"] = float(times.max())
    out["duration"] = float(out["last"] - out["first"])
    out["frames"] = len(obs)
    out["max_conf"] = float(max(o[1] for o in obs))
    out["mean_conf"] = float(np.mean([o[1] for o in obs]))
    cx = np.array([(o[2][0] + o[2][2]) / 2 for o in obs])
    cy = np.array([(o[2][1] + o[2][3]) / 2 for o in obs])
    hh = np.array([max(1.0, o[2][3] - o[2][1]) for o in obs])
    sp = 0.0
    if len(obs) >= 3:
        dt_ = np.diff(times)
        d = np.hypot(np.diff(cx), np.diff(cy)) / ((hh[1:] + hh[:-1]) / 2)
        v = d / np.maximum(dt_, 1e-3)
        v = np.convolve(v, np.ones(3) / 3, mode="valid") if len(v) >= 3 else v
        sp = float(np.max(v)) if len(v) else 0.0
    out["max_speed"] = sp
    out["path"] = [[round(float(a), 1), round(float(b), 1)] for a, b in zip(cx[::max(1, len(cx) // 40)], cy[::max(1, len(cy) // 40)])]
    step = 1.0 / fps
    out["zones"] = {z: n * step for z, n in t.get("zone_frames", {}).items()}
    out["hidden_face_s"] = t.get("hidden_frames", 0) * step
    out["hands_up_s"] = t.get("hands_up", 0) * step
    out["crouch_s"] = t.get("crouch", 0) * step
    out["faces_found"] = t.get("faces_found", 0)
    return out

In [ ]:
%%writefile /content/cctv_analysis/annotate.py
"""Drawing helpers: boxes, labels, header bars, zones, contact sheets, heatmaps."""
from __future__ import annotations

from typing import List, Optional, Sequence

import cv2
import numpy as np

PALETTE = [(56, 56, 255), (151, 157, 255), (31, 112, 255), (29, 178, 255), (49, 210, 207),
           (10, 249, 72), (23, 204, 146), (134, 219, 61), (52, 147, 26), (187, 212, 0),
           (168, 153, 44), (255, 194, 0), (147, 69, 52), (255, 115, 100), (236, 24, 0),
           (255, 56, 132), (133, 0, 82), (255, 56, 203), (200, 149, 255), (199, 55, 255)]


def color_for(i) -> tuple:
    if i is None:
        return PALETTE[0]
    k = i if isinstance(i, int) else sum(map(ord, str(i)))
    return PALETTE[(k * 7) % len(PALETTE)]


def label(img, text, x, y, color, scale=0.5):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    y = max(th + 4, y)
    cv2.rectangle(img, (x, y - th - 4), (x + tw + 4, y + bl - 2), color, -1)
    cv2.putText(img, text, (x + 2, y - 3), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)


def draw_zones(img, zones: dict):
    h, w = img.shape[:2]
    over = img.copy()
    for name, poly in zones.items():
        pts = np.array([[int(x * w), int(y * h)] for x, y in poly], np.int32)
        cv2.fillPoly(over, [pts], (0, 0, 255))
        cv2.polylines(img, [pts], True, (0, 0, 255), 2)
        label(img, name, int(pts[:, 0].min()), int(pts[:, 1].min()) + 14, (0, 0, 180))
    cv2.addWeighted(over, 0.15, img, 0.85, 0, img)


def draw_dets(frame, dets: List[dict], faces: Sequence[dict] = (), ids: Optional[dict] = None,
              header: str = "", zones: Optional[dict] = None) -> np.ndarray:
    img = frame.copy()
    s = max(0.45, img.shape[1] / 1600)
    lw = max(1, int(round(img.shape[1] / 640)))
    if zones:
        draw_zones(img, zones)
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["xyxy"]]
        c = color_for(d.get("track"))
        cv2.rectangle(img, (x1, y1), (x2, y2), c, lw + 1)
        txt = f"{d['cls']} {d['conf']:.2f}"
        if d.get("track") is not None:
            txt = f"#{d['track']} " + txt
            if ids and ids.get(d["track"]):
                txt += f" [{ids[d['track']]}]"
        label(img, txt, x1, y1, c, s)
    for f in faces:
        x1, y1, x2, y2 = [int(v) for v in f["box"]]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), lw)
        label(img, f"face {f['score']:.2f}", x1, y2 + 16, (0, 140, 140), s * 0.9)
    if header:
        bar = int(28 * s / 0.5)
        cv2.rectangle(img, (0, img.shape[0] - bar), (img.shape[1], img.shape[0]), (0, 0, 0), -1)
        cv2.putText(img, header, (8, img.shape[0] - int(bar * 0.3)), cv2.FONT_HERSHEY_SIMPLEX,
                    s * 1.1, (255, 255, 255), 1, cv2.LINE_AA)
    return img


def contact_sheet(frames: List[np.ndarray], captions: List[str], cols: int = 3, tile_w: int = 480) -> np.ndarray:
    if not frames:
        return np.zeros((10, 10, 3), np.uint8)
    tiles = []
    for f, c in zip(frames, captions):
        h = int(f.shape[0] * tile_w / f.shape[1])
        t = cv2.resize(f, (tile_w, h), interpolation=cv2.INTER_AREA)
        label(t, c, 4, 18, (40, 40, 40), 0.5)
        tiles.append(t)
    th = max(t.shape[0] for t in tiles)
    tiles = [cv2.copyMakeBorder(t, 0, th - t.shape[0], 0, 0, cv2.BORDER_CONSTANT) for t in tiles]
    cols = min(cols, len(tiles))
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    rows = [np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)]
    return np.vstack(rows)


def heatmap_overlay(base_gray_or_bgr: np.ndarray, heat: np.ndarray) -> np.ndarray:
    base = base_gray_or_bgr
    if base.ndim == 2:
        base = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
    hm = cv2.resize(heat, (base.shape[1], base.shape[0]))
    hm = np.uint8(255 * np.sqrt(np.clip(hm, 0, 1)))
    col = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    return cv2.addWeighted(base, 0.55, col, 0.45, 0)


def thumb(img: np.ndarray, w: int = 320) -> np.ndarray:
    h = int(img.shape[0] * w / img.shape[1])
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)

In [ ]:
%%writefile /content/cctv_analysis/storage.py
"""SQLite store: the single source of truth. Makes runs resumable (finished
videos are skipped) and cumulative (reports always cover everything)."""
from __future__ import annotations

import json
import os
import re
import shutil
import sqlite3
from typing import Any, Dict, Iterable, List, Optional


def _inside(out_dir: str, full: str) -> Optional[str]:
    try:
        r = os.path.relpath(os.path.abspath(full), os.path.abspath(out_dir))
    except ValueError:                       # another drive on Windows
        return None
    return None if r == ".." or r.startswith(".." + os.sep) or r.startswith("../") else r.replace(os.sep, "/")


def resolve(out_dir: str, p: Optional[str]) -> str:
    """Where a file the analysis stored is now ('' if nowhere).

    Paths are stored absolute. When the output folder has moved – the external disk got
    another drive letter, the case was copied to another computer – the stored prefix is stale;
    the longest trailing part of the stored path that exists under `out_dir` is the file."""
    if not p:
        return ""
    if not (os.path.isabs(p) or re.match(r"^[A-Za-z]:[\\/]|^\\\\", p)):        # also a Windows path read elsewhere
        full = os.path.join(out_dir, p)
        return full if os.path.exists(full) else ""
    if os.path.exists(p):
        return p
    parts = [x for x in re.split(r"[\\/]+", p) if x]
    for i in range(1, len(parts)):
        cand = os.path.join(out_dir, *parts[i:])
        if os.path.exists(cand):
            return cand
    return ""


def rel_to(out_dir: str, p: Optional[str]) -> str:
    """A path stored in analysis.db as a path relative to the output folder (for the app's file
    route), with '/' separators; '' when missing or outside it. Follows a moved folder."""
    if not p:
        return ""
    r = _inside(out_dir, p if os.path.isabs(p) else os.path.join(out_dir, p))
    if r is None:
        moved = resolve(out_dir, p)
        r = _inside(out_dir, moved) if moved else None
    return r or ""


SCHEMA = """
CREATE TABLE IF NOT EXISTS videos (
  id INTEGER PRIMARY KEY AUTOINCREMENT, path TEXT UNIQUE, rel_path TEXT, fingerprint TEXT,
  camera TEXT, start_dt TEXT, end_dt TEXT, time_source TEXT, duration REAL, fps REAL, width INT,
  height INT, codec TEXT, size_mb REAL, status TEXT, error TEXT, n_events INT, motion_seconds REAL,
  processed_at TEXT, proc_seconds REAL, heatmap_path TEXT, drive_link TEXT, models TEXT,
  lighting_changes INT, osd_masked INT);
CREATE TABLE IF NOT EXISTS events (
  event_id TEXT PRIMARY KEY, video_id INT, video_path TEXT, rel_path TEXT, camera TEXT,
  start_off REAL, end_off REAL, start_dt TEXT, end_dt TEXT, duration REAL, kind TEXT,
  motion_peak REAL, motion_mean REAL, max_persons INT, n_tracks INT, n_person_tracks INT,
  objects TEXT, faces_detected INT, identities TEXT, best_conf REAL, score REAL, level TEXT,
  flags TEXT, semantic TEXT, vlm TEXT, snapshots TEXT, contact_sheet TEXT, clip_path TEXT,
  folder TEXT, after_hours INT, frames_analyzed INT);
CREATE TABLE IF NOT EXISTS tracks (
  event_id TEXT, track INT, cls TEXT, first_off REAL, last_off REAL, first_dt TEXT, duration REAL,
  frames INT, max_conf REAL, mean_conf REAL, max_speed REAL, zones TEXT, identity TEXT,
  identity_sim REAL, known_name TEXT, face_crop TEXT, person_crop TEXT, hidden_face_s REAL,
  posture TEXT, faces_found INT, path TEXT, PRIMARY KEY (event_id, track));
CREATE TABLE IF NOT EXISTS faces (
  id INTEGER PRIMARY KEY AUTOINCREMENT, event_id TEXT, track INT, t_off REAL, dt TEXT, score REAL,
  quality REAL, box TEXT, crop_path TEXT, identity TEXT, similarity REAL, known_name TEXT);
CREATE TABLE IF NOT EXISTS detections (
  event_id TEXT, t_off REAL, dt TEXT, cls TEXT, conf REAL, track INT,
  x1 REAL, y1 REAL, x2 REAL, y2 REAL);
CREATE TABLE IF NOT EXISTS tamper (
  video_id INT, start_off REAL, end_off REAL, start_dt TEXT, kind TEXT);
CREATE TABLE IF NOT EXISTS file_hashes (
  path TEXT PRIMARY KEY, size INT, mtime INT, qhash TEXT);
CREATE TABLE IF NOT EXISTS reel_segments (
  day TEXT, idx INT, event_id TEXT, reel_start REAL, reel_end REAL, seg_path TEXT, reel_path TEXT,
  PRIMARY KEY (day, idx));
CREATE TABLE IF NOT EXISTS face_alerts (
  id INTEGER PRIMARY KEY AUTOINCREMENT, run_id TEXT, kind TEXT, identity TEXT, dt TEXT, camera TEXT,
  event_id TEXT, video TEXT, t_off REAL, face_crop TEXT, person_crop TEXT, note TEXT);
CREATE TABLE IF NOT EXISTS runs (
  run_id TEXT PRIMARY KEY, started TEXT, finished TEXT, videos INT, events INT, new_faces INT,
  duplicates INT, notes TEXT);
CREATE TABLE IF NOT EXISTS plates (
  id INTEGER PRIMARY KEY AUTOINCREMENT, event_id TEXT, track INT, vehicle TEXT, camera TEXT, t_off REAL,
  dt TEXT, text TEXT, confidence REAL, valid_format INT, n_reads INT, reads TEXT, method TEXT,
  crop_path TEXT, enhanced_path TEXT, watchlist TEXT);
CREATE INDEX IF NOT EXISTS ix_ev_video ON events(video_id);
CREATE INDEX IF NOT EXISTS ix_det_ev ON detections(event_id);
CREATE INDEX IF NOT EXISTS ix_ev_start ON events(start_dt);
CREATE INDEX IF NOT EXISTS ix_ev_cam ON events(camera, start_dt);
CREATE INDEX IF NOT EXISTS ix_tr_ident ON tracks(identity);
CREATE INDEX IF NOT EXISTS ix_faces_ev ON faces(event_id);
CREATE INDEX IF NOT EXISTS ix_plates_text ON plates(text);
CREATE INDEX IF NOT EXISTS ix_plates_ev ON plates(event_id);
CREATE INDEX IF NOT EXISTS ix_alerts_kind ON face_alerts(kind, run_id);
"""


class Store:
    def __init__(self, local_path: str, remote_path: str = ""):
        self.local = local_path
        self.remote = remote_path
        os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
        if remote_path and os.path.exists(remote_path) and not os.path.exists(local_path):
            shutil.copy2(remote_path, local_path)
        self.db = sqlite3.connect(local_path)
        self.db.row_factory = sqlite3.Row
        self.db.executescript(SCHEMA)
        self._migrate()
        self.db.execute("CREATE INDEX IF NOT EXISTS ix_ev_run ON events(run_id)")
        self.db.execute("CREATE INDEX IF NOT EXISTS ix_ev_score ON events(score DESC, start_dt)")
        self.db.commit()

    # columns added in later versions – keeps databases from older runs working
    _ADDED = {"videos": {"qhash": "TEXT", "duplicate_of": "TEXT", "run_id": "TEXT"},
              "events": {"run_id": "TEXT", "time_source": "TEXT", "luma_p99": "REAL", "plates": "TEXT",
                         "crossings": "TEXT"},
              "tracks": {"watchlist": "TEXT", "crossings": "TEXT"}}

    def _migrate(self):
        for table, cols in self._ADDED.items():
            have = {r[1] for r in self.db.execute(f"PRAGMA table_info({table})")}
            for c, typ in cols.items():
                if c not in have:
                    self.db.execute(f"ALTER TABLE {table} ADD COLUMN {c} {typ}")

    # ------------------------------------------------------------------ util
    @staticmethod
    def _enc(v):
        if isinstance(v, (dict, list, tuple)):
            return json.dumps(v, default=str)
        return v

    def insert(self, table: str, row: Dict[str, Any], replace=True):
        keys = list(row)
        q = f"INSERT OR {'REPLACE' if replace else 'IGNORE'} INTO {table} ({','.join(keys)}) VALUES ({','.join('?' * len(keys))})"
        cur = self.db.execute(q, [self._enc(row[k]) for k in keys])
        return cur.lastrowid

    def insert_many(self, table: str, rows: Iterable[Dict[str, Any]]):
        rows = list(rows)
        if not rows:
            return
        keys = list(rows[0])
        q = f"INSERT INTO {table} ({','.join(keys)}) VALUES ({','.join('?' * len(keys))})"
        self.db.executemany(q, [[self._enc(r.get(k)) for k in keys] for r in rows])

    def query(self, sql: str, args=()) -> List[dict]:
        return [dict(r) for r in self.db.execute(sql, args).fetchall()]

    # --------------------------------------------------------------- videos
    def cached_hash(self, path: str, size: int, mtime: int):
        r = self.db.execute("SELECT qhash FROM file_hashes WHERE path=? AND size=? AND mtime=?",
                            (path, size, mtime)).fetchone()
        return r["qhash"] if r else None

    def put_hash(self, path: str, size: int, mtime: int, qhash: str):
        self.db.execute("INSERT OR REPLACE INTO file_hashes VALUES (?,?,?,?)", (path, size, mtime, qhash))

    def done_with_hash(self, qhash: str, exclude_path: str = ""):
        r = self.db.execute("SELECT path FROM videos WHERE qhash=? AND status='done' AND path!=?",
                            (qhash, exclude_path)).fetchone()
        return r["path"] if r else None

    def video_done(self, path: str, fingerprint: str) -> bool:
        r = self.db.execute("SELECT status, fingerprint FROM videos WHERE path=?", (path,)).fetchone()
        return bool(r and r["status"] == "done" and r["fingerprint"] == fingerprint)

    purged_events: List[str] = []

    def start_video(self, row: dict) -> int:
        old = self.db.execute("SELECT id FROM videos WHERE path=?", (row["path"],)).fetchone()
        if old:
            vid = old["id"]
            self.purge_video(vid)
            sets = ",".join(f"{k}=?" for k in row)
            self.db.execute(f"UPDATE videos SET {sets} WHERE id=?", [self._enc(v) for v in row.values()] + [vid])
        else:
            vid = self.insert("videos", row)
        self.db.commit()
        return vid

    def purge_video(self, vid: int) -> List[str]:
        """Remove everything derived from one video (before re-analysing it); returns the event ids."""
        evs = [r["event_id"] for r in self.query("SELECT event_id FROM events WHERE video_id=?", (vid,))]
        for t in ("tracks", "faces", "detections", "plates", "face_alerts", "reel_segments"):
            self.db.executemany(f"DELETE FROM {t} WHERE event_id=?", [(e,) for e in evs])
        self.db.execute("DELETE FROM events WHERE video_id=?", (vid,))
        self.db.execute("DELETE FROM tamper WHERE video_id=?", (vid,))
        self.purged_events = evs
        return evs

    def update_video(self, vid: int, **kw):
        sets = ",".join(f"{k}=?" for k in kw)
        self.db.execute(f"UPDATE videos SET {sets} WHERE id=?", [self._enc(v) for v in kw.values()] + [vid])
        self.db.commit()

    def commit(self, sync: bool = True):
        self.db.commit()
        if sync and self.remote:
            try:
                os.makedirs(os.path.dirname(self.remote), exist_ok=True)
                bk = sqlite3.connect(self.remote + ".tmp")
                self.db.backup(bk)
                bk.close()
                os.replace(self.remote + ".tmp", self.remote)
            except Exception:
                pass

    def close(self):
        self.commit()
        self.db.close()

In [ ]:
%%writefile /content/cctv_analysis/reviews.py
"""Reviewer decisions per event (status + notes), kept in <output>/reviews.db.

Separate from analysis.db on purpose: the analyzer replaces analysis.db with its working copy
after every video, so anything written there by another process would be lost. Reports and
the app read reviews from here."""
from __future__ import annotations

import datetime as dt
import os
import sqlite3
from typing import Dict, Optional

STATUSES = ["Not reviewed", "OK / normal", "Suspicious", "Theft confirmed", "Reported", "False alarm"]


class Reviews:
    def __init__(self, output_dir: str):
        self.path = os.path.join(output_dir, "reviews.db")
        self.db = sqlite3.connect(self.path, timeout=10)
        self.db.execute("CREATE TABLE IF NOT EXISTS reviews (event_id TEXT PRIMARY KEY, status TEXT, notes TEXT, "
                        "reviewer TEXT, updated TEXT)")
        self.db.commit()

    def set(self, event_id: str, status: str, notes: str = "", reviewer: str = "") -> dict:
        if status not in STATUSES:
            raise ValueError(f"status must be one of {STATUSES}")
        row = (event_id, status, (notes or "")[:4000], (reviewer or "")[:100],
               dt.datetime.now().astimezone().isoformat(timespec="seconds"))
        with self.db:
            self.db.execute("INSERT INTO reviews VALUES (?,?,?,?,?) ON CONFLICT(event_id) DO UPDATE SET "
                            "status=excluded.status, notes=excluded.notes, reviewer=excluded.reviewer, "
                            "updated=excluded.updated", row)
        return dict(zip(("event_id", "status", "notes", "reviewer", "updated"), row))

    def get(self, event_id: str) -> Optional[dict]:
        r = self.db.execute("SELECT event_id, status, notes, reviewer, updated FROM reviews WHERE event_id=?",
                            (event_id,)).fetchone()
        return dict(zip(("event_id", "status", "notes", "reviewer", "updated"), r)) if r else None

    def all(self) -> Dict[str, dict]:
        return {r[0]: {"status": r[1], "notes": r[2], "reviewer": r[3], "updated": r[4]}
                for r in self.db.execute("SELECT event_id, status, notes, reviewer, updated FROM reviews")}

    def close(self):
        self.db.close()


def load(output_dir: str) -> Dict[str, dict]:
    """All reviews of an output folder ({} when none)."""
    if not os.path.exists(os.path.join(output_dir, "reviews.db")):
        return {}
    r = Reviews(output_dir)
    try:
        return r.all()
    finally:
        r.close()

In [ ]:
%%writefile /content/cctv_analysis/audit.py
"""Append-only, hash-chained audit log (JSON lines). Each entry stores the SHA-256 of the
previous entry, so editing, deleting or reordering any line breaks ``verify()`` from that
line on – the log is tamper-evident (not tamper-proof: keep copies of exported manifests)."""
from __future__ import annotations

import datetime as dt
import getpass
import hashlib
import json
import os
import threading
from typing import Tuple

GENESIS = "0" * 64
_LOCK = threading.Lock()


def _digest(entry: dict) -> str:
    body = {k: v for k, v in entry.items() if k != "hash"}
    return hashlib.sha256(json.dumps(body, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


class AuditLog:
    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)

    def _last_hash(self) -> str:
        if not os.path.exists(self.path):
            return GENESIS
        last = GENESIS
        with open(self.path, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    last = json.loads(line).get("hash", last)
        return last

    def append(self, action: str, **details) -> dict:
        try:
            user = getpass.getuser()
        except Exception:
            user = ""
        with _LOCK:
            entry = {"ts": dt.datetime.now().astimezone().isoformat(timespec="seconds"), "user": user,
                     "action": action, "details": details, "prev": self._last_hash()}
            entry["hash"] = _digest(entry)
            with open(self.path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(entry, sort_keys=True, default=str) + "\n")
        return entry

    def entries(self, limit: int = 500) -> list:
        if not os.path.exists(self.path):
            return []
        with open(self.path, encoding="utf-8") as fh:
            rows = [json.loads(x) for x in fh if x.strip()]
        return rows[-limit:]

    def verify(self) -> Tuple[bool, int, int]:
        """(intact, entries checked, first broken line number or 0)."""
        if not os.path.exists(self.path):
            return True, 0, 0
        prev, n = GENESIS, 0
        with open(self.path, encoding="utf-8") as fh:
            for i, line in enumerate(fh, 1):
                if not line.strip():
                    continue
                try:
                    e = json.loads(line)
                except json.JSONDecodeError:
                    return False, n, i
                if e.get("prev") != prev or _digest(e) != e.get("hash"):
                    return False, n, i
                prev = e["hash"]
                n += 1
        return True, n, 0

In [ ]:
%%writefile /content/cctv_analysis/reels.py
"""Daily highlight reels: every movement of a day (all cameras, all videos) cut out of the
original recordings and joined in true time order.

* Picture is left untouched by default (``reel_overlay=False``): no added text; only scaled /
  letter-boxed to a common size so the parts can be joined.
* Layout: ``highlights/<YYYY>/<MM-Mon>/<YYYY-MM-DD>_highlights.mp4``, plus
  ``<YYYY-MM-DD>_index.csv`` and ``<YYYY-MM-DD>_timeline.srt`` next to it (reel time → camera,
  clock time, source video, offset). The .srt is deliberately *not* named like the video, so
  players do not auto-display it; drag it onto the player to see the clock.
* Order: events with an exact recording time by clock time; events whose time is only
  approximate afterwards, in natural file-name order (never by download time).
* Segments are encoded once and cached under ``highlights/.segments`` keyed by a render
  signature, so a day is cheaply re-assembled when new videos arrive and stale renders
  (e.g. from an older version with burnt-in text) are never reused.
"""
from __future__ import annotations

import csv
import datetime as dt
import glob
import hashlib
import json
import logging
import os
import shutil
import subprocess

from . import timeline as T
from . import video_io as V

log = logging.getLogger("cctv")
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
UNKNOWN = "unknown-date"


def _duration(path: str) -> float:
    try:
        r = subprocess.run([V.FFPROBE, "-v", "error", "-show_entries", "format=duration", "-of", "json", path],
                           capture_output=True, text=True, timeout=60)
        return float(json.loads(r.stdout)["format"]["duration"])
    except Exception:
        return 0.0


def reel_dir(output_dir: str, day: str) -> str:
    """highlights/2026/02-Feb for '2026-02-04'; highlights/unknown-date otherwise."""
    root = os.path.join(output_dir, "highlights")
    try:
        d = dt.date.fromisoformat(day)
    except ValueError:
        return os.path.join(root, UNKNOWN)
    return os.path.join(root, f"{d.year:04d}", f"{d.month:02d}-{MONTHS[d.month - 1]}")


def reel_path(output_dir: str, day: str) -> str:
    return os.path.abspath(os.path.join(reel_dir(output_dir, day), f"{day}_highlights.mp4"))


def _size(cfg):
    W = int(cfg.reel_width) // 2 * 2
    return W, int(round(W * 9 / 16)) // 2 * 2


def _signature(e: dict, cfg) -> str:
    W, H = _size(cfg)
    key = json.dumps([e["video_path"], e.get("fingerprint"), e["start_off"], e["end_off"], W, H, int(cfg.reel_fps),
                      cfg.reel_max_event_seconds, bool(cfg.reel_overlay), e.get("start_dt") if cfg.reel_overlay else ""])
    return hashlib.sha1(key.encode()).hexdigest()[:12]


def _encode_clean(e: dict, out: str, cfg, dur: float) -> bool:
    W, H = _size(cfg)
    vf = (f"scale={W}:{H}:force_original_aspect_ratio=decrease,pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,"
          f"setsar=1,fps={int(cfg.reel_fps)}")
    src = V.decode_path(e["video_path"])
    pre, post = V.seek_args(src, e["start_off"])
    cmd = [V.FFMPEG, "-nostdin", "-y", "-v", "error", *pre, *V._input_args(src), "-i", src, *post,
           "-t", f"{max(0.5, dur):.3f}", "-map", "0:v:0",
           "-an", "-sn", "-vf", vf, *V.h264_args(23), "-video_track_timescale", "15360", out]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    if r.returncode != 0:
        log.warning("reel: ffmpeg failed for %s: %s", e["event_id"], r.stderr[-300:])
    return r.returncode == 0


def _encode_overlay(e: dict, out: str, cfg, dur: float) -> bool:
    """Optional (reel_overlay=True): running clock + citation drawn with OpenCV."""
    import cv2
    import numpy as np
    W, H = _size(cfg)
    fps = int(cfg.reel_fps)
    meta = V.probe(e["video_path"])
    t0 = dt.datetime.strptime(e["start_dt"], "%Y-%m-%d %H:%M:%S") if e.get("start_dt") else None
    info = f"{e['camera']} | {e['level']} {e['score']:g} | {e['event_id']} | {e['rel_path']}"
    sc = W / 1100
    proc = subprocess.Popen([V.FFMPEG, "-nostdin", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
                             "-s", f"{W}x{H}", "-r", str(fps), "-i", "pipe:0", *V.h264_args(23),
                             "-video_track_timescale", "15360", out], stdin=subprocess.PIPE, stderr=subprocess.PIPE)

    def bar(img, text, y, scale, color):
        (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        cv2.rectangle(img, (4, y - th - 8), (12 + tw, y + bl), (0, 0, 0), -1)
        cv2.putText(img, text, (8, y - 4), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)

    n = 0
    try:
        for t, fr in V.iter_frames(e["video_path"], fps, W, e["start_off"], max(0.5, dur),
                                   src_size=(meta["width"], meta["height"])):
            h, w = fr.shape[:2]
            s = min(W / w, H / h)
            img = np.zeros((H, W, 3), np.uint8)
            rs = cv2.resize(fr, (max(2, int(w * s)), max(2, int(h * s))), interpolation=cv2.INTER_AREA)
            y0, x0 = (H - rs.shape[0]) // 2, (W - rs.shape[1]) // 2
            img[y0:y0 + rs.shape[0], x0:x0 + rs.shape[1]] = rs
            clock = (t0 + dt.timedelta(seconds=t - e["start_off"])).strftime("%Y-%m-%d %H:%M:%S") if t0 else ""
            bar(img, f"{clock}   {e['camera']}", int(34 * sc) + 8, 0.9 * sc, (0, 255, 255))
            bar(img, f"{info} @ {V.fmt_offset(t)}", H - 6, 0.55 * sc, (255, 255, 255))
            proc.stdin.write(img.tobytes())
            n += 1
        proc.stdin.close()
        proc.wait(timeout=600)
    except Exception as ex:
        log.warning("reel: encode failed for %s: %s", e["event_id"], ex)
        proc.kill()
        return False
    return proc.returncode == 0 and n > 0


def encode_segment(e: dict, out: str, cfg) -> bool:
    if not os.path.exists(e["video_path"]):
        log.warning("reel: source missing for %s (%s)", e["event_id"], e["video_path"])
        return False
    dur = e["end_off"] - e["start_off"]
    if cfg.reel_max_event_seconds:
        dur = min(dur, cfg.reel_max_event_seconds)
    os.makedirs(os.path.dirname(out), exist_ok=True)
    part = out + ".part.mp4"
    ok = (_encode_overlay if cfg.reel_overlay else _encode_clean)(e, part, cfg, dur)
    if ok and os.path.exists(part) and os.path.getsize(part) > 1000:
        os.replace(part, out)
        return True
    if os.path.exists(part):
        os.remove(part)
    log.warning("reel: could not encode %s", e["event_id"])
    return False


def is_blank(e: dict) -> bool:
    """Nothing detected and the picture centre is black (night clip with no IR, lens covered
    without a tamper flag ...) – not worth watching in a highlight reel."""
    lp = e.get("luma_p99")
    return (e.get("kind") in ("motion-only", "lighting-change") and e.get("level") == "NORMAL"
            and lp is not None and lp < 40)


def events_for_day(store, day: str, scope: str, skip_blank: bool = True) -> list:
    sel = ("SELECT e.*, v.time_source AS video_time_source, v.fingerprint FROM events e "
           "LEFT JOIN videos v ON v.id = e.video_id ")
    if day == UNKNOWN:
        rows = store.query(sel + "WHERE e.start_dt IS NULL")
    else:
        rows = store.query(sel + "WHERE substr(e.start_dt,1,10) = ?", (day,))
    if scope == "people":
        rows = [r for r in rows if r["n_person_tracks"]]
    elif scope == "flagged":
        rows = [r for r in rows if r["level"] in ("HIGH", "REVIEW")]
    if skip_blank:
        rows = [r for r in rows if not is_blank(r)]
    def key(r):
        src = r.get("time_source") or r.get("video_time_source")
        if r["start_dt"] and T.is_exact(src):
            return (0, r["start_dt"], [], r["camera"] or "", r["start_off"])
        return (1, "", V.natural_key(r["rel_path"] or r["video_path"] or ""), r["camera"] or "", r["start_off"])
    return sorted(rows, key=key)


def _srt_time(sec: float) -> str:
    ms = int(round(sec * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def _write_index(reel: str, day: str, parts) -> None:
    base = os.path.join(os.path.dirname(reel), day)
    with open(base + "_index.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["#", "reel_start", "reel_end", "camera", "clock_start", "clock_end", "video", "offset_start",
                    "offset_end", "level", "score", "event_id", "time_source"])
        t = 0.0
        for i, (e, _, d) in enumerate(parts, 1):
            w.writerow([i, V.fmt_offset(t), V.fmt_offset(t + d), e["camera"], e["start_dt"] or "", e["end_dt"] or "",
                        e["rel_path"], V.fmt_offset(e["start_off"]), V.fmt_offset(e["end_off"]), e["level"],
                        e["score"], e["event_id"], e.get("time_source") or e.get("video_time_source") or ""])
            t += d
    with open(base + "_timeline.srt", "w", encoding="utf-8") as fh:
        t = 0.0
        for i, (e, _, d) in enumerate(parts, 1):
            fh.write(f"{i}\n{_srt_time(t)} --> {_srt_time(t + d)}\n"
                     f"{e['start_dt'] or 'time unknown'}  {e['camera']}  {e['rel_path']} @ {V.fmt_offset(e['start_off'])}\n\n")
            t += d


def _publish(tmp: str, final: str) -> str:
    os.makedirs(os.path.dirname(final), exist_ok=True)
    try:
        shutil.copyfile(tmp, final)
    except PermissionError:      # reel open in a player (Windows) -> timestamped copy
        root, ext = os.path.splitext(final)
        final = f"{root}_{dt.datetime.now():%Y%m%d_%H%M%S}{ext}"
        shutil.copyfile(tmp, final)
    os.remove(tmp)
    return final


def remove_day(store, cfg, day: str) -> None:
    """Delete a day's reel + index files (its events are gone or moved to another day)."""
    p = reel_path(cfg.output_dir, day)
    base = os.path.join(os.path.dirname(p), day)
    for f in (p, base + "_index.csv", base + "_timeline.srt"):
        if os.path.exists(f):
            os.remove(f)
    _remove_legacy(cfg, day)
    store.db.execute("DELETE FROM reel_segments WHERE day=?", (day,))


def _remove_legacy(cfg, day: str) -> None:
    """Reels written by earlier versions at highlights/<year>/<day>_highlights.mp4."""
    old = os.path.join(cfg.output_dir, "highlights", day[:4] if day[:4].isdigit() else "unknown",
                       f"{day}_highlights.mp4")
    if os.path.abspath(old) != reel_path(cfg.output_dir, day) and os.path.exists(old):
        os.remove(old)
        log.info("removed old-layout reel %s", old)


def build_day_reel(store, cfg, day: str) -> str:
    if not V.FFMPEG:
        log.warning("ffmpeg not found – highlight reels cannot be built")
        return ""
    events = events_for_day(store, day, cfg.reel_scope, cfg.reel_skip_blank)
    _remove_legacy(cfg, day)
    if not events:
        remove_day(store, cfg, day)
        return ""
    seg_dir = os.path.abspath(os.path.join(cfg.output_dir, "highlights", ".segments"))
    reel = reel_path(cfg.output_dir, day)
    parts = []
    for e in events:
        seg = os.path.join(seg_dir, f"{e['event_id']}-{_signature(e, cfg)}.mp4")
        if not (os.path.exists(seg) and os.path.getsize(seg) > 1000):
            if not encode_segment(e, seg, cfg):
                continue
        d = _duration(seg)
        if d > 0:
            parts.append((e, seg, d))
    if not parts:
        remove_day(store, cfg, day)
        return ""
    os.makedirs(cfg.work_dir, exist_ok=True)
    lst = os.path.abspath(os.path.join(cfg.work_dir, f"reel_{day}.txt"))
    with open(lst, "w", encoding="utf-8") as fh:
        fh.writelines("file '" + p.replace("'", r"'\''") + "'\n" for _, p, _ in parts)
    tmp = os.path.join(cfg.work_dir, f"{day}_highlights.mp4")
    r = subprocess.run([V.FFMPEG, "-nostdin", "-y", "-v", "error", "-f", "concat", "-safe", "0", "-i", lst,
                        "-c", "copy", "-movflags", "+faststart", tmp], capture_output=True, text=True)
    if r.returncode != 0:
        log.warning("reel concat failed for %s: %s", day, r.stderr[-300:])
        return ""
    reel = _publish(tmp, reel)
    _write_index(reel, day, parts)
    store.db.execute("DELETE FROM reel_segments WHERE day=?", (day,))
    t = 0.0
    for i, (e, seg, d) in enumerate(parts, 1):
        store.insert("reel_segments", {"day": day, "idx": i, "event_id": e["event_id"], "reel_start": round(t, 2),
                                       "reel_end": round(t + d, 2), "seg_path": seg, "reel_path": reel})
        t += d
    store.commit(sync=False)
    log.info("Highlight reel %s: %d movement(s), %s long -> %s", day, len(parts), V.fmt_offset(t), reel)
    return reel


def _prune_segments(store, cfg) -> int:
    used = {r["seg_path"] for r in store.query("SELECT seg_path FROM reel_segments")}
    n = 0
    for f in glob.glob(os.path.join(os.path.abspath(cfg.output_dir), "highlights", ".segments", "*.mp4")):
        if f not in used:
            os.remove(f)
            n += 1
    return n


def build_reels(store, cfg, days=None) -> list:
    """(Re)build reels for `days` (default: every day with events but no reel yet). Days that have a
    reel but no longer any events (times corrected, videos removed) lose their reel."""
    if not cfg.make_highlight_reel:
        return []
    all_days = {(r["d"] or UNKNOWN) for r in store.query("SELECT DISTINCT substr(start_dt,1,10) d FROM events")}
    have = {r["day"] for r in store.query("SELECT DISTINCT day FROM reel_segments")}
    for d in have - all_days:
        remove_day(store, cfg, d)
        log.info("removed highlight reel of %s (no events left on that day)", d)
    if days is None:
        days = all_days - have
    out = []
    for d in sorted(set(days) & all_days):
        try:
            p = build_day_reel(store, cfg, d)
            if p:
                out.append(p)
        except Exception as e:
            log.exception("reel for %s failed: %s", d, e)
    n = _prune_segments(store, cfg)
    if n:
        log.info("pruned %d unused reel segment(s)", n)
    store.commit(sync=False)
    return out

In [ ]:
%%writefile /content/cctv_analysis/report.py
"""Reports built from the SQLite store: Excel workbook (with thumbnails,
clickable citations, review columns, charts), HTML dashboard, PNG charts and a
highlight reel of the most suspicious events."""
from __future__ import annotations

import base64
import csv
import datetime as dt
import html
import json
import logging
import os
import re
from collections import Counter, defaultdict

import cv2

from . import video_io as V
from .rules import WEIGHTS

log = logging.getLogger("cctv")
LEVEL_COLORS = {"HIGH": "#F8696B", "REVIEW": "#FFB347", "LOW": "#FFF2A8", "NORMAL": "#E2EFDA"}
EXCEL_MAX_ROWS = 1_000_000


def _j(s, default=None):
    if s in (None, ""):
        return default
    try:
        return json.loads(s)
    except Exception:
        return default


def _rel(path: str, base: str) -> str:
    if not path:
        return ""
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path


def _thumb(src: str, out_dir: str, w: int = 200) -> str:
    if not src or not os.path.exists(src):
        return ""
    dst = os.path.join(out_dir, "thumbs", os.path.relpath(src, out_dir).replace(os.sep, "__"))
    dst = os.path.splitext(dst)[0] + ".jpg"
    if os.path.exists(dst):
        return dst
    img = cv2.imread(src)
    if img is None:
        return ""
    h = max(1, int(img.shape[0] * w / img.shape[1]))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    cv2.imwrite(dst, cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 80])
    return dst


def _flags_text(flags) -> str:
    return "; ".join(f"{f['flag']} (+{f['points']:g}): {f['reason']}" for f in flags or [])


def _vlm_text(v) -> str:
    if not v:
        return ""
    d = v.get("description") or ""
    if v.get("suspicious"):
        d = f"[SUSPICIOUS: {v.get('category') or 'yes'}] " + d + (f" — {v['reason']}" if v.get("reason") else "")
    return d


def _range_sql(start, end, col="start_dt"):
    """SQL filter for a [start, end] day range (YYYY-MM-DD, inclusive). None = all-time."""
    if not start:
        return "", ()
    return f" WHERE substr({col},1,10) BETWEEN ? AND ?", (start, end)


def load(store, start=None, end=None):
    w, a = _range_sql(start, end)
    events = store.query("SELECT * FROM events" + w + " ORDER BY COALESCE(start_dt, ''), camera, start_off", a)
    for e in events:
        for k in ("flags", "semantic", "vlm", "snapshots", "objects", "identities", "plates", "crossings"):
            e[k] = _j(e.get(k), [] if k not in ("vlm", "objects", "crossings") else (None if k == "vlm" else {}))
    videos = store.query("SELECT * FROM videos ORDER BY camera, start_dt, path")
    tracks = store.query("SELECT t.*, e.camera, e.video_path, e.rel_path, e.start_dt AS ev_dt FROM tracks t "
                         "JOIN events e USING(event_id)" + w.replace("start_dt", "e.start_dt") +
                         " ORDER BY t.first_dt, t.event_id, t.track", a)
    faces = store.query("SELECT f.*, e.camera, e.rel_path, e.video_path FROM faces f JOIN events e USING(event_id)"
                        + w.replace("start_dt", "e.start_dt") + " ORDER BY f.dt, f.event_id", a)
    tamper = store.query("SELECT t.*, v.rel_path, v.camera FROM tamper t JOIN videos v ON v.id=t.video_id"
                         + w.replace("start_dt", "t.start_dt") + " ORDER BY t.start_dt", a)
    if start:
        ids = {e["video_id"] for e in events}
        videos = [v for v in videos if v["id"] in ids or (v["start_dt"] or "")[:10] and start <= (v["start_dt"] or "")[:10] <= end]
    return events, videos, tracks, faces, tamper


def hours_text(cfg) -> str:
    from .rules import hours_enabled
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    closed = ", ".join(days[d] for d in cfg.closed_weekdays or [] if 0 <= d < 7)
    h = f"{cfg.open_hour}–{cfg.close_hour}" if hours_enabled(cfg) else "not set (open 24 h – no after-hours flag)"
    return h + (f"; closed on {closed}" if closed else "")


def identities_table(tracks):
    ids = defaultdict(lambda: {"tracks": 0, "events": set(), "cameras": set(), "first": None, "last": None,
                               "name": "", "sample": "", "best_sim": 0})
    for t in tracks:
        if not t["identity"]:
            continue
        r = ids[t["identity"]]
        r["tracks"] += 1
        r["events"].add(t["event_id"])
        r["cameras"].add(t["camera"])
        when = t["first_dt"] or ""
        r["first"] = min(filter(None, [r["first"], when]), default=when)
        r["last"] = max(filter(None, [r["last"], when]), default=when)
        if t["known_name"]:
            r["name"] = t["known_name"]
        if t["face_crop"] and (t["identity_sim"] or 0) >= r["best_sim"]:
            r["sample"], r["best_sim"] = t["face_crop"], t["identity_sim"] or 0
    return dict(sorted(ids.items(), key=lambda kv: -len(kv[1]["events"])))


# ====================================================================== Excel
def build_excel(store, cfg, path, start=None, end=None, link_base=None, title="CCTV Analysis Report"):
    """Workbook for all-time (start=None) or for the day range [start, end].
    `link_base` = folder the final workbook will live in (relative file links)."""
    import xlsxwriter
    out = link_base or cfg.output_dir
    events, videos, tracks, faces, tamper = load(store, start, end)
    idt = identities_table(tracks)
    from .reviews import load as load_reviews
    reviews = load_reviews(cfg.output_dir)
    wb = xlsxwriter.Workbook(path, {"strings_to_urls": False, "nan_inf_to_errors": True})
    F = {
        "h": wb.add_format({"bold": True, "bg_color": "#1F3864", "font_color": "white", "text_wrap": True,
                            "valign": "vcenter", "border": 1}),
        "title": wb.add_format({"bold": True, "font_size": 16, "font_color": "#1F3864"}),
        "b": wb.add_format({"bold": True}),
        "wrap": wb.add_format({"text_wrap": True, "valign": "top"}),
        "top": wb.add_format({"valign": "top"}),
        "link": wb.add_format({"font_color": "blue", "underline": 1, "valign": "top"}),
        "num": wb.add_format({"num_format": "0.00", "valign": "top"}),
        "int": wb.add_format({"num_format": "0", "valign": "top"}),
        "pct": wb.add_format({"num_format": "0.0%", "valign": "top"}),
    }
    for lv, c in LEVEL_COLORS.items():
        F[lv] = wb.add_format({"bg_color": c, "bold": True, "valign": "top", "align": "center"})

    def link(ws, r, c, target, text):
        if target:
            ws.write_url(r, c, target if target.startswith("http") else "external:" + target, F["link"], text)
        else:
            ws.write_blank(r, c, None)

    thumbs_used = [0]

    def image(ws, r, c, src, w=150):
        if not cfg.excel_thumbnails or thumbs_used[0] >= cfg.excel_max_thumbnails:
            return
        try:                       # a broken/missing image must never break the report
            t = _thumb(src, cfg.output_dir, 200)
            if t:
                img = cv2.imread(t)
                if img is None:
                    return
                sc = w / img.shape[1]
                ws.insert_image(r, c, t, {"x_scale": sc, "y_scale": sc, "x_offset": 2, "y_offset": 2,
                                          "object_position": 1})
                thumbs_used[0] += 1
        except Exception as ex:
            log.debug("thumbnail skipped for %s: %s", src, ex)

    # ---------------------------------------------------------- Read Me
    ws = wb.add_worksheet("Read Me")
    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 110, F["wrap"])
    ws.write(0, 0, title, F["title"])
    ws.write(1, 0, "Generated")
    ws.write(1, 1, dt.datetime.now().strftime("%Y-%m-%d %H:%M") +
             (f"  ·  covers {start} → {end}" if start else "  ·  covers all analysed footage"))
    rows = [
        ("How to review", "Start with the 'Events' sheet. It is sorted by time; filter the Level column to HIGH/REVIEW "
                          "or sort by Score. Each row cites the exact video file, the offset inside the video "
                          "(HH:MM:SS) and the real clock time. Click 'Clip' to watch the event, 'Snapshot' or "
                          "'Contact sheet' for stills, 'Video (Drive)' to open the original recording."),
        ("Links", "File links are relative to this workbook. They work when the workbook is opened from the "
                  "output folder (Google Drive for desktop, or after downloading the whole folder). "
                  "Drive links open in the browser."),
        ("Seek in VLC", "Open the original file in VLC and jump to 'Video offset start' (Ctrl+T)."),
        ("Score / Level", "Score 0-100 is the sum of explainable flags (see 'Flags'). HIGH ≥ 70, REVIEW ≥ "
                          f"{cfg.suspicion_alert_threshold:g}, LOW > 0, NORMAL = 0. It is a triage aid, not proof."),
        ("Identities", "P#### are anonymous face clusters that persist between runs (same person ≈ same ID). "
                       "Put photos in the known-faces folder (one sub-folder per person) to label staff/owner."),
        ("Review columns", "'Review status' and 'Notes' are for you; they are kept only in this file."),
        ("Shop hours", hours_text(cfg) + f". Timezone {cfg.timezone}."),
        ("Highlights", "highlights/<year>/<MM-Mon>/<date>_highlights.mp4 holds every movement of a day in time order, "
                       "with the original picture (no added text). The 'Highlights' sheet (and <date>_index.csv / "
                       "<date>_timeline.srt next to the video) map reel time → camera, clock time, video + offset."),
        ("Recording time", "'Time source' says where each time came from: filename, osd(...) = the DVR clock read "
                           "from the picture, metadata, or file-modified(approx). Approximate times are marked."),
        ("Number plates", "The 'Number Plates' sheet lists every plate read, with confidence, the plate image and "
                          "(when enhancement is on) the enhanced image. Check low-confidence reads by eye."),
        ("Weekly files", "reports/<year>/<dd-dd-Mon-yyyy-W##>.xlsx contain one week each; CCTV_Report.xlsx has "
                         "everything. Unique_Faces.xlsx lists every distinct face with NEW ones highlighted."),
    ]
    for i, (a, b) in enumerate(rows, 3):
        ws.write(i, 0, a, F["b"])
        ws.write(i, 1, b)
    r0 = len(rows) + 4
    ws.write(r0, 0, "Flag", F["h"])
    ws.write(r0, 1, "Points (max) / meaning", F["h"])
    meaning = {
        "after_hours": "movement/people outside opening hours", "camera_tamper": "lens covered, blackout, blur or camera moved",
        "weapon_like_object": "knife / scissors / bat detected", "restricted_zone": "person inside a configured zone (e.g. cash counter)",
        "loitering": f"person stayed ≥ {cfg.loiter_seconds:g}s", "running": "very fast movement",
        "crowd": f"≥ {cfg.crowd_size} people at once", "face_hidden": "person facing camera but face not visible (mask/helmet)",
        "hands_up": "both hands above shoulders", "crouching": "crouching / bending low",
        "lights_after_hours": "lights switched on/off while the shop is closed", "semantic_match": "SigLIP-2 image-text match with a suspicious description",
        "vlm_suspicious": "the vision-language model judged the event suspicious",
        "unknown_person_after_hours": "unrecognised face after hours", "repeat_visitor": "same unknown face in ≥ 5 events",
        "long_event": "event ≥ 5 min with people",
        "watchlist_face": "a person from the face watchlist folder", "watchlist_plate": "a plate on the plate watchlist",
        "unauthorized_person": "person who is not a known face on an authorised-only camera"}
    for i, (k, v) in enumerate(WEIGHTS.items(), r0 + 1):
        ws.write(i, 0, k)
        ws.write(i, 1, f"{v} — {meaning.get(k, '')}")

    # ---------------------------------------------------------- Summary
    ws = wb.add_worksheet("Summary")
    ws.set_column(0, 0, 34)
    ws.set_column(1, 8, 16)
    ws.write(0, 0, "Summary", F["title"])
    tot_dur = sum(v["duration"] or 0 for v in videos)
    tot_motion = sum(v["motion_seconds"] or 0 for v in videos)
    lv = Counter(e["level"] for e in events)
    kpis = [
        ("Videos analysed", sum(1 for v in videos if v["status"] == "done")),
        ("Videos failed", sum(1 for v in videos if v["status"] not in ("done",))),
        ("Footage analysed (hours)", round(tot_dur / 3600, 2)),
        ("Motion time (hours)", round(tot_motion / 3600, 2)),
        ("Footage you can skip", f"{(1 - tot_motion / tot_dur) * 100:.1f}%" if tot_dur else "-"),
        ("Events", len(events)),
        ("Events with people", sum(1 for e in events if e["n_person_tracks"])),
        ("HIGH", lv.get("HIGH", 0)), ("REVIEW", lv.get("REVIEW", 0)), ("LOW", lv.get("LOW", 0)),
        ("NORMAL", lv.get("NORMAL", 0)),
        ("Person tracks", sum(1 for t in tracks if t["cls"] == "person")),
        ("Face samples", len(faces)),
        ("Distinct face identities", len(idt)),
        ("Tamper incidents", len(tamper)),
        ("Date range", f"{min((e['start_dt'] for e in events if e['start_dt']), default='-')} → "
                       f"{max((e['end_dt'] for e in events if e['end_dt']), default='-')}"),
    ]
    for i, (a, b) in enumerate(kpis, 2):
        ws.write(i, 0, a, F["b"])
        ws.write(i, 1, b)
    r = len(kpis) + 3
    ws.write(r, 0, "Top suspicious events", F["b"])
    hdr = ["Event ID", "Level", "Score", "Camera", "Start", "Video", "Offset", "Main reasons"]
    for c, h in enumerate(hdr):
        ws.write(r + 1, c, h, F["h"])
    top = sorted(events, key=lambda e: -(e["score"] or 0))[:25]
    for i, e in enumerate(top, r + 2):
        ws.write_url(i, 0, f"internal:'Events'!A{events.index(e) + 2}", F["link"], e["event_id"])
        ws.write(i, 1, e["level"], F.get(e["level"]))
        ws.write(i, 2, e["score"])
        ws.write(i, 3, e["camera"])
        ws.write(i, 4, e["start_dt"] or "")
        ws.write(i, 5, e["rel_path"])
        ws.write(i, 6, V.fmt_offset(e["start_off"]))
        ws.write(i, 7, ", ".join(f["flag"] for f in e["flags"]))
    ws.set_column(7, 7, 60)
    # hourly table + native chart
    hr = r + 3 + len(top) + 1
    ws.write(hr, 0, "Events by hour of day", F["b"])
    ws.write(hr + 1, 0, "Hour", F["h"])
    ws.write(hr + 1, 1, "Events", F["h"])
    ws.write(hr + 1, 2, "With people", F["h"])
    ws.write(hr + 1, 3, "REVIEW+HIGH", F["h"])
    byh = defaultdict(lambda: [0, 0, 0])
    for e in events:
        if e["start_dt"]:
            h = int(e["start_dt"][11:13])
            byh[h][0] += 1
            byh[h][1] += bool(e["n_person_tracks"])
            byh[h][2] += e["level"] in ("HIGH", "REVIEW")
    for h in range(24):
        ws.write(hr + 2 + h, 0, f"{h:02d}:00")
        for c in range(3):
            ws.write(hr + 2 + h, 1 + c, byh[h][c])
    ch = wb.add_chart({"type": "column"})
    for c, name in ((1, "Events"), (3, "REVIEW+HIGH")):
        ch.add_series({"name": name, "categories": ["Summary", hr + 2, 0, hr + 25, 0],
                       "values": ["Summary", hr + 2, c, hr + 25, c]})
    ch.set_title({"name": "Activity by hour"})
    ch.set_size({"width": 720, "height": 300})
    ws.insert_chart(2, 3, ch)
    # per day per camera
    byd = defaultdict(Counter)
    for e in events:
        byd[(e["start_dt"] or "unknown")[:10]][e["camera"]] += 1
    cams = sorted({e["camera"] for e in events})
    dr = hr + 28
    ws.write(dr, 0, "Events per day and camera", F["b"])
    ws.write(dr + 1, 0, "Date", F["h"])
    for c, cam in enumerate(cams, 1):
        ws.write(dr + 1, c, cam, F["h"])
    for i, (d, cnt) in enumerate(sorted(byd.items()), dr + 2):
        ws.write(i, 0, d)
        for c, cam in enumerate(cams, 1):
            ws.write(i, c, cnt.get(cam, 0))

    # ---------------------------------------------------------- Events
    ws = wb.add_worksheet("Events")
    cols = [("Event ID", 26), ("Level", 9), ("Score", 7), ("Camera", 10), ("Date", 11), ("Start time", 10),
            ("End time", 10), ("Duration (s)", 9), ("Video file", 34), ("Video offset start", 10),
            ("Video offset end", 10), ("Seek (s)", 8), ("Kind", 11), ("Max people at once", 8),
            ("Person tracks", 8), ("Objects (unique tracks)", 26), ("Faces detected", 8), ("Identities", 18),
            ("Best detection conf", 9), ("Flags / reasons", 60), ("AI description (VLM)", 60),
            ("Scene tag (SigLIP-2)", 34), ("Snapshot", 24), ("Snapshot file", 12), ("Contact sheet", 12),
            ("Clip", 8), ("Video (Drive)", 10), ("Event folder", 12), ("After hours", 7), ("Motion peak", 8),
            ("Frames analysed", 8), ("Time source", 14), ("Number plates", 16), ("Review status", 14), ("Notes", 30)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    vmap = {v["id"]: v for v in videos}
    snap_col = [c for c, (h, _) in enumerate(cols) if h == "Snapshot"][0]
    for i, e in enumerate(events, 1):
        v = vmap.get(e["video_id"], {})
        snaps = e["snapshots"] or []
        best = max(snaps, key=lambda s: s.get("persons", 0)) if snaps else {}
        sem = max(e["semantic"] or [{}], key=lambda s: s.get("suspicious_score", 0)) if e["semantic"] else {}
        row = [e["event_id"], e["level"], e["score"], e["camera"], (e["start_dt"] or "")[:10],
               (e["start_dt"] or "")[11:], (e["end_dt"] or "")[11:], e["duration"], e["rel_path"],
               V.fmt_offset(e["start_off"]), V.fmt_offset(e["end_off"]), round(e["start_off"], 1), e["kind"],
               e["max_persons"], e["n_person_tracks"],
               ", ".join(f"{k}×{n}" for k, n in sorted((e["objects"] or {}).items(), key=lambda kv: -kv[1])),
               e["faces_detected"], ", ".join(e["identities"] or []), e["best_conf"], _flags_text(e["flags"]),
               _vlm_text(e["vlm"]),
               f"{sem.get('top_label', '')} ({sem.get('top_score', 0):.2f})" if sem else ""]
        for c, val in enumerate(row):
            if c == 1:
                ws.write(i, c, val, F.get(val, F["top"]))
            elif c in (19, 20):
                ws.write(i, c, val, F["wrap"])
            else:
                ws.write(i, c, val, F["top"])
        c = snap_col
        image(ws, i, c, best.get("annotated", ""), 160)
        link(ws, i, c + 1, _rel(best.get("annotated", ""), out), "open")
        link(ws, i, c + 2, _rel(e["contact_sheet"], out), "sheet")
        link(ws, i, c + 3, _rel(e["clip_path"], out), "clip" if e["clip_path"] else "")
        link(ws, i, c + 4, v.get("drive_link") or "", "drive" if v.get("drive_link") else "")
        link(ws, i, c + 5, _rel(e["folder"], out), "folder")
        ws.write(i, c + 6, "yes" if e["after_hours"] else "")
        ws.write(i, c + 7, e["motion_peak"])
        ws.write(i, c + 8, e["frames_analyzed"])
        ws.write(i, c + 9, e.get("time_source") or v.get("time_source", ""))
        ws.write(i, c + 10, ", ".join(e.get("plates") or []))
        rv = reviews.get(e["event_id"])
        if rv:
            ws.write(i, c + 11, rv["status"])
            ws.write(i, c + 12, rv["notes"] or "", F["wrap"])
        ws.set_row(i, 92)
    n = len(events)
    if n:
        ws.autofilter(0, 0, n, len(cols) - 1)
        ws.conditional_format(1, 2, n, 2, {"type": "3_color_scale", "min_color": "#63BE7B",
                                           "mid_color": "#FFEB84", "max_color": "#F8696B"})
        ws.data_validation(1, len(cols) - 2, n, len(cols) - 2, {
            "validate": "list", "source": ["Not reviewed", "OK / normal", "Suspicious", "Theft confirmed",
                                           "Reported", "False alarm"]})
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- People / tracks
    ws = wb.add_worksheet("People & Objects")
    cols = [("Event ID", 26), ("Track", 6), ("Class", 10), ("Camera", 9), ("First seen", 19), ("Seen for (s)", 8),
            ("Video file", 30), ("Video offset", 10), ("Max conf", 7), ("Mean conf", 7), ("Max speed (body-h/s)", 9),
            ("Zones (s)", 18), ("Identity", 9), ("Known name", 12), ("Match similarity", 8), ("Faces found", 7),
            ("Face hidden (s)", 8), ("Posture (s)", 26), ("Person crop", 14), ("Face", 12), ("Crop file", 8),
            ("Face file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, t in enumerate(tracks[:EXCEL_MAX_ROWS - 1], 1):
        z = _j(t["zones"], {}) or {}
        p = _j(t["posture"], {}) or {}
        vals = [t["event_id"], t["track"], t["cls"], t["camera"], t["first_dt"] or "", t["duration"], t["rel_path"],
                V.fmt_offset(t["first_off"]), t["max_conf"], t["mean_conf"], t["max_speed"],
                ", ".join(f"{k}: {v:g}" for k, v in z.items()), t["identity"] or "", t["known_name"] or "",
                t["identity_sim"], t["faces_found"], t["hidden_face_s"],
                ", ".join(f"{k}={v:g}" for k, v in p.items() if v)]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        big = i <= cfg.excel_max_thumbnails and (t["person_crop"] or t["face_crop"])
        if big:
            image(ws, i, 18, t["person_crop"], 60)
            image(ws, i, 19, t["face_crop"], 60)
            ws.set_row(i, 90)
        link(ws, i, 20, _rel(t["person_crop"], out), "crop" if t["person_crop"] else "")
        link(ws, i, 21, _rel(t["face_crop"], out), "face" if t["face_crop"] else "")
    if tracks:
        ws.autofilter(0, 0, min(len(tracks), EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Identities
    ws = wb.add_worksheet("Face Identities")
    cols = [("Identity", 10), ("Known name", 14), ("Sample face", 14), ("Events", 8), ("Appearances", 8),
            ("Cameras", 14), ("First seen", 19), ("Last seen", 19), ("Event IDs", 80)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, (pid, r_) in enumerate(idt.items(), 1):
        vals = [pid, r_["name"], "", len(r_["events"]), r_["tracks"], ", ".join(sorted(r_["cameras"])),
                r_["first"], r_["last"], ", ".join(sorted(r_["events"]))]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["wrap"] if c == 8 else F["top"])
        image(ws, i, 2, r_["sample"], 80)
        ws.set_row(i, 85)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Faces
    ws = wb.add_worksheet("Faces")
    cols = [("Event ID", 26), ("Track", 6), ("Time", 19), ("Camera", 9), ("Video file", 30), ("Video offset", 10),
            ("Detector score", 8), ("Quality", 7), ("Identity", 9), ("Known name", 12), ("Similarity", 8),
            ("Face", 12), ("Face file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, f in enumerate(faces[:EXCEL_MAX_ROWS - 1], 1):
        vals = [f["event_id"], f["track"], f["dt"] or "", f["camera"], f["rel_path"], V.fmt_offset(f["t_off"]),
                f["score"], f["quality"], f["identity"] or "", f["known_name"] or "", f["similarity"]]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        if i <= 1500:
            image(ws, i, 11, f["crop_path"], 60)
            ws.set_row(i, 70)
        link(ws, i, 12, _rel(f["crop_path"], out), "face")
    if faces:
        ws.autofilter(0, 0, min(len(faces), EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Detections
    ws = wb.add_worksheet("Detections (1 per s)")
    cols = ["Event ID", "Video offset (s)", "Time", "Class", "Confidence", "Track", "x1", "y1", "x2", "y2"]
    for c, h in enumerate(cols):
        ws.write(0, c, h, F["h"])
    ws.set_column(0, 0, 26)
    ws.set_column(2, 2, 19)
    w_, a_ = _range_sql(start, end, "dt")
    cur = store.db.execute("SELECT event_id,t_off,dt,cls,conf,track,x1,y1,x2,y2 FROM detections" + w_ +
                           " ORDER BY dt, event_id, t_off", a_)
    csv_path = os.path.join(cfg.work_dir if start else cfg.output_dir, "detections_full.csv")
    i = 0
    with open(csv_path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(cols)
        for row in cur:
            row = list(row)
            w.writerow(row)
            i += 1
            if i < EXCEL_MAX_ROWS:
                ws.write_row(i, 0, row)
    if i:
        ws.autofilter(0, 0, min(i, EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 0)

    # ---------------------------------------------------------- Videos
    ws = wb.add_worksheet("Videos")
    cols = [("Video file", 40), ("Camera", 9), ("Recording start", 19), ("Recording end", 19), ("Time source", 16),
            ("Duration", 10), ("Resolution", 11), ("FPS", 6), ("Codec", 8), ("Size (MB)", 9), ("Events", 7),
            ("Motion (s)", 9), ("Motion %", 8), ("Lighting changes", 8), ("Clock masked", 7),
            ("Processing (s)", 9), ("Speed (x real-time)", 9), ("Status", 8), ("Error", 40), ("Heatmap", 10),
            ("Drive link", 10), ("Models", 40)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, v in enumerate(videos, 1):
        d = v["duration"] or 0
        vals = [v["rel_path"], v["camera"], v["start_dt"] or "", v["end_dt"] or "", v["time_source"],
                V.fmt_offset(d), f"{v['width']}x{v['height']}", round(v["fps"] or 0, 2), v["codec"],
                round(v["size_mb"] or 0, 1), v["n_events"], round(v["motion_seconds"] or 0, 1)]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        ws.write(i, 12, (v["motion_seconds"] or 0) / d if d else 0, F["pct"])
        ws.write(i, 13, v["lighting_changes"])
        ws.write(i, 14, "yes" if v["osd_masked"] else "")
        ws.write(i, 15, round(v["proc_seconds"] or 0, 1))
        ws.write(i, 16, round(d / v["proc_seconds"], 1) if v["proc_seconds"] else "")
        ws.write(i, 17, v["status"])
        ws.write(i, 18, v["error"] or "", F["wrap"])
        link(ws, i, 19, _rel(v["heatmap_path"], out), "heatmap" if v["heatmap_path"] else "")
        link(ws, i, 20, v["drive_link"] or "", "drive" if v["drive_link"] else "")
        ws.write(i, 21, v["models"] or "")
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Tamper
    ws = wb.add_worksheet("Tamper & Camera")
    cols = [("Video file", 40), ("Camera", 9), ("Start", 19), ("Video offset", 10), ("Duration (s)", 9), ("Kind", 30)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, t in enumerate(tamper, 1):
        ws.write_row(i, 0, [t["rel_path"], t["camera"], t["start_dt"] or "", V.fmt_offset(t["start_off"]),
                            round(t["end_off"] - t["start_off"], 1), t["kind"]])

    # ---------------------------------------------------------- Hourly heat table
    ws = wb.add_worksheet("Activity Grid")
    ws.write(0, 0, "People-events per date × hour (colour = busier)", F["b"])
    ws.write(1, 0, "Date", F["h"])
    for h in range(24):
        ws.write(1, 1 + h, f"{h:02d}", F["h"])
    grid = defaultdict(Counter)
    for e in events:
        if e["start_dt"] and e["n_person_tracks"]:
            grid[e["start_dt"][:10]][int(e["start_dt"][11:13])] += 1
    for i, d in enumerate(sorted(grid), 2):
        ws.write(i, 0, d)
        for h in range(24):
            ws.write(i, 1 + h, grid[d].get(h, 0))
    if grid:
        ws.conditional_format(2, 1, 1 + len(grid), 24, {"type": "2_color_scale", "min_color": "#FFFFFF",
                                                         "max_color": "#F8696B"})
    ws.set_column(0, 0, 12)
    ws.set_column(1, 24, 4)

    # ---------------------------------------------------------- Highlights
    ws = wb.add_worksheet("Highlights")
    cols = [("Day", 11), ("Reel", 10), ("#", 5), ("Reel time start", 10), ("Reel time end", 10), ("Event ID", 26),
            ("Camera", 9), ("Clock start", 10), ("Clock end", 10), ("Video file", 34), ("Video offset start", 10),
            ("Video offset end", 10), ("Level", 9), ("Score", 7), ("People", 7), ("Reasons", 60)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    w_, a_ = _range_sql(start, end, "r.day")
    segs = store.query("SELECT r.*, e.camera, e.start_dt, e.end_dt, e.rel_path, e.start_off, e.end_off, e.level, "
                       "e.score, e.max_persons, e.flags FROM reel_segments r JOIN events e USING(event_id)" + w_ +
                       " ORDER BY r.day, r.idx", a_)
    for i, g in enumerate(segs, 1):
        ws.write(i, 0, g["day"])
        link(ws, i, 1, _rel(g["reel_path"], out), "play reel")
        ws.write_row(i, 2, [g["idx"], V.fmt_offset(g["reel_start"]), V.fmt_offset(g["reel_end"]), g["event_id"],
                            g["camera"], (g["start_dt"] or "")[11:], (g["end_dt"] or "")[11:], g["rel_path"],
                            V.fmt_offset(g["start_off"]), V.fmt_offset(g["end_off"])])
        ws.write(i, 12, g["level"], F.get(g["level"], F["top"]))
        ws.write(i, 13, g["score"])
        ws.write(i, 14, g["max_persons"])
        ws.write(i, 15, ", ".join(f["flag"] for f in _j(g["flags"], []) or []))
    if segs:
        ws.autofilter(0, 0, len(segs), len(cols) - 1)
    ws.freeze_panes(1, 0)

    # ---------------------------------------------------------- Number plates
    ws = wb.add_worksheet("Number Plates")
    cols = [("Plate", 14), ("Confidence", 9), ("Format OK", 8), ("Time", 19), ("Camera", 10), ("Vehicle", 9),
            ("Event ID", 26), ("Video file", 34), ("Video offset", 10), ("Watchlist match", 14), ("Method", 16),
            ("Reads", 6), ("Plate image", 22), ("Enhanced image", 22), ("Plate file", 8), ("Enhanced file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    w_, a_ = _range_sql(start, end, "p.dt")
    plates = store.query("SELECT p.*, e.rel_path FROM plates p JOIN events e USING(event_id)" + w_ +
                         (" AND" if w_ else " WHERE") + " p.text != '' ORDER BY p.dt", a_)
    for i, p_ in enumerate(plates, 1):
        ws.write_row(i, 0, [p_["text"], p_["confidence"], "yes" if p_["valid_format"] else "no", p_["dt"] or "",
                            p_["camera"], p_["vehicle"], p_["event_id"], p_["rel_path"], V.fmt_offset(p_["t_off"] or 0),
                            p_["watchlist"] or "", p_["method"], p_["n_reads"]], F["top"])
        image(ws, i, 12, p_["crop_path"], 150)
        image(ws, i, 13, p_["enhanced_path"], 150)
        link(ws, i, 14, _rel(p_["crop_path"], out), "plate" if p_["crop_path"] else "")
        link(ws, i, 15, _rel(p_["enhanced_path"], out), "enhanced" if p_["enhanced_path"] else "")
        ws.set_row(i, 48)
    if plates:
        ws.autofilter(0, 0, len(plates), len(cols) - 1)
    else:
        ws.write(1, 0, "No plates (enable 'Read vehicle number plates' / use_plates).")
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Footfall
    ws = wb.add_worksheet("Footfall")
    ws.write(0, 0, "Counting-line crossings per hour (the footfall count) and person tracks. A person can be split "
                   "into several tracks (occlusion, low frame rate), so 'Person tracks' is an upper bound, not a head "
                   "count – use counting lines (count_lines) for entrances.", F["b"])
    hdr = ["Date", "Hour", "Camera", "Person tracks", "Events with people"]
    lines = sorted({ln for e in events for ln in (e.get("crossings") or {})})
    for ln in lines:
        hdr += [f"{ln} in", f"{ln} out"]
    for c, h in enumerate(hdr):
        ws.write(1, c, h, F["h"])
    ff = defaultdict(lambda: {"people": 0, "events": 0, "lines": defaultdict(lambda: [0, 0])})
    for t in tracks:
        if t["cls"] == "person" and t["first_dt"]:
            ff[(t["first_dt"][:10], t["first_dt"][11:13], t["camera"])]["people"] += 1
    for e in events:
        if e["start_dt"]:
            k = (e["start_dt"][:10], e["start_dt"][11:13], e["camera"])
            ff[k]["events"] += bool(e["n_person_tracks"])
            for ln, c_ in (e.get("crossings") or {}).items():
                ff[k]["lines"][ln][0] += c_.get("in", 0)
                ff[k]["lines"][ln][1] += c_.get("out", 0)
    for i, (k, v) in enumerate(sorted(ff.items()), 2):
        row = [k[0], f"{k[1]}:00", k[2], v["people"], v["events"]]
        for ln in lines:
            row += v["lines"][ln]
        ws.write_row(i, 0, row)
    ws.set_column(0, 2, 12)
    ws.freeze_panes(2, 0)

    # ---------------------------------------------------------- Config
    ws = wb.add_worksheet("Config")
    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 120, F["wrap"])
    for i, (k, v) in enumerate(cfg.to_dict().items()):
        ws.write(i, 0, k, F["b"])
        ws.write(i, 1, json.dumps(v) if isinstance(v, (list, dict)) else str(v))
    wb.close()
    log.info("Excel report: %s (%d events, %d thumbnails)", path, len(events), thumbs_used[0])


# ====================================================================== HTML
def build_html(store, cfg, path, max_embedded=1500, run_id=""):
    """Self-contained dashboard: tabs Events / Faces / Highlights / Videos, filters for level, camera,
    date range, identity, people-only, free text; NEW faces badged."""
    out = os.path.dirname(path) or cfg.output_dir
    events, videos, tracks, faces, tamper = load(store)
    idt = identities_table(tracks)
    E = html.escape
    if not run_id:
        r = store.query("SELECT run_id FROM runs ORDER BY started DESC LIMIT 1")
        run_id = r[0]["run_id"] if r else ""
    new_ids = set()
    jp = os.path.join(cfg.output_dir, "identities.json")
    idinfo = {}
    if os.path.exists(jp):
        try:
            with open(jp) as fh:
                idinfo = {x["id"]: x for x in json.load(fh)}
            new_ids = {k for k, x in idinfo.items() if run_id and x.get("first_run") == run_id and not x.get("name")}
        except Exception:
            pass
    by_score = sorted(events, key=lambda e: -(e["score"] or 0))
    embed_ids = {e["event_id"] for e in by_score[:max_embedded]}

    def b64(src, w=360):
        try:
            t = _thumb(src, cfg.output_dir, w)
            if not t:
                return ""
            with open(t, "rb") as fh:
                return "data:image/jpeg;base64," + base64.b64encode(fh.read()).decode()
        except Exception:
            return ""

    cards = []
    for e in events:
        snaps = e["snapshots"] or []
        best = max(snaps, key=lambda s: s.get("persons", 0)) if snaps else {}
        img = b64(best.get("annotated", "")) if e["event_id"] in embed_ids else ""
        flags = "".join(f"<li><b>{E(f['flag'])}</b> +{f['points']:g}: {E(f['reason'])}</li>" for f in e["flags"])
        vlm = E(_vlm_text(e["vlm"]))
        links = [f'<a href="{E(_rel(e["contact_sheet"], out))}" target="_blank">contact sheet</a>']
        if e["clip_path"]:
            links.append(f'<a href="{E(_rel(e["clip_path"], out))}" target="_blank">clip</a>')
        links.append(f'<a href="{E(_rel(e["folder"], out))}" target="_blank">folder</a>')
        objs = ", ".join(f"{k}×{n}" for k, n in (e["objects"] or {}).items())
        if e.get("plates"):
            objs += " · plates " + ", ".join(e["plates"])
        ids_ = e["identities"] or []
        idhtml = " ".join(f'<span class="pid{" new" if i in new_ids else ""}">{E(i)}{" NEW" if i in new_ids else ""}</span>'
                          for i in ids_)
        cards.append(f"""
<div class="card" data-level="{e['level']}" data-cam="{E(e['camera'] or '')}" data-date="{(e['start_dt'] or '')[:10]}"
 data-score="{e['score']}" data-people="{e['n_person_tracks']}" data-ids=" {E(' '.join(ids_))} " data-new="{int(any(i in new_ids for i in ids_))}"
 data-text="{E((e['event_id'] + ' ' + objs + ' ' + vlm + ' ' + ' '.join(ids_) + ' ' + ' '.join(f['flag'] for f in e['flags'])).lower())}">
 <div class="hd"><span class="lv {e['level']}">{e['level']} {e['score']:g}</span> <b>{E(e['event_id'])}</b></div>
 {'<a href="' + E(_rel(best.get('annotated', ''), out)) + '" target="_blank"><img loading="lazy" src="' + img + '"></a>' if img else ''}
 <div class="meta">🕒 {e['start_dt'] or '?'} → {(e['end_dt'] or '')[11:]} ({e['duration']:.0f}s)<br>
 🎞 {E(e['rel_path'])} @ <b>{V.fmt_offset(e['start_off'])}</b>–{V.fmt_offset(e['end_off'])}<br>
 👥 {e['max_persons']} max · {e['n_person_tracks']} tracks · faces {e['faces_detected']} {idhtml}<br>
 📦 {E(objs)}</div>
 {'<div class="vlm">🤖 ' + vlm + '</div>' if vlm else ''}
 {'<ul class="flags">' + flags + '</ul>' if flags else ''}
 <div class="links">{' · '.join(links)}</div></div>""")
    cams = sorted({e["camera"] for e in events if e["camera"]})
    days_all = sorted({(e["start_dt"] or "")[:10] for e in events if e["start_dt"]})
    # faces tab
    idcards = []
    for pid, r_ in idt.items():
        x = idinfo.get(pid, {})
        isnew = pid in new_ids
        days = sorted({(t["first_dt"] or "")[:10] for t in tracks if t["identity"] == pid and t["first_dt"]})
        idcards.append(
            f'<div class="id{" newcard" if isnew else ""}" data-pid="{pid}"><img src="{b64(r_["sample"] or x.get("sample", ""), 120)}">'
            f'<br><b>{pid}</b> {E(r_["name"])}{" <span class=badge>NEW</span>" if isnew else ""}'
            f'<br>{len(r_["events"])} events · {len(days)} day(s)<br><small>{E(", ".join(sorted(r_["cameras"])))}<br>'
            f'{E((r_["first"] or "")[:16])} → {E((r_["last"] or "")[:16])}</small><br>'
            f'<a href="#" onclick="showPid(\'{pid}\');return false">show events</a></div>')
    # highlights tab
    segs = store.query("SELECT r.*, e.camera, e.start_dt, e.level, e.score FROM reel_segments r "
                       "JOIN events e USING(event_id) ORDER BY r.day DESC, r.idx")
    hl = defaultdict(list)
    for g in segs:
        hl[g["day"]].append(g)
    hlhtml = []
    for d, gs in hl.items():
        rows = "".join(f"<tr><td>{g['idx']}</td><td>{V.fmt_offset(g['reel_start'])}</td><td>{E(g['camera'] or '')}</td>"
                       f"<td>{E((g['start_dt'] or '')[11:])}</td><td><span class='lv {g['level']}'>{g['level']} {g['score']:g}</span></td>"
                       f"<td>{E(g['event_id'])}</td></tr>" for g in gs)
        hlhtml.append(f'<details><summary><b>{d}</b> – {len(gs)} movement(s), {V.fmt_offset(gs[-1]["reel_end"])} · '
                      f'<a href="{E(_rel(gs[0]["reel_path"], out))}" target="_blank">▶ play reel</a></summary>'
                      f'<table><tr><th>#</th><th>Reel time</th><th>Camera</th><th>Clock</th><th>Level</th><th>Event</th></tr>{rows}</table></details>')
    vrows = "".join(f"<tr><td>{E(v['rel_path'] or '')}</td><td>{E(v['camera'] or '')}</td><td>{E(v['start_dt'] or '')}</td>"
                    f"<td>{V.fmt_offset(v['duration'] or 0)}</td><td>{v['n_events'] or 0}</td><td>{E(v['status'] or '')}"
                    f"{(' of ' + E(v['duplicate_of'] or '')) if v['status'] == 'duplicate' else ''}</td></tr>" for v in videos)
    tot = sum(v["duration"] or 0 for v in videos)
    mot = sum(v["motion_seconds"] or 0 for v in videos)
    lv = Counter(e["level"] for e in events)
    doc = f"""<!doctype html><html><head><meta charset="utf-8"><title>CCTV Analysis Report</title>
<meta name="viewport" content="width=device-width,initial-scale=1">
<style>body{{font-family:system-ui,Segoe UI,Arial;margin:0;background:#f4f6fa;color:#222}}
header{{background:#1f3864;color:#fff;padding:14px 20px}} h1{{margin:0;font-size:22px}}
.kpi{{display:flex;flex-wrap:wrap;gap:10px;padding:12px 20px}} .kpi div{{background:#fff;border-radius:8px;padding:10px 14px;box-shadow:0 1px 3px #0002}}
.kpi b{{display:block;font-size:20px}} .bar{{padding:6px 20px;display:flex;gap:10px;flex-wrap:wrap;align-items:center;position:sticky;top:0;background:#f4f6faee;z-index:2}}
.tabs{{padding:0 20px;display:flex;gap:4px}} .tabs button{{border:0;padding:8px 16px;border-radius:8px 8px 0 0;background:#dde3ee;cursor:pointer;font-size:14px}}
.tabs button.on{{background:#1f3864;color:#fff}} .tab{{display:none}} .tab.on{{display:block}}
.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(360px,1fr));gap:12px;padding:12px 20px}}
.card{{background:#fff;border-radius:10px;box-shadow:0 1px 4px #0002;padding:10px;font-size:13px}} .card img{{width:100%;border-radius:6px}}
.lv{{padding:2px 8px;border-radius:10px;font-weight:bold;font-size:12px}} .HIGH{{background:{LEVEL_COLORS['HIGH']}}} .REVIEW{{background:{LEVEL_COLORS['REVIEW']}}}
.LOW{{background:{LEVEL_COLORS['LOW']}}} .NORMAL{{background:{LEVEL_COLORS['NORMAL']}}} .meta{{margin:6px 0;line-height:1.5}}
.vlm{{background:#eef3ff;border-radius:6px;padding:6px;margin:4px 0}} .flags{{margin:4px 0;padding-left:18px;color:#a33}}
.pid{{background:#e8ecf3;border-radius:6px;padding:0 5px;font-size:12px}} .pid.new,.badge{{background:#ffeb3b;font-weight:bold;border-radius:6px;padding:0 5px}}
.ids{{display:flex;flex-wrap:wrap;gap:10px;padding:12px 20px}} .id{{background:#fff;border-radius:8px;padding:8px;text-align:center;font-size:12px;width:150px;box-shadow:0 1px 3px #0002}}
.id img{{width:110px;border-radius:6px}} .newcard{{outline:3px solid #ffd600}}
table{{border-collapse:collapse;background:#fff;margin:6px 0;font-size:13px}} td,th{{border:1px solid #ddd;padding:3px 8px;text-align:left}}
details{{background:#fff;margin:8px 20px;padding:8px 12px;border-radius:8px}} .pad{{padding:12px 20px}}</style></head><body>
<header><h1>CCTV Analysis Report</h1><div>Generated {dt.datetime.now():%Y-%m-%d %H:%M} · shop hours {E(hours_text(cfg))}</div></header>
<div class="kpi"><div><b>{len(videos)}</b>videos</div><div><b>{tot / 3600:.1f} h</b>footage</div>
<div><b>{mot / 3600:.1f} h</b>with motion</div><div><b>{len(events)}</b>events</div>
<div><b>{lv.get('HIGH', 0)}</b>HIGH</div><div><b>{lv.get('REVIEW', 0)}</b>REVIEW</div><div><b>{len(idt)}</b>face IDs</div>
<div><b>{len(new_ids)}</b>NEW faces (last run)</div><div><b>{len(tamper)}</b>tamper</div></div>
<div class="tabs"><button class="on" data-t="ev">Events</button><button data-t="fa">Faces</button><button data-t="hl">Highlights</button><button data-t="vi">Videos</button></div>
<div id="ev" class="tab on"><div class="bar">Level <select id="lv"><option value="">all</option><option>HIGH</option><option>REVIEW</option><option>LOW</option><option>NORMAL</option></select>
Camera <select id="cam"><option value="">all</option>{''.join(f'<option>{E(c)}</option>' for c in cams)}</select>
From <input id="d1" type="date" value="{days_all[0] if days_all else ''}"> to <input id="d2" type="date" value="{days_all[-1] if days_all else ''}">
Person <input id="pid" size="6" placeholder="P0003"> <label><input id="ppl" type="checkbox"> people only</label> <label><input id="nw" type="checkbox"> new faces</label>
Search <input id="q" placeholder="bag, theft, running…"> Sort <select id="sort"><option value="time">time</option><option value="score">score</option></select>
<span id="cnt"></span></div>
<div class="grid" id="grid">{''.join(cards)}</div></div>
<div id="fa" class="tab"><div class="pad">Yellow = face first seen in the latest run. Full list with first/last seen: <b>Unique_Faces.xlsx</b>.</div><div class="ids">{''.join(idcards)}</div></div>
<div id="hl" class="tab"><div class="pad">One video per day with every movement from all cameras, in time order (highlights/YYYY/MM-Mon/). The table maps reel time to camera and clock time.</div>{''.join(hlhtml) or '<div class="pad">no reels yet</div>'}</div>
<div id="vi" class="tab"><div class="pad"><table><tr><th>Video</th><th>Camera</th><th>Start</th><th>Duration</th><th>Events</th><th>Status</th></tr>{vrows}</table></div></div>
<script>
const g=document.getElementById('grid'),cards=[...g.children],$=i=>document.getElementById(i);
function f(){{const lv=$('lv').value,cam=$('cam').value,d1=$('d1').value,d2=$('d2').value,pid=$('pid').value.trim().toUpperCase(),
q=$('q').value.toLowerCase(),p=$('ppl').checked,nw=$('nw').checked,s=$('sort').value;let n=0;
let arr=cards.slice(); if(s=='score') arr.sort((a,b)=>b.dataset.score-a.dataset.score); arr.forEach(c=>g.appendChild(c));
cards.forEach(c=>{{const d=c.dataset.date;const ok=(!lv||c.dataset.level==lv)&&(!cam||c.dataset.cam==cam)&&(!d1||!d||d>=d1)&&(!d2||!d||d<=d2)
&&(!pid||c.dataset.ids.includes(' '+pid+' '))&&(!p||c.dataset.people>0)&&(!nw||c.dataset.new==1)&&(!q||c.dataset.text.includes(q));
c.style.display=ok?'':'none'; n+=ok;}}); $('cnt').textContent=n+' events';}}
function tab(t){{document.querySelectorAll('.tab').forEach(x=>x.classList.toggle('on',x.id==t));document.querySelectorAll('.tabs button').forEach(b=>b.classList.toggle('on',b.dataset.t==t));}}
document.querySelectorAll('.tabs button').forEach(b=>b.onclick=()=>tab(b.dataset.t));
function showPid(p){{$('pid').value=p;tab('ev');f();}}
document.querySelectorAll('.bar select,.bar input').forEach(e=>e.addEventListener('input',f)); f();
</script></body></html>"""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(doc)
    log.info("HTML report: %s", path)


# ====================================================================== charts
def build_charts(store, cfg):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    events = load(store)[0]
    ev = [e for e in events if e["start_dt"]]
    if not ev:
        return []
    os.makedirs(os.path.join(cfg.output_dir, "charts"), exist_ok=True)
    paths = []
    cams = sorted({e["camera"] for e in ev})
    days = sorted({e["start_dt"][:10] for e in ev})[-31:]
    fig, ax = plt.subplots(figsize=(14, max(3, 0.45 * len(days) * max(1, len(cams)) + 1)))
    ylabels = []
    y = 0
    colors = {"HIGH": "#d62728", "REVIEW": "#ff7f0e", "LOW": "#bcbd22", "NORMAL": "#2ca02c"}
    for d in days:
        for c in cams:
            for e in ev:
                if e["start_dt"][:10] == d and e["camera"] == c:
                    h0 = int(e["start_dt"][11:13]) + int(e["start_dt"][14:16]) / 60 + int(e["start_dt"][17:19]) / 3600
                    ax.barh(y, max(e["duration"] / 3600, 0.01), left=h0, color=colors.get(e["level"], "grey"), height=0.7)
            ylabels.append(f"{d} {c}")
            y += 1
    ax.set_yticks(range(len(ylabels)))
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))
    ax.set_xlabel("hour of day")
    from .rules import hours_enabled
    if hours_enabled(cfg):
        o = [int(x) for x in cfg.open_hour.split(":")[:2]] + [0]
        c_ = [int(x) for x in cfg.close_hour.split(":")[:2]] + [0]
        ax.axvspan(0, o[0] + o[1] / 60, color="#000", alpha=0.06)
        ax.axvspan(c_[0] + c_[1] / 60, 24, color="#000", alpha=0.06)
    ax.set_title("Event timeline (colour = level, shaded = closed hours)")
    ax.invert_yaxis()
    fig.tight_layout()
    p = os.path.join(cfg.output_dir, "charts", "timeline.png")
    fig.savefig(p, dpi=110)
    plt.close(fig)
    paths.append(p)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    hours = Counter(int(e["start_dt"][11:13]) for e in ev if e["n_person_tracks"])
    ax.bar(range(24), [hours.get(h, 0) for h in range(24)], color="#1f77b4")
    ax.set_xticks(range(24))
    ax.set_title("Events with people by hour of day")
    fig.tight_layout()
    p = os.path.join(cfg.output_dir, "charts", "people_by_hour.png")
    fig.savefig(p, dpi=110)
    plt.close(fig)
    paths.append(p)
    return paths


# ================================================================ unique faces
def build_unique_faces(store, cfg, path, run_id=""):
    """Unique_Faces.xlsx – one row per distinct person (face identity), NEW faces highlighted,
    plus the log of new-face alerts and unauthorised-person alerts."""
    import xlsxwriter
    jp = os.path.join(cfg.output_dir, "identities.json")
    ids = []
    if os.path.exists(jp):
        try:
            with open(jp) as fh:
                ids = json.load(fh)
        except Exception as e:
            log.warning("identities.json unreadable: %s", e)
    if not run_id:
        r = store.query("SELECT run_id FROM runs ORDER BY started DESC LIMIT 1")
        run_id = r[0]["run_id"] if r else ""
    base = os.path.dirname(path)
    stats = defaultdict(lambda: {"days": set(), "events": set(), "cams": set(), "first": None, "last": None})
    for t in store.query("SELECT t.identity, t.event_id, t.first_dt, e.camera FROM tracks t JOIN events e "
                         "USING(event_id) WHERE t.identity IS NOT NULL AND t.identity != ''"):
        st = stats[t["identity"]]
        st["events"].add(t["event_id"])
        st["cams"].add(t["camera"])
        if t["first_dt"]:
            st["days"].add(t["first_dt"][:10])
            st["first"] = min(filter(None, [st["first"], t["first_dt"]]))
            st["last"] = max(filter(None, [st["last"], t["first_dt"]]))
    evinfo = {e["event_id"]: e for e in store.query("SELECT event_id, rel_path, start_off, start_dt, camera FROM events")}
    first_alert = {}
    for a in store.query("SELECT identity, t_off, video FROM face_alerts WHERE kind='new_face' ORDER BY id"):
        first_alert.setdefault(a["identity"], a)
    wb = xlsxwriter.Workbook(path, {"strings_to_urls": False})
    H = wb.add_format({"bold": True, "bg_color": "#1F3864", "font_color": "white", "text_wrap": True, "border": 1})
    T = wb.add_format({"valign": "top"})
    NEW = wb.add_format({"valign": "top", "bg_color": "#FFEB3B", "bold": True})
    KNOWN = wb.add_format({"valign": "top", "bg_color": "#C6EFCE"})
    LINK = wb.add_format({"font_color": "blue", "underline": 1, "valign": "top"})
    n_img = [0]

    def image(ws, r, c, src, w=80):
        if not src or n_img[0] >= cfg.excel_max_thumbnails:
            return
        try:
            t = _thumb(src, cfg.output_dir, 160)
            img = cv2.imread(t) if t else None
            if img is not None:
                sc = w / img.shape[1]
                ws.insert_image(r, c, t, {"x_scale": sc, "y_scale": sc, "x_offset": 2, "y_offset": 2,
                                          "object_position": 1})
                n_img[0] += 1
        except Exception:
            pass

    def link(ws, r, c, target, text):
        if target:
            ws.write_url(r, c, "external:" + _rel(target, base), LINK, text)

    rows = []
    for rec in ids:
        st = stats.get(rec["id"], {"days": set(), "events": set(rec.get("events") or []), "cams": set(),
                                   "first": None, "last": None})
        n_ev = len(st["events"] or set(rec.get("events") or []))
        n_days = len(st["days"])
        if rec.get("name"):
            status = "Known"
        elif rec.get("first_run") == run_id and run_id:
            status = "NEW"
        elif n_days >= 3 or n_ev >= 5:
            status = "Regular"
        else:
            status = "Seen before"
        rows.append((rec, st, n_ev, n_days, status))
    order = {"NEW": 0, "Known": 2, "Regular": 1, "Seen before": 1}
    rows.sort(key=lambda r: (order[r[4]], -(len(r[1]["days"])), r[0]["id"]))

    ws = wb.add_worksheet("Unique Faces")
    cols = [("Face", 13), ("Identity", 9), ("Status", 11), ("Name", 14), ("First seen", 19), ("Last seen", 19),
            ("Days seen", 7), ("Events", 7), ("Face samples", 8), ("Cameras", 16), ("First camera", 10),
            ("First event", 26), ("First video", 34), ("Offset", 9), ("First run", 16), ("Face file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, H)
        ws.set_column(c, c, w)
    for i, (rec, st, n_ev, n_days, status) in enumerate(rows, 1):
        fmt = NEW if status == "NEW" else KNOWN if status == "Known" else T
        fe = evinfo.get(rec.get("first_event") or "", {})
        cams = sorted(set(st["cams"]) | set(rec.get("cameras") or []) | ({rec["first_camera"]} if rec.get("first_camera") else set()))
        vals = ["", rec["id"], status, rec.get("name") or "", st["first"] or rec.get("first_seen") or "",
                st["last"] or rec.get("last_seen") or "", n_days, n_ev, rec.get("count", 0), ", ".join(cams),
                rec.get("first_camera") or fe.get("camera", ""), rec.get("first_event") or "",
                fe.get("rel_path", ""),
                V.fmt_offset(first_alert[rec["id"]]["t_off"] or 0) if rec["id"] in first_alert
                else (V.fmt_offset(fe["start_off"]) if fe else ""), rec.get("first_run") or ""]
        for c, v in enumerate(vals):
            ws.write(i, c, v, fmt)
        image(ws, i, 0, rec.get("sample"))
        link(ws, i, 15, rec.get("sample"), "face")
        ws.set_row(i, 82)
    if rows:
        ws.autofilter(0, 0, len(rows), len(cols) - 1)
    ws.freeze_panes(1, 2)

    for sheet, kind in (("New faces log", "new_face"), ("Unauthorized alerts", "unauthorized")):
        ws = wb.add_worksheet(sheet)
        cols = [("Face", 13), ("Person", 13), ("Identity", 9), ("Run", 16), ("Time", 19), ("Camera", 10),
                ("Event ID", 26), ("Video file", 34), ("Offset", 9), ("Note", 40)]
        for c, (h, w) in enumerate(cols):
            ws.write(0, c, h, H)
            ws.set_column(c, c, w)
        al = store.query("SELECT * FROM face_alerts WHERE kind=? ORDER BY id DESC", (kind,))
        for i, a in enumerate(al, 1):
            fmt = NEW if a["run_id"] == run_id else T
            ws.write_row(i, 2, [a["identity"], a["run_id"], a["dt"] or "", a["camera"], a["event_id"], a["video"],
                                V.fmt_offset(a["t_off"] or 0), a["note"]], fmt)
            image(ws, i, 0, a["face_crop"])
            image(ws, i, 1, a["person_crop"], 60)
            ws.set_row(i, 82)
        if not al:
            ws.write(1, 2, "none")
        ws.freeze_panes(1, 0)
    wb.close()
    log.info("Unique faces workbook: %s (%d identities, %d new this run)", path, len(rows),
             sum(r[4] == "NEW" for r in rows))
    return path


# ================================================================ weeks
_MON = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def week_bounds(day: str, week_start: str = "sunday"):
    d = dt.date.fromisoformat(day)
    shift = (d.weekday() + 1) % 7 if week_start.lower().startswith("sun") else d.weekday()
    s = d - dt.timedelta(days=shift)
    return s, s + dt.timedelta(days=6)


def week_label(s: dt.date, e: dt.date, week_start: str = "sunday") -> str:
    """11-17-Oct-2026-W41 · 27-Sep-03-Oct-2026-W39 · 28-Dec-2025-03-Jan-2026-W52"""
    if week_start.lower().startswith("sun"):
        wk = int(s.strftime("%U"))          # a Sunday is always in week >= 1
    else:
        wk = s.isocalendar()[1]
    if s.year != e.year:
        rng = f"{s.day:02d}-{_MON[s.month - 1]}-{s.year}-{e.day:02d}-{_MON[e.month - 1]}-{e.year}"
    elif s.month != e.month:
        rng = f"{s.day:02d}-{_MON[s.month - 1]}-{e.day:02d}-{_MON[e.month - 1]}-{e.year}"
    else:
        rng = f"{s.day:02d}-{e.day:02d}-{_MON[s.month - 1]}-{e.year}"
    return f"{rng}-W{wk:02d}"


def _publish(tmp, final):
    import shutil
    os.makedirs(os.path.dirname(final), exist_ok=True)
    try:
        shutil.copyfile(tmp, final)
    except PermissionError:      # workbook open elsewhere -> timestamped copy
        final = final.replace(".xlsx", f"_{dt.datetime.now():%Y%m%d_%H%M%S}.xlsx")
        shutil.copyfile(tmp, final)
    try:
        os.remove(tmp)
    except OSError:
        pass
    return final


_WEEK_FILE = re.compile(r"^\d{2}-(?:[A-Z][a-z]{2}-)?(?:\d{4}-)?\d{2}-[A-Z][a-z]{2}-\d{4}-W\d{2}(?:_\d{8}_\d{6})?\.xlsx$")


def _remove_stale_weeks(cfg, current_labels) -> None:
    """Weekly workbooks for weeks that no longer have events (e.g. after times were corrected)."""
    root = os.path.join(cfg.output_dir, "reports")
    if not os.path.isdir(root):
        return
    for year in os.listdir(root):
        d = os.path.join(root, year)
        if not os.path.isdir(d):
            continue
        for f in os.listdir(d):
            if _WEEK_FILE.match(f) and f.split("_")[0].replace(".xlsx", "") not in current_labels:
                os.remove(os.path.join(d, f))
                log.info("removed weekly report without events: %s", f)


def build_weekly(store, cfg, new_days=None) -> list:
    days = sorted({r["d"] for r in store.query("SELECT DISTINCT substr(start_dt,1,10) d FROM events "
                                               "WHERE start_dt IS NOT NULL")})
    weeks: dict = {}
    for d in days:
        try:
            s, e = week_bounds(d, cfg.week_start)
        except ValueError:
            continue
        weeks.setdefault(s, e)
    touched = set()
    for d in new_days or []:
        try:
            touched.add(week_bounds(d, cfg.week_start)[0])
        except ValueError:
            pass
    out = []
    _remove_stale_weeks(cfg, {week_label(s, e, cfg.week_start) for s, e in weeks.items()})
    for s, e in sorted(weeks.items()):
        label = week_label(s, e, cfg.week_start)
        final = os.path.join(cfg.output_dir, "reports", str(s.year), label + ".xlsx")
        if new_days is not None and s not in touched and os.path.exists(final):
            continue                       # untouched week, already built
        tmp = os.path.join(cfg.work_dir, label + ".xlsx")
        try:
            build_excel(store, cfg, tmp, s.isoformat(), e.isoformat(), link_base=os.path.dirname(final),
                        title=f"CCTV weekly report {label}")
            out.append(_publish(tmp, final))
        except Exception as ex:
            log.exception("weekly report %s failed: %s", label, ex)
    return out


def build_reports(store, cfg, new_days=None, run_id="") -> dict:
    """Build every output. `new_days` = days touched by this run (None = rebuild what is missing)."""
    out: dict = {}
    os.makedirs(cfg.work_dir, exist_ok=True)
    if cfg.make_highlight_reel:
        try:
            from .reels import build_reels
            days = None if new_days is None else set(new_days)
            if days is not None:       # also fill days that never got a reel (e.g. ffmpeg failed earlier)
                have = {r["day"] for r in store.query("SELECT DISTINCT day FROM reel_segments")}
                days |= {(r["d"] or "unknown-date") for r in
                         store.query("SELECT DISTINCT substr(start_dt,1,10) d FROM events")} - have
            out["highlights"] = build_reels(store, cfg, days)
        except Exception as e:
            log.exception("highlight reels failed: %s", e)
    if cfg.master_report:
        try:
            tmp = os.path.join(cfg.work_dir, "CCTV_Report.xlsx")
            build_excel(store, cfg, tmp)
            out["excel"] = _publish(tmp, os.path.join(cfg.output_dir, "CCTV_Report.xlsx"))
        except Exception as e:
            log.exception("Excel report failed: %s", e)
    if cfg.weekly_reports:
        try:
            out["weekly"] = build_weekly(store, cfg, new_days)
        except Exception as e:
            log.exception("weekly reports failed: %s", e)
    if cfg.use_faces:
        try:
            tmp = os.path.join(cfg.work_dir, "Unique_Faces.xlsx")
            build_unique_faces(store, cfg, tmp, run_id)
            out["unique_faces"] = _publish(tmp, os.path.join(cfg.output_dir, "Unique_Faces.xlsx"))
        except Exception as e:
            log.exception("Unique faces workbook failed: %s", e)
    if cfg.make_html:
        try:
            out["html"] = os.path.join(cfg.output_dir, "report.html")
            build_html(store, cfg, out["html"], run_id=run_id)
        except Exception as e:
            log.exception("HTML report failed: %s", e)
    try:
        out["charts"] = build_charts(store, cfg)
    except Exception as e:
        log.exception("charts failed: %s", e)
    return out

In [ ]:
%%writefile /content/cctv_analysis/pipeline.py
"""Orchestrates the whole analysis: discovery -> planning (probe + recording time, then
chronological order) -> motion scan -> per-event deep analysis -> storage -> reports.
Safe to re-run: finished videos are skipped. Cancellable between videos and events."""
from __future__ import annotations

import datetime as dt
import gc
import heapq
import json
import logging
import os
import re
import sys
import threading
import time
import traceback
from collections import Counter, defaultdict
from typing import Callable, Dict, List, Optional

import cv2
import numpy as np

from . import annotate as A
from . import rules as R
from . import timeline as T
from . import video_io as V
from .config import Config
from .faces import sharpness
from .motion import scan_motion
from .storage import Store

log = logging.getLogger("cctv")
PEOPLE_LIKE = {"person"}
VEHICLES = {"bicycle", "car", "motorcycle", "bus", "truck"}
PLATE_VEHICLES = {"car", "motorcycle", "bus", "truck"}
ANIMALS = {"dog", "cat", "bird", "horse", "cow", "sheep"}


class Cancelled(Exception):
    """Raised inside a video when the user cancels; the video is left 'cancelled', not 'error'."""


def setup_logging(out_dir: str, verbose: bool = True):
    os.makedirs(out_dir, exist_ok=True)
    log.setLevel(logging.INFO)
    for h in list(log.handlers):            # replace our own handlers; keep ones the app attached
        if getattr(h, "_cctv_run", False):
            h.close()
            log.removeHandler(h)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S")
    fh = logging.FileHandler(os.path.join(out_dir, "run_log.txt"), encoding="utf-8")
    fh.setFormatter(fmt)
    setattr(fh, "_cctv_run", True)
    log.addHandler(fh)
    if verbose:
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        setattr(sh, "_cctv_run", True)
        log.addHandler(sh)
    log.propagate = False


def _safe(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)[:80]


def _iso(d: Optional[dt.datetime]) -> Optional[str]:
    return d.strftime("%Y-%m-%d %H:%M:%S") if d else None


def _at(base: Optional[dt.datetime], sec: float) -> Optional[str]:
    return _iso(base + dt.timedelta(seconds=sec)) if base else None


def _imwrite(path: str, img: np.ndarray, q: int = 88) -> str:
    """Write an image; never let a single failed image stop the analysis."""
    try:
        if img is None or getattr(img, "size", 0) == 0:
            return ""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, q]):
            return path
    except Exception as e:
        log.warning("could not save image %s: %s", path, e)
    return ""


def frame_brightness(frames: List[np.ndarray]) -> float:
    """99th-percentile brightness of the picture centre (DVR clock bands excluded): near 0 for a
    black/blank picture even when white OSD text is burnt in."""
    vals = []
    for f in frames:
        g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) if f.ndim == 3 else f
        h = g.shape[0]
        c = g[int(h * 0.2):int(h * 0.8)]
        if c.size:
            vals.append(float(np.percentile(c, 99)))
    return float(np.median(vals)) if vals else 0.0


def _motion_angle(obs, t: float) -> float:
    """Direction of motion (radians, image coords) of a track around time t – the direction of
    any motion blur on the object."""
    i = min(range(len(obs)), key=lambda j: abs(obs[j][0] - t))
    a, b = obs[max(0, i - 1)], obs[min(len(obs) - 1, i + 1)]
    dx = (b[2][0] + b[2][2] - a[2][0] - a[2][2]) / 2
    dy = (b[2][1] + b[2][3] - a[2][1] - a[2][3]) / 2
    return float(np.arctan2(dy, dx)) if (dx or dy) else 0.0


class Analyzer:
    def __init__(self, cfg: Config, progress: Optional[Callable[[dict], None]] = None,
                 cancel: Optional[threading.Event] = None):
        problems = cfg.validate()
        if problems:
            raise ValueError("Invalid configuration:\n  - " + "\n  - ".join(problems))
        self.cfg = cfg
        self.progress = progress
        self.cancel = cancel or threading.Event()
        os.makedirs(cfg.output_dir, exist_ok=True)
        os.makedirs(cfg.work_dir, exist_ok=True)
        setup_logging(cfg.output_dir)
        cfg.save(os.path.join(cfg.output_dir, "config_used.json"))
        from . import modelcache
        modelcache.init(cfg.model_cache_dir)
        self.run_id = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
        self.new_days: set = set()
        self.new_faces: List[dict] = []
        self.store = Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))
        from .detector import resolve_device
        V.HW_DECODE = cfg.hw_decode
        self.device = resolve_device(cfg.device)
        log.info("Device: %s", self.device)
        self._det = self._faces = self._sem = self._vlm = self._ocr = self._plates = None
        self._vlm_failed = self._plates_failed = False
        self.idb = self.watch = None
        self._vlm_count = 0
        self._vlm_errors = 0
        self.events_failed = 0
        self.sem_keys, self.sem_embs = [], []
        idx = os.path.join(cfg.output_dir, "semantic_index.npz")
        if os.path.exists(idx):
            try:
                z = np.load(idx, allow_pickle=True)
                self.sem_keys, self.sem_embs = list(z["keys"]), list(z["emb"])
            except Exception:
                pass

    def _emit(self, **kw):
        if self.progress:
            try:
                self.progress(kw)
            except Exception as e:      # a UI problem must never stop the analysis
                log.debug("progress callback failed: %s", e)

    def _check_cancel(self):
        if self.cancel.is_set():
            raise Cancelled()

    # ------------------------------------------------------------ lazy models
    @property
    def det(self):
        if self._det is None:
            from .detector import Detector
            self._det = Detector(self.cfg)
        return self._det

    @property
    def faces(self):
        if self._faces is None and self.cfg.use_faces:
            from .faces import FaceEngine, IdentityDB, Watchlist
            try:
                self._faces = FaceEngine(self.cfg)
                if self.cfg.face_identities:
                    self.idb = IdentityDB(os.path.join(self.cfg.output_dir, "identities"),
                                          self.cfg.face_match_threshold, self.cfg.known_face_threshold)
                    self.idb.run_id = self.run_id
                    self.idb.enroll_known(self._faces, self.cfg.known_faces_dir)
                if self.cfg.face_watchlist_dir:
                    self.watch = Watchlist(self._faces, self.cfg.face_watchlist_dir, self.cfg.known_face_threshold)
            except Exception as e:
                log.warning("Face analysis disabled: %s", e)
                self.cfg.use_faces = False
        return self._faces

    @property
    def sem(self):
        if self._sem is None and self.cfg.use_semantic:
            try:
                from .semantic import Semantic
                self._sem = Semantic(self.cfg, self.device)
            except Exception as e:
                log.warning("Semantic tagging disabled: %s", e)
                self.cfg.use_semantic = False
        return self._sem

    @property
    def vlm(self):
        if self._vlm is None and self.cfg.use_vlm and not self._vlm_failed:
            try:
                from .vlm import make_vlm
                log.info("Loading VLM %s via %s ...", self.cfg.vlm_model, self.cfg.vlm_provider)
                self._vlm = make_vlm(self.cfg, self.device)
            except Exception as e:
                log.warning("VLM disabled: %s", e)
                self._vlm_failed = True
        return self._vlm

    def _vlm_failure(self, ex):
        """Three failures in a row (server down, model missing) disable the VLM for this run instead
        of waiting for a timeout on every event."""
        self._vlm_errors += 1
        log.warning("  VLM failed (%d): %s", self._vlm_errors, ex)
        if self._vlm_errors >= 3 and not self._vlm_failed:
            self._vlm_failed = True
            self._vlm = None
            log.error("AI descriptions disabled for the rest of this run after 3 consecutive failures – check "
                      "the model / server (%s %s)", self.cfg.vlm_provider, self.cfg.vlm_base_url or self.cfg.vlm_model)

    def _vlm_if_enabled(self):
        return self.vlm if self.cfg.use_vlm else None

    @property
    def ocr(self):
        if self._ocr is None:
            from .ocr import get_ocr
            self._ocr = get_ocr(self.cfg.ocr_engine, vlm_getter=self._vlm_if_enabled)
        return self._ocr

    @property
    def plates(self):
        if self._plates is None and self.cfg.use_plates and not self._plates_failed:
            try:
                from .plates import PlateReader
                self._plates = PlateReader(self.cfg, lambda: self.ocr,
                                           self._vlm_if_enabled if self.cfg.plate_vlm_read else None,
                                           self._vlm_failure)
            except Exception as e:
                log.warning("Number-plate reading disabled: %s", e)
                self._plates_failed = True
        return self._plates

    # ------------------------------------------------------------------- run
    def _select(self, videos):
        """Skip finished videos (same path+size+mtime, or same content elsewhere) and duplicates."""
        cfg, st = self.cfg, self.store
        todo, stats, seen = [], Counter(), {}
        t0 = time.time()
        for p in videos:
            try:
                fst = os.stat(p)
            except OSError:
                continue
            if not cfg.reprocess and st.video_done(p, V.file_fingerprint(p)):
                stats["already_done"] += 1
                continue
            qh = None
            if cfg.dedupe:
                qh = st.cached_hash(p, fst.st_size, int(fst.st_mtime))
                if qh is None:
                    try:
                        qh = V.quick_hash(p)
                        st.put_hash(p, fst.st_size, int(fst.st_mtime), qh)
                    except OSError as e:
                        log.warning("cannot read %s: %s", p, e)
                        continue
                other = seen.get(qh) or (None if cfg.reprocess else st.done_with_hash(qh, p))
                if other:
                    stats["duplicates"] += 1
                    log.info("  duplicate skipped: %s  (same content as %s)", p, other)
                    rel = os.path.relpath(p, cfg.input_dir)
                    old = st.db.execute("SELECT id FROM videos WHERE path=?", (p,)).fetchone()
                    row = {"path": p, "rel_path": rel, "fingerprint": V.file_fingerprint(p), "qhash": qh,
                           "status": "duplicate", "duplicate_of": os.path.relpath(other, cfg.input_dir),
                           "size_mb": fst.st_size / 1e6, "run_id": self.run_id,
                           "processed_at": dt.datetime.now().isoformat(timespec="seconds")}
                    if old:
                        st.update_video(old["id"], **{k: v for k, v in row.items() if k != "path"})
                    else:
                        st.insert("videos", row)
                    continue
                seen[qh] = p
            todo.append((p, qh))
        st.commit(sync=False)
        log.info("%d new video(s) to analyse · %d already done · %d duplicate(s) skipped [%.0fs]",
                 len(todo), stats["already_done"], stats["duplicates"], time.time() - t0)
        return todo, stats

    def plan(self, todo) -> List[dict]:
        """Probe every video and resolve its recording time, then order chronologically
        (videos with unknown/approximate time last, in natural file-name order). Processing in time
        order makes face identities, 'new face' alerts and logs follow the real timeline."""
        items = []
        for i, (p, qh) in enumerate(todo, 1):
            self._check_cancel()
            self._emit(stage="planning", index=i, total=len(todo), path=p)
            item = {"path": p, "qhash": qh, "meta": None, "start_dt": None, "time_source": "unknown"}
            try:
                meta = V.probe(p)
                item["meta"] = meta
                if meta["width"] and meta["duration"]:
                    item["start_dt"], item["time_source"] = T.resolve_start_time(p, meta, self.cfg, self.ocr)
            except Exception as e:
                log.warning("could not plan %s: %s", p, e)
            items.append(item)
        far = dt.datetime.max.replace(tzinfo=dt.timezone.utc)
        items.sort(key=lambda x: (not T.is_exact(x["time_source"]),
                                  x["start_dt"] if (x["start_dt"] and T.is_exact(x["time_source"])) else far,
                                  V.natural_key(os.path.relpath(x["path"], self.cfg.input_dir)
                                              if os.path.isdir(self.cfg.input_dir) else x["path"])))
        return items

    def run(self, build_reports: bool = True) -> dict:
        cfg = self.cfg
        started = dt.datetime.now().isoformat(timespec="seconds")
        self._emit(stage="discovering")
        videos = V.find_videos(cfg)
        log.info("Found %d video file(s) under %s%s", len(videos), cfg.input_dir,
                 f" (folders matching {cfg.folder_keywords})" if cfg.folder_keywords else "")
        todo, stats = self._select(videos)
        t_all = time.time()
        try:
            plan = self.plan(todo)
        except Cancelled:
            plan, stats["cancelled"] = [], 1
        streak, last_error = 0, None
        for i, item in enumerate(plan, 1):
            p = item["path"]
            if self.cancel.is_set():
                stats["cancelled"] = 1
                stats["videos_deferred"] = len(plan) - i + 1
                log.warning("Cancelled – %d video(s) left for the next run", len(plan) - i + 1)
                break
            if cfg.max_runtime_minutes and (time.time() - t_all) / 60 >= cfg.max_runtime_minutes:
                log.warning("Time budget of %.0f min reached – %d video(s) left for the next run",
                            cfg.max_runtime_minutes, len(plan) - i + 1)
                stats["videos_deferred"] = len(plan) - i + 1
                break
            log.info("[%d/%d] %s", i, len(plan), p)
            self._emit(stage="video", index=i, total=len(plan), path=p, done_seconds=round(time.time() - t_all, 1))
            try:
                n = self.process_video(p, item["qhash"], item)
                stats["videos_ok"] += 1
                stats["events"] += n
            except KeyboardInterrupt:
                raise
            except Cancelled:
                stats["cancelled"] = 1
                stats["videos_deferred"] = len(plan) - i + 1
                self._mark(p, item["qhash"], "cancelled", "cancelled by user")
                log.warning("Cancelled during %s – it will be analysed again next run", p)
                break
            except Exception as e:
                stats["videos_failed"] += 1
                log.error("FAILED %s: %s\n%s", p, e, traceback.format_exc())
                self._mark(p, item["qhash"], "error", str(e)[:2000])
                same = str(e)[:200]
                streak = streak + 1 if same == last_error else 1
                last_error = same
                if streak >= 3:
                    stats["aborted"] = 1
                    stats["videos_deferred"] = len(plan) - i
                    log.error("Stopping: the same error on %d videos in a row – the installation or models are "
                              "probably broken: %s", streak, same)
                    break
            else:
                streak, last_error = 0, None
            gc.collect()
        self._save_side_data()
        stats["new_faces"] = len(self.new_faces)
        if self.events_failed:
            stats["events_failed"] = self.events_failed
        self.store.insert("runs", {"run_id": self.run_id, "started": started,
                                   "finished": dt.datetime.now().isoformat(timespec="seconds"),
                                   "videos": stats["videos_ok"], "events": stats["events"],
                                   "new_faces": len(self.new_faces), "duplicates": stats["duplicates"],
                                   "notes": json.dumps(dict(stats))})
        self.store.commit()
        log.info("Analysis finished in %.1f min: %s", (time.time() - t_all) / 60, dict(stats))
        if build_reports:
            self._emit(stage="reports")
            from .report import build_reports as br
            try:
                self.outputs = br(self.store, cfg, self.new_days, self.run_id)
            except Exception as e:
                log.exception("report building failed: %s", e)
        self.store.commit()
        self.print_summary(stats)
        self._emit(stage="finished", stats=dict(stats))
        return dict(stats)

    def _mark(self, p, qh, status, error):
        try:
            self.store.db.execute("UPDATE videos SET status=?, error=? WHERE path=?", (status, error, p))
            if not self.store.db.execute("SELECT 1 FROM videos WHERE path=?", (p,)).fetchone():
                self.store.insert("videos", {"path": p, "rel_path": os.path.relpath(p, self.cfg.input_dir),
                                             "status": status, "error": error, "qhash": qh,
                                             "fingerprint": V.file_fingerprint(p), "run_id": self.run_id})
            self.store.commit()
        except Exception:
            pass

    def print_summary(self, stats):
        log.info("=" * 70)
        log.info("RUN SUMMARY %s: %d video(s) analysed, %d event(s), %d duplicate(s) skipped, %d failed",
                 self.run_id, stats.get("videos_ok", 0), stats.get("events", 0), stats.get("duplicates", 0),
                 stats.get("videos_failed", 0))
        if self.new_days:
            log.info("Days updated: %s", ", ".join(sorted(self.new_days)))
        hi = self.store.query("SELECT event_id, level, score, start_dt, camera FROM events WHERE run_id=? AND "
                              "level IN ('HIGH','REVIEW') ORDER BY score DESC LIMIT 10", (self.run_id,))
        for e in hi:
            log.info("  ⚠️  %s %3.0f  %s  %s  %s", e["level"], e["score"], e["start_dt"], e["camera"], e["event_id"])
        if self.new_faces:
            log.warning("🆕 %d NEW face(s) never seen before – review Unique_Faces.xlsx:", len(self.new_faces))
            for f in self.new_faces[:30]:
                log.warning("     %s  first seen %s on %s (%s @ %s)", f["identity"], f["dt"], f["camera"],
                            f["video"], V.fmt_offset(f["t_off"] or 0))
        elif self.cfg.use_faces and self.cfg.face_identities:
            log.info("No new faces in this run.")
        al = self.store.query("SELECT COUNT(*) n FROM face_alerts WHERE run_id=? AND kind='unauthorized'", (self.run_id,))
        if al and al[0]["n"]:
            log.warning("🚫 %d unauthorised-person alert(s) on authorised-only cameras", al[0]["n"])
        pl = self.store.query("SELECT COUNT(*) n FROM plates p JOIN events e USING(event_id) WHERE e.run_id=? "
                              "AND p.text != ''", (self.run_id,))
        if pl and pl[0]["n"]:
            log.info("🚗 %d number plate(s) read in this run", pl[0]["n"])
        log.info("=" * 70)

    def _save_side_data(self):
        if self.idb is not None:
            self.idb.save()
        if self.sem_embs:
            np.savez_compressed(os.path.join(self.cfg.output_dir, "semantic_index.npz"),
                                emb=np.stack(self.sem_embs).astype(np.float16), keys=np.array(self.sem_keys, dtype=object))

    # --------------------------------------------------------- single video
    def process_video(self, path: str, qhash: Optional[str] = None, planned: Optional[dict] = None) -> int:
        cfg = self.cfg
        t0 = time.time()
        if planned and planned.get("meta"):
            meta, start_dt, tsrc = planned["meta"], planned["start_dt"], planned["time_source"]
        else:
            meta = V.probe(path)
            start_dt, tsrc = (T.resolve_start_time(path, meta, cfg, self.ocr) if meta["width"] and meta["duration"]
                              else (None, "unknown"))
        if not meta["width"] or not meta["duration"]:
            raise RuntimeError(f"unreadable video (probe: {meta})")
        camera = V.camera_name(path, cfg)
        rel = os.path.relpath(path, cfg.input_dir) if os.path.isdir(cfg.input_dir) else os.path.basename(path)
        vrow = {"path": path, "rel_path": rel, "fingerprint": V.file_fingerprint(path), "camera": camera,
                "start_dt": _iso(start_dt), "time_source": tsrc,
                "end_dt": _at(start_dt, meta["duration"]),
                "duration": meta["duration"], "fps": meta["fps"], "width": meta["width"],
                "height": meta["height"], "codec": meta["codec"], "size_mb": os.path.getsize(path) / 1e6,
                "status": "running", "error": None, "processed_at": dt.datetime.now().isoformat(timespec="seconds"),
                "drive_link": V.drive_link(path) if cfg.drive_links else "", "qhash": qhash,
                "run_id": self.run_id, "duplicate_of": None}
        vid = self.store.start_video(vrow)
        if self.store.purged_events and self.faces is not None and self.idb is not None:
            self.idb.forget_events(self.store.purged_events)
        log.info("  %s | %dx%d %.1ffps %s | %s | start %s (%s)", camera, meta["width"], meta["height"],
                 meta["fps"], V.fmt_offset(meta["duration"]), meta["codec"], _iso(start_dt), tsrc)
        if not T.is_exact(tsrc):
            log.warning("  recording time is not exact (%s) – add a filename pattern or time override, or make "
                        "sure the DVR clock is visible and an OCR engine is installed", tsrc)
        self._emit(stage="motion", path=path)
        mr = scan_motion(path, meta, cfg, camera)
        self._check_cancel()
        log.info("  motion pass: %d frames, %d event(s), %d tamper, %d lighting change(s) [%.0fs]",
                 mr.frames, len(mr.events), len(mr.tamper), len(mr.lighting_changes), time.time() - t0)
        vdir = os.path.join(cfg.output_dir, "events", _safe(camera or "cam"),
                            start_dt.strftime("%Y-%m-%d") if start_dt else "unknown-date")
        heat_path = ""
        if mr.heatmap is not None and mr.preview is not None:
            heat_path = _imwrite(os.path.join(cfg.output_dir, "heatmaps", _safe(rel) + ".jpg"),
                                 A.heatmap_overlay(mr.preview, mr.heatmap))
        for tp in mr.tamper:
            self.store.insert("tamper", {"video_id": vid, "start_off": tp["start"], "end_off": tp["end"],
                                         "start_dt": _at(start_dt, tp["start"]), "kind": tp["kind"]}, replace=False)
        # camera tampering or lighting changes outside motion events become their own events
        for kind, ivs in (("tamper", mr.tamper), ("lighting", [{"start": t, "end": t} for t in mr.lighting_changes])):
            for iv in ivs:
                if not any(e["start"] <= iv["end"] and e["end"] >= iv["start"] for e in mr.events):
                    mr.events.append({"start": max(0.0, iv["start"] - 3), "end": min(meta["duration"], iv["end"] + 4),
                                      "motion_peak": 0.0, "motion_mean": 0.0, "motion_active_s": 0.0, "origin": kind})
        mr.events.sort(key=lambda e: e["start"])
        for ev in mr.events:
            ev["lighting_changes"] = sum(1 for t in mr.lighting_changes if ev["start"] <= t <= ev["end"])
        n_saved = n_failed = 0
        first_error = None
        for k, ev in enumerate(mr.events, 1):
            self._check_cancel()
            self._emit(stage="event", path=path, event_index=k, events=len(mr.events))
            try:
                if self.analyze_event(path, meta, vid, rel, camera, start_dt, tsrc, ev, k, vdir, mr.tamper):
                    n_saved += 1
            except Cancelled:
                raise
            except Exception as e:
                n_failed += 1
                first_error = first_error or e
                log.error("  event %d failed: %s\n%s", k, e, traceback.format_exc())
            if k % 10 == 0:
                self.store.commit(sync=False)
        self.events_failed += n_failed
        if n_failed and not n_saved:
            # nothing could be analysed (e.g. the detector cannot run): this video is NOT done
            raise RuntimeError(f"all {n_failed} event(s) failed: {first_error}")
        motion_s = float(sum(e["end"] - e["start"] for e in mr.events))
        self.store.update_video(vid, status="done", n_events=n_saved, motion_seconds=motion_s,
                                proc_seconds=time.time() - t0, heatmap_path=heat_path,
                                lighting_changes=len(mr.lighting_changes),
                                osd_masked=int(mr.osd_mask is not None),
                                models=json.dumps({"detector": self._det.model_name if self._det else "",
                                                   "pose": self._det.pose_name if self._det else "",
                                                   "semantic": cfg.semantic_model if self._sem else "",
                                                   "vlm": cfg.vlm_model if self._vlm else "",
                                                   "ocr": self._ocr.name if self._ocr else "",
                                                   "plates": (f"{self._plates.detector.name}/{self._plates.reader.name}"
                                                              if self._plates else "")}))
        self._save_side_data()
        self.store.commit()
        log.info("  done: %d event(s) in %.1fs (%.1fx real-time)", n_saved, time.time() - t0,
                 meta["duration"] / max(1e-3, time.time() - t0))
        return n_saved

    def _event_clock(self, path, meta, start_dt, tsrc, s):
        return T.event_clock(path, meta, start_dt, tsrc, s, self.cfg, self.ocr)

    # --------------------------------------------------------- single event
    def analyze_event(self, path, meta, vid, rel, camera, start_dt, tsrc, ev, k, vdir, tamper) -> bool:
        cfg = self.cfg
        s, e = ev["start"], ev["end"]
        dur = max(0.5, e - s)
        base, ev_src = self._event_clock(path, meta, start_dt, tsrc, s)
        ev_dt = base + dt.timedelta(seconds=s) if base else None
        ev["start_dt"] = ev_dt
        ev["camera"] = camera
        eid = f"{_safe(camera or 'cam')}-{ev_dt:%Y%m%d-%H%M%S}-{vid:04d}{k:03d}" if ev_dt else \
            f"{_safe(os.path.splitext(os.path.basename(path))[0])}-{vid:04d}{k:03d}"
        if base is not start_dt and ev_dt:
            vdir = os.path.join(os.path.dirname(vdir), ev_dt.strftime("%Y-%m-%d"))
        edir = os.path.join(vdir, eid)
        zones = R.zones_for(cfg, camera)
        lines = R.lines_for(cfg, camera)
        det = self.det
        det.reset()
        fe = self.faces
        need_emb = fe is not None and (self.idb is not None or self.watch is not None)
        plate_reader = self.plates
        K = max(1, cfg.snapshots_per_event)
        slots: List[Optional[tuple]] = [None] * K
        tracks: Dict[int, dict] = {}
        frame_people = []
        det_rows = []
        last_logged: Dict[int, float] = {}
        cls_counter = defaultdict(set)
        faces_total = 0
        plate_cands: Dict[int, list] = defaultdict(list)     # track -> heap of (score, t, normalised box)
        H = W = None
        nframes = 0
        from .detector import posture
        for t, frame in V.iter_frames(path, cfg.detect_fps, cfg.detect_width, s, dur,
                                      src_size=(meta["width"], meta["height"])):
            if nframes % 25 == 0:
                self._check_cancel()
            nframes += 1
            H, W = frame.shape[:2]
            dets = det(frame)
            persons = [d for d in dets if d["cls"] == "person"]
            # ---- faces (only for people still needing good faces)
            ffaces = []
            if fe is not None and persons:
                need = [p for p in persons if p["track"] is None or
                        len(tracks.get(p["track"], {}).get("faces", [])) < cfg.faces_per_track or nframes % 4 == 0]
                ffaces = fe.detect_in_people(frame, need)
                attempted = {p["track"] for p in need}
            else:
                attempted = set()
            faces_total += len(ffaces)
            face_by_track = {f["track"]: f for f in ffaces if f["track"] is not None}
            for d in dets:
                tid = d["track"]
                if tid is None:
                    continue
                tr = tracks.setdefault(tid, {"track": tid, "cls": d["cls"], "obs": [], "votes": Counter(),
                                             "zone_frames": Counter(), "hidden_frames": 0, "hands_up": 0,
                                             "crouch": 0, "faces": [], "best_crop": None,
                                             "faces_found": 0})
                tr["votes"][d["cls"]] += d["conf"]
                tr["cls"] = tr["votes"].most_common(1)[0][0]
                tr["obs"].append((t, d["conf"], d["xyxy"]))
                cls_counter[d["cls"]].add(tid)
                x1, y1, x2, y2 = d["xyxy"]
                if plate_reader is not None and d["cls"] in PLATE_VEHICLES and (y2 - y1) >= cfg.plate_min_vehicle_px:
                    crop = frame[int(y1):int(y2), int(x1):int(x2)]
                    edge = x1 < 2 or y1 < 2 or x2 > W - 2 or y2 > H - 2
                    sc = d["conf"] * (y2 - y1) * np.sqrt(max(1.0, sharpness(crop))) * (0.5 if edge else 1.0)
                    hp = plate_cands[tid]
                    item = (sc, t, [x1 / W, y1 / H, x2 / W, y2 / H])
                    if len(hp) < cfg.plate_reads_per_track:
                        heapq.heappush(hp, item)
                    elif sc > hp[0][0]:
                        heapq.heapreplace(hp, item)
                if d["cls"] == "person":
                    fx, fy = ((x1 + x2) / 2) / W, y2 / H
                    for zn, poly in zones.items():
                        if R.point_in_poly(fx, fy, poly):
                            tr["zone_frames"][zn] += 1
                    ps = posture(d["kpts"])
                    tr["hands_up"] += ps["hands_up"]
                    tr["crouch"] += ps["crouching"]
                    if fe is not None and tid in attempted and tid not in face_by_track and \
                            ps["face_visible_pose"] and (y2 - y1) > 0.3 * H:
                        tr["hidden_frames"] += 1
                    # best full-body crop: confident, large, not cut by the border
                    edge = x1 < 2 or y1 < 2 or x2 > W - 2 or y2 > H - 2
                    sc = d["conf"] * np.sqrt((x2 - x1) * (y2 - y1)) * (0.6 if edge else 1.0)
                    if tr["best_crop"] is None or sc > tr["best_crop"][0]:
                        pad = 0.06
                        cx1, cy1 = int(max(0, x1 - pad * (x2 - x1))), int(max(0, y1 - pad * (y2 - y1)))
                        cx2, cy2 = int(min(W, x2 + pad * (x2 - x1))), int(min(H, y2 + pad * (y2 - y1)))
                        tr["best_crop"] = (sc, frame[cy1:cy2, cx1:cx2].copy(), t)
                    f = face_by_track.get(tid)
                    if f is not None:
                        tr["faces_found"] += 1
                        q = fe.quality(frame, f)
                        if len(tr["faces"]) < cfg.faces_per_track or q > min(x[0] for x in tr["faces"]):
                            emb = fe.embed(frame, f) if need_emb else None
                            tr["faces"].append((q, t, f, emb, fe.crop(frame, f["box"])))
                            tr["faces"].sort(key=lambda x: -x[0])
                            del tr["faces"][cfg.faces_per_track:]
                # sampled detection log: one row per track per second
                if t - last_logged.get(tid, -9) >= 1.0:
                    last_logged[tid] = t
                    det_rows.append({"event_id": eid, "t_off": round(t, 2), "dt": _at(base, t),
                                     "cls": d["cls"], "conf": round(d["conf"], 3), "track": tid,
                                     "x1": round(x1 / W, 4), "y1": round(y1 / H, 4),
                                     "x2": round(x2 / W, 4), "y2": round(y2 / H, 4)})
            frame_people.append((t, len(persons)))
            interest = 2.0 * len(persons) + sum(d["conf"] for d in dets) + 0.7 * len(ffaces) + 0.01
            si = min(K - 1, int((t - s) / dur * K))
            if slots[si] is None or interest > slots[si][0]:
                slots[si] = (interest, t, frame.copy(), dets, ffaces)
        if nframes == 0:
            log.warning("  event %s: no frames decoded", eid)
            return False

        # ---------------- track summaries + identities
        fps = cfg.detect_fps
        summaries = []
        id_map = {}
        crossings: Dict[str, Dict[str, int]] = defaultdict(lambda: {"in": 0, "out": 0})
        for tid, tr in tracks.items():
            if len(tr["obs"]) < 2 and max(o[1] for o in tr["obs"]) < 0.6:
                continue                                  # one-off noise
            sm = R.summarize_track(tr, fps, H)
            sm.update({"identity": "", "identity_sim": None, "known_name": "", "face_crop": "",
                       "person_crop": "", "identity_visits": 0, "watchlist": "", "crossings": {}})
            if lines and tr["cls"] == "person":
                pts = [(((o[2][0] + o[2][2]) / 2) / W, o[2][3] / H) for o in tr["obs"]]
                sm["crossings"] = R.count_crossings(pts, lines)
                for ln, c in sm["crossings"].items():
                    crossings[ln]["in"] += c["in"]
                    crossings[ln]["out"] += c["out"]
            if tr["best_crop"] is not None and tr["best_crop"][1].size:
                sm["person_crop"] = _imwrite(os.path.join(edir, "people", f"track{tid:03d}_{tr['cls'].replace(' ', '_')}.jpg"),
                                             tr["best_crop"][1], cfg.snapshot_jpeg_quality)
            embs = [x[3] for x in tr["faces"] if x[3] is not None and x[0] >= cfg.face_min_quality]
            if tr["faces"]:
                q, ft, f, _, crop = tr["faces"][0]
                if crop.size:
                    sm["face_crop"] = _imwrite(os.path.join(edir, "faces", f"track{tid:03d}_best.jpg"), crop, 92)
            if embs:
                m = np.mean(embs, 0)
                m /= np.linalg.norm(m) + 1e-9
                if self.watch is not None:
                    name, _ = self.watch.match(m)
                    sm["watchlist"] = name
                if self.idb is not None:
                    when = _at(base, sm["first"]) or rel
                    pid, sim, kn = self.idb.assign(m, when or "", sm["face_crop"], eid, camera)
                    if self.idb.last_new and not kn:
                        alert = {"run_id": self.run_id, "kind": "new_face", "identity": pid, "dt": when,
                                 "camera": camera, "event_id": eid, "video": rel, "t_off": round(sm["first"], 2),
                                 "face_crop": sm["face_crop"], "person_crop": sm["person_crop"],
                                 "note": "face never seen before"}
                        self.store.insert("face_alerts", alert, replace=False)
                        self.new_faces.append(alert)
                    if sm["face_crop"]:
                        self.idb.set_sample(pid, sm["face_crop"])
                    sm.update({"identity": pid, "identity_sim": round(sim, 3), "known_name": kn})
                    sm["identity_visits"] = next((len(r.get("events", [])) for r in self.idb.ids if r["id"] == pid), 1)
                    id_map[tid] = kn or pid
            if sm["watchlist"]:
                id_map[tid] = f"WATCH:{sm['watchlist']}"
                self.store.insert("face_alerts", {
                    "run_id": self.run_id, "kind": "watchlist", "identity": sm["identity"] or sm["watchlist"],
                    "dt": _at(base, sm["first"]), "camera": camera, "event_id": eid, "video": rel,
                    "t_off": round(sm["first"], 2), "face_crop": sm["face_crop"], "person_crop": sm["person_crop"],
                    "note": f"matches watchlist person '{sm['watchlist']}'"}, replace=False)
            for j, (q, ft, f, _, crop) in enumerate(tr["faces"]):
                cp = sm["face_crop"] if j == 0 else (
                    _imwrite(os.path.join(edir, "faces", f"track{tid:03d}_{j}.jpg"), crop, 90) if crop.size else "")
                self.store.insert("faces", {"event_id": eid, "track": tid, "t_off": round(ft, 2),
                                            "dt": _at(base, ft),
                                            "score": round(f["score"], 3), "quality": round(q, 3),
                                            "box": [round(v, 1) for v in f["box"]], "crop_path": cp,
                                            "identity": sm["identity"], "similarity": sm["identity_sim"],
                                            "known_name": sm["known_name"]}, replace=False)
            summaries.append(sm)

        # ---------------- number plates (full-resolution re-read of the best vehicle frames)
        plates = []
        if plate_reader is not None:
            kept_tracks = {sm["track"] for sm in summaries}
            for tid, hp in plate_cands.items():
                if tid not in kept_tracks:
                    continue
                self._check_cancel()
                samples = []
                obs = tracks[tid]["obs"]
                for _, t, nb in sorted(hp, key=lambda x: x[1]):
                    full = V.read_frame_at(path, t, 0, (meta["width"], meta["height"]))
                    if full is None:
                        continue
                    fh, fw = full.shape[:2]
                    samples.append((t, full, [nb[0] * fw, nb[1] * fh, nb[2] * fw, nb[3] * fh], _motion_angle(obs, t)))
                try:
                    res = plate_reader.read_track(samples)
                except Exception as ex:
                    log.warning("  plate reading failed for track %d: %s", tid, ex)
                    res = None
                if not res:
                    continue
                from .plates import is_reportable, watchlist_match
                ok = is_reportable(res["text"], res["confidence"], res["valid_format"])
                wl = watchlist_match(res["text"], cfg.plate_watchlist, cfg.plate_watchlist_max_edits) if ok else ""
                cp = _imwrite(os.path.join(edir, "plates", f"track{tid:03d}_plate.jpg"), res["crop"], 95)
                ep = _imwrite(os.path.join(edir, "plates", f"track{tid:03d}_enhanced.jpg"), res["enhanced"], 95) \
                    if res.get("enhanced") is not None else ""
                row = {"event_id": eid, "track": tid, "vehicle": tracks[tid]["cls"], "camera": camera,
                       "t_off": round(res["t_off"], 2), "dt": _at(base, res["t_off"]), "text": res["text"],
                       "confidence": res["confidence"], "valid_format": int(res["valid_format"]),
                       "n_reads": len(res["reads"]), "reads": res["reads"], "method": res["method"],
                       "crop_path": cp, "enhanced_path": ep, "watchlist": wl}
                self.store.insert("plates", row, replace=False)
                if ok:
                    plates.append({"text": res["text"], "confidence": res["confidence"], "track": tid,
                                   "vehicle": row["vehicle"], "watchlist": wl})
                    id_map[tid] = res["text"]
        ev["plates"] = plates

        persons = [x for x in summaries if x["cls"] == "person"]
        objects = {c: len(ids) for c, ids in cls_counter.items()}
        max_persons = max((n for _, n in frame_people), default=0)
        ev["max_persons"] = max_persons
        kinds = set(objects)
        kind = "person" if kinds & PEOPLE_LIKE else "vehicle" if kinds & VEHICLES else \
            "animal" if kinds & ANIMALS else ("object" if kinds else
                                              {"tamper": "camera-tamper", "lighting": "lighting-change"}.get(
                                                  ev.get("origin", ""), "motion-only"))
        if cfg.require_person_for_event and kind in ("motion-only", "object", "lighting-change"):
            return False

        # ---------------- keyframes, semantic tags
        kept = [x for x in slots if x is not None]
        luma = frame_brightness([x[2] for x in kept])
        semantic = []
        if self.sem is not None and kept:
            try:
                embs = self.sem.encode_images([x[2] for x in kept])
                semantic = self.sem.classify(embs)
                for x, emb in zip(kept, embs):
                    self.sem_keys.append(f"{eid}|{x[1]:.2f}")
                    self.sem_embs.append(emb.astype(np.float16))
            except Exception as ex:
                log.warning("  semantic failed: %s", ex)
        prelim, _ = R.score_event(ev, summaries, cfg, None, semantic, tamper)

        # ---------------- snapshots
        snaps = []
        sheet_imgs, sheet_caps, raw_imgs = [], [], []
        for j, (_, t, fr, dets, ffs) in enumerate(kept, 1):
            when = base + dt.timedelta(seconds=t) if base else None
            hdr = f"{camera} | {_iso(when) or ''} | {eid} | video {V.fmt_offset(t)}"
            ann = A.draw_dets(fr, dets, ffs, id_map, hdr, zones)
            ap = _imwrite(os.path.join(edir, f"snap{j:02d}_{V.fmt_offset(t).replace(':', '')}_annotated.jpg"),
                          ann, cfg.snapshot_jpeg_quality)
            rp = _imwrite(os.path.join(edir, f"snap{j:02d}_{V.fmt_offset(t).replace(':', '')}_raw.jpg"),
                          fr, cfg.snapshot_jpeg_quality) if cfg.save_raw_snapshots else ""
            npers = sum(1 for d in dets if d["cls"] == "person")
            snaps.append({"t_off": round(t, 2), "dt": _iso(when), "annotated": ap, "raw": rp,
                          "persons": npers, "objects": sorted({d["cls"] for d in dets}),
                          "semantic": semantic[j - 1]["top_label"] if j - 1 < len(semantic) else ""})
            sheet_imgs.append(ann)
            raw_imgs.append(fr)
            sheet_caps.append(f"{j}. {V.fmt_offset(t)}  ({_iso(when)[11:] if when else ''})")
        sheet = A.contact_sheet(sheet_imgs, sheet_caps)
        sheet_path = _imwrite(os.path.join(edir, "contact_sheet.jpg"), sheet, 85)

        # ---------------- VLM description
        vlm_res = None
        want_vlm = cfg.use_vlm and (cfg.vlm_scope == "all" or (cfg.vlm_scope == "people" and persons) or
                                    (cfg.vlm_scope == "flagged" and (prelim >= cfg.suspicion_alert_threshold * 0.5 or
                                                                     (persons and R.is_after_hours(ev_dt, cfg)))))
        if want_vlm and self._vlm_count < cfg.vlm_max_events and self.vlm is not None:
            try:
                ctx = f"{len(persons)} person track(s); objects: {', '.join(sorted(objects))}"
                if plates:
                    ctx += "; plates: " + ", ".join(p["text"] for p in plates)
                vlm_res = self.vlm.describe_event(A.contact_sheet(raw_imgs, sheet_caps, tile_w=448), ctx)
                self._vlm_count += 1
                self._vlm_errors = 0
            except Exception as ex:
                self._vlm_failure(ex)

        score, flags = R.score_event(ev, summaries, cfg, vlm_res, semantic, tamper)
        level = "HIGH" if score >= 70 else "REVIEW" if score >= cfg.suspicion_alert_threshold else \
            "LOW" if score > 0 else "NORMAL"

        # ---------------- clip
        clip = ""
        if cfg.save_clips and (cfg.clip_scope == "all" or (cfg.clip_scope == "people" and (persons or score > 0)) or
                               (cfg.clip_scope == "flagged" and score >= cfg.suspicion_alert_threshold)):
            cp = os.path.join(edir, "clip.mp4")
            lab = f"{camera} {eid} start {_iso(ev_dt) or V.fmt_offset(s)}" if cfg.clip_overlay else ""
            if V.export_clip(path, s, min(dur, cfg.clip_max_seconds), cp, cfg.clip_width, lab):
                clip = cp

        # ---------------- persist
        for sm in summaries:
            self.store.insert("tracks", {
                "event_id": eid, "track": sm["track"], "cls": sm["cls"], "first_off": round(sm["first"], 2),
                "last_off": round(sm["last"], 2), "first_dt": _at(base, sm["first"]),
                "duration": round(sm["duration"], 2), "frames": sm["frames"], "max_conf": round(sm["max_conf"], 3),
                "mean_conf": round(sm["mean_conf"], 3), "max_speed": round(sm["max_speed"], 2),
                "zones": {z: round(v, 1) for z, v in sm["zones"].items()}, "identity": sm["identity"],
                "identity_sim": sm["identity_sim"], "known_name": sm["known_name"], "face_crop": sm["face_crop"],
                "person_crop": sm["person_crop"], "hidden_face_s": round(sm["hidden_face_s"], 1),
                "posture": {"hands_up_s": round(sm["hands_up_s"], 1), "crouch_s": round(sm["crouch_s"], 1)},
                "faces_found": sm["faces_found"], "path": sm["path"], "watchlist": sm["watchlist"],
                "crossings": sm["crossings"]})
        self.store.insert_many("detections", det_rows)
        idents = sorted({(x["known_name"] or x["identity"]) for x in persons if x["identity"]})
        self.store.insert("events", {
            "event_id": eid, "video_id": vid, "video_path": path, "rel_path": rel, "camera": camera,
            "start_off": round(s, 2), "end_off": round(e, 2), "start_dt": _iso(ev_dt),
            "end_dt": _at(base, e), "duration": round(dur, 2),
            "kind": kind, "motion_peak": round(ev["motion_peak"], 4), "motion_mean": round(ev["motion_mean"], 4),
            "max_persons": max_persons, "n_tracks": len(summaries), "n_person_tracks": len(persons),
            "objects": objects, "faces_detected": faces_total, "identities": idents,
            "best_conf": round(max((x["max_conf"] for x in summaries), default=0.0), 3),
            "score": round(score, 1), "level": level, "flags": flags,
            "semantic": semantic[:K], "vlm": vlm_res, "snapshots": snaps, "contact_sheet": sheet_path,
            "clip_path": clip, "folder": edir, "after_hours": int(R.is_after_hours(ev_dt, cfg)),
            "frames_analyzed": nframes, "run_id": self.run_id, "time_source": ev_src,
            "luma_p99": round(luma, 1), "plates": [p["text"] for p in plates], "crossings": dict(crossings)})
        self.new_days.add(ev_dt.strftime("%Y-%m-%d") if ev_dt else "unknown-date")
        if any(f["flag"] == "unauthorized_person" for f in flags):
            for sm in persons:
                if not sm.get("known_name"):
                    self.store.insert("face_alerts", {
                        "run_id": self.run_id, "kind": "unauthorized", "identity": sm.get("identity") or "",
                        "dt": _at(base, sm["first"]), "camera": camera, "event_id": eid, "video": rel,
                        "t_off": round(sm["first"], 2), "face_crop": sm.get("face_crop", ""),
                        "person_crop": sm.get("person_crop", ""),
                        "note": "not a known person on an authorised-only camera"}, replace=False)
        log.info("  event %s %s→%s %s persons=%d tracks=%d faces=%d plates=%s score=%.0f %s", eid, V.fmt_offset(s),
                 V.fmt_offset(e), kind, len(persons), len(summaries), faces_total,
                 ",".join(p["text"] for p in plates) or "-", score, ",".join(f["flag"] for f in flags))
        return True


def run(cfg: Config, progress=None, build_reports=True, cancel=None) -> dict:
    return Analyzer(cfg, progress, cancel).run(build_reports)

In [ ]:
%%writefile /content/cctv_analysis/retime.py
"""Fix recording times of already-analysed footage without analysing it again.

For every finished video: resolve the start time again (current rules: file name → DVR clock
→ metadata → file time), re-read each event's own DVR clock, then recompute every stored
timestamp as ``base + offset`` (events, tracks, faces, detections, plates, alerts, tamper),
refresh identity first/last-seen, and rebuild the reels/reports of every day that changed.
Event IDs and event folders keep their original names (they are identifiers, not times).

  python -m cctv_analysis --input /videos --output /results --retime
"""
from __future__ import annotations

import datetime as dt
import json
import logging
import os
from typing import Optional

from . import timeline as T
from . import video_io as V
from .storage import Store

log = logging.getLogger("cctv")
FMT = "%Y-%m-%d %H:%M:%S"


def _iso(d: Optional[dt.datetime]) -> Optional[str]:
    return d.strftime(FMT) if d else None


def _shift_sql(table: str, col: str, off_col: str, key: str):
    # SQLite strftime truncates fractional seconds exactly like the analyzer's strftime does
    return (f"UPDATE {table} SET {col} = CASE WHEN ? IS NULL THEN NULL ELSE "
            f"strftime('%Y-%m-%d %H:%M:%S', ?, '+' || {off_col} || ' seconds') END WHERE {key} = ?")


def retime(cfg, progress=None, cancel=None, build_reports: bool = True) -> dict:
    from . import modelcache
    from .ocr import get_ocr
    problems = cfg.validate()
    if problems:
        raise ValueError("Invalid configuration:\n  - " + "\n  - ".join(problems))
    modelcache.init(cfg.model_cache_dir)
    store = Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))
    ocr = get_ocr(cfg.ocr_engine)
    vids = store.query("SELECT id, path, start_dt, time_source FROM videos WHERE status='done' ORDER BY id")
    stats: dict = {"videos": 0, "videos_changed": 0, "events_changed": 0, "missing": 0}
    days = set()
    db = store.db
    for i, v in enumerate(vids, 1):
        if cancel is not None and cancel.is_set():
            stats["cancelled"] = 1
            break
        if progress:
            progress({"stage": "retime", "index": i, "total": len(vids), "path": v["path"],
                      "percent": round(100 * i / max(1, len(vids)), 1)})
        if not os.path.exists(v["path"]):
            stats["missing"] += 1
            log.warning("retime: source missing, kept as is: %s", v["path"])
            continue
        stats["videos"] += 1
        meta = V.probe(v["path"])
        start, src = T.resolve_start_time(v["path"], meta, cfg, ocr)
        if _iso(start) != v["start_dt"] or src != v["time_source"]:
            stats["videos_changed"] += 1
            log.info("retime %s: %s (%s) -> %s (%s)", os.path.basename(v["path"]), v["start_dt"], v["time_source"],
                     _iso(start), src)
        db.execute("UPDATE videos SET start_dt=?, end_dt=?, time_source=? WHERE id=?",
                   (_iso(start), _iso(start + dt.timedelta(seconds=meta["duration"])) if start else None, src, v["id"]))
        b = _iso(start)
        db.execute("UPDATE tamper SET start_dt = CASE WHEN ? IS NULL THEN NULL ELSE "
                   "strftime('%Y-%m-%d %H:%M:%S', ?, '+' || start_off || ' seconds') END WHERE video_id = ?",
                   (b, b, v["id"]))
        for e in store.query("SELECT event_id, start_off, end_off, start_dt FROM events WHERE video_id=?", (v["id"],)):
            base, esrc = T.event_clock(v["path"], meta, start, src, e["start_off"], cfg, ocr)
            bs = _iso(base)
            new_start = _iso(base + dt.timedelta(seconds=e["start_off"])) if base else None
            if new_start != e["start_dt"]:
                stats["events_changed"] += 1
            days.add((e["start_dt"] or "unknown-date")[:10] if e["start_dt"] else "unknown-date")
            days.add(new_start[:10] if new_start else "unknown-date")
            db.execute("UPDATE events SET start_dt=?, end_dt=?, time_source=? WHERE event_id=?",
                       (new_start, _iso(base + dt.timedelta(seconds=e["end_off"])) if base else None, esrc,
                        e["event_id"]))
            for table, col, off in (("tracks", "first_dt", "first_off"), ("faces", "dt", "t_off"),
                                    ("detections", "dt", "t_off"), ("plates", "dt", "t_off"),
                                    ("face_alerts", "dt", "t_off")):
                db.execute(_shift_sql(table, col, off, "event_id"), (bs, bs, e["event_id"]))
        store.commit(sync=False)
    _refresh_identities(store, cfg)
    store.commit()
    stats["days_affected"] = sorted(days)
    if build_reports:
        from .report import build_reports as br
        br(store, cfg, days)
        store.commit()
    store.close()
    log.info("retime finished: %s", stats)
    return stats


def _refresh_identities(store, cfg) -> None:
    path = os.path.join(cfg.output_dir, "identities.json")
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8") as fh:
        ids = json.load(fh)
    span = {r["identity"]: (r["a"], r["b"]) for r in store.query(
        "SELECT identity, MIN(first_dt) a, MAX(first_dt) b FROM tracks WHERE identity IS NOT NULL AND identity != '' "
        "GROUP BY identity")}
    for r in ids:
        if r["id"] in span and span[r["id"]][0]:
            r["first_seen"], r["last_seen"] = span[r["id"]]
    with open(path + ".tmp", "w", encoding="utf-8") as fh:
        json.dump(ids, fh, indent=1)
    os.replace(path + ".tmp", path)

In [ ]:
%%writefile /content/cctv_analysis/evidence.py
"""Evidence packages for police / insurers / legal review.

``export_evidence`` writes one ZIP containing, per selected event:

* ``original_segment.<ext>`` – the event cut from the source recording by **stream copy**
  (no re-encoding: picture data identical to the recording; the cut starts at a keyframe, so it
  can begin a few seconds before the event, which is recorded in event.json),
* the event clip, raw + annotated snapshots, contact sheet, face / person / plate crops,
* ``event.json`` – times, time source, camera, source file + offsets, flags with reasons, AI
  description, plates, identities,

plus ``incident_report.html`` (printable), ``manifest.json`` (SHA-256 of every file in the
package and of each *full* source recording, tool version, operator, case id, export time)
and ``manifest.sha256`` (the manifest's own hash, to quote in correspondence).

``redact=True`` blurs faces (and number plates when ``redact_plates``) in every image and clip
and leaves out the unredacted original segments – for sharing with people who must not see
bystanders.
"""
from __future__ import annotations

import datetime as dt
import hashlib
import html
import json
import logging
import os
import shutil
import sqlite3
import subprocess
import tempfile
import zipfile
from typing import TYPE_CHECKING, Dict, Iterable, List, Optional, Union

import cv2
import numpy as np

from . import __version__
from . import video_io as V
from .storage import resolve

if TYPE_CHECKING:
    from .plates import ClassicPlateDetector, YoloPlateDetector

log = logging.getLogger("cctv")


def sha256_file(path: str, chunk: int = 4 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _cut_original(src: str, start: float, end: float, out: str) -> bool:
    if not V.FFMPEG:
        return False
    if V.is_raw(src):
        # a copy cut of a raw stream starts at the first keyframe AFTER the seek point: start early
        # so the cut covers the whole event (DVR GOPs are 1-2 s; 3 s leaves a margin)
        start = max(0.0, start - 3.0)
    pre, post = V.seek_args(src, start, copy=True)          # always the original, never a proxy
    r = subprocess.run([V.FFMPEG, "-nostdin", "-y", "-v", "error", *pre, *V._input_args(src), "-i", src, *post,
                        "-t", f"{max(0.5, end - start):.3f}", "-map", "0", "-c", "copy", "-avoid_negative_ts",
                        "make_zero", out], capture_output=True, timeout=1800)
    return r.returncode == 0 and os.path.exists(out) and os.path.getsize(out) > 0


class Redactor:
    """Blurs faces (YuNet) and optionally number plates (plate detector) in images and videos."""

    def __init__(self, cfg, plates: bool = False):
        from .faces import FaceEngine
        self.fe = FaceEngine(cfg)
        self.pd: Optional[Union["YoloPlateDetector", "ClassicPlateDetector"]] = None
        if plates:
            try:
                from .plates import YoloPlateDetector
                self.pd = YoloPlateDetector()
            except Exception as e:
                from .plates import ClassicPlateDetector
                log.warning("plate detector unavailable for redaction (%s) – using OpenCV plate finder", e)
                self.pd = ClassicPlateDetector()

    def boxes(self, img: np.ndarray) -> List[List[int]]:
        out = []
        for f in self.fe.detect_full(img):
            x1, y1, x2, y2 = f["box"]
            w, h = x2 - x1, y2 - y1
            out.append([int(x1 - 0.25 * w), int(y1 - 0.35 * h), int(x2 + 0.25 * w), int(y2 + 0.25 * h)])
        if self.pd is not None:
            out += [[int(v) for v in b] for b, _ in self.pd.find(img)]
        return out

    def image(self, img: np.ndarray) -> np.ndarray:
        img = img.copy()
        H, W = img.shape[:2]
        for x1, y1, x2, y2 in self.boxes(img):
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
            if x2 > x1 and y2 > y1:
                roi = img[y1:y2, x1:x2]
                k = max(15, (max(roi.shape[:2]) // 3) | 1)
                img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
        return img

    def video(self, src: str, out: str, fps: float = 10.0, width: int = 960) -> bool:
        meta = V.probe(src)
        if not meta["width"]:
            return False
        w = min(width, meta["width"]) // 2 * 2
        h = int(round(meta["height"] * w / meta["width"])) // 2 * 2
        proc = subprocess.Popen([V.FFMPEG, "-nostdin", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
                                 "-s", f"{w}x{h}", "-r", str(fps), "-i", "pipe:0", *V.h264_args(24), out],
                                stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
        n = 0
        try:
            for _, f in V.iter_frames(src, fps, w, src_size=(meta["width"], meta["height"])):
                f = cv2.resize(f, (w, h)) if f.shape[1] != w or f.shape[0] != h else f
                proc.stdin.write(self.image(f).tobytes())
                n += 1
            proc.stdin.close()
            proc.wait(timeout=1800)
        except Exception as e:
            log.warning("redaction of %s failed: %s", src, e)
            proc.kill()
            return False
        return proc.returncode == 0 and n > 0


def _report_html(case: dict, events: List[dict], files: dict) -> str:
    E = html.escape
    rows = []
    for e in events:
        flags = "".join(f"<li><b>{E(f['flag'])}</b> (+{f['points']:g}) {E(f['reason'])}</li>" for f in e["flags"])
        snaps = "".join(f'<img src="{E(p)}" style="width:32%;margin:2px">' for p in e["images"][:6])
        vlm = (e.get("vlm") or {}).get("description", "")
        plates = ", ".join(f"{p['text']} ({p['confidence']:.2f})" for p in e["plates"]) or "–"
        rows.append(f"""<section><h2>{E(e['event_id'])} — {E(e['level'])} {e['score']:g}</h2>
<table><tr><th>Camera</th><td>{E(e['camera'] or '')}</td><th>Recorded</th><td>{E(e['start_dt'] or 'unknown')} → {E((e['end_dt'] or '')[11:])}</td></tr>
<tr><th>Time source</th><td>{E(e['time_source'] or '')}</td><th>Duration</th><td>{e['duration']:.0f} s</td></tr>
<tr><th>Source file</th><td colspan=3>{E(e['rel_path'])} @ {V.fmt_offset(e['start_off'])}–{V.fmt_offset(e['end_off'])}<br>
<small>SHA-256 {E(files.get(e['video_path'], ''))}</small></td></tr>
<tr><th>Original segment</th><td colspan=3>{E(e.get('original_note', ''))}</td></tr>
<tr><th>Number plates</th><td>{E(plates)}</td><th>People / identities</th><td>{e['n_person_tracks']} / {E(', '.join(e['identities'] or []))}</td></tr></table>
{f'<p><b>AI description:</b> {E(vlm)}</p>' if vlm else ''}<ul>{flags or '<li>no flags</li>'}</ul>{snaps}</section>""")
    return f"""<!doctype html><html><head><meta charset="utf-8"><title>Incident report {E(case['case_id'] or '')}</title>
<style>body{{font-family:Arial,sans-serif;margin:24px;color:#111}} h1{{font-size:20px}} h2{{font-size:16px;border-bottom:1px solid #999}}
table{{border-collapse:collapse;width:100%;font-size:13px;margin:6px 0}} th,td{{border:1px solid #bbb;padding:4px;text-align:left;vertical-align:top}}
th{{background:#eee;width:14%}} section{{page-break-inside:avoid;margin-bottom:18px}} small{{color:#555}}
@media print {{ a{{color:#000}} }}</style></head><body>
<h1>CCTV incident report{(' — case ' + E(case['case_id'])) if case['case_id'] else ''}</h1>
<table><tr><th>Exported</th><td>{E(case['exported_at'])}</td><th>Operator</th><td>{E(case['operator'] or '')}</td></tr>
<tr><th>Tool</th><td>home-camera-analysis {E(__version__)}</td><th>Events</th><td>{len(events)}</td></tr>
<tr><th>Redacted</th><td>{'yes – faces' + (' and plates' if case['redact_plates'] else '') + ' blurred; originals not included' if case['redacted'] else 'no'}</td>
<th>Notes</th><td>{E(case['notes'] or '')}</td></tr></table>
<p><small>Integrity: every file in this package is listed with its SHA-256 in manifest.json; the manifest's own hash is in
manifest.sha256. Recording times come from the source named in "Time source" (filename, DVR clock via OCR, metadata, or an
approximate file time). Scores are a triage aid, not a finding of fact.</small></p>
{''.join(rows)}</body></html>"""


def export_evidence(out_dir: str, event_ids: Iterable[str], dest_zip: str, cfg=None, operator: str = "",
                    case_id: str = "", notes: str = "", redact: bool = False, redact_plates: bool = False,
                    progress=None) -> dict:
    from .config import Config
    cfg = cfg or Config(output_dir=out_dir)
    ids = list(dict.fromkeys(event_ids))
    if not ids:
        raise ValueError("no events selected")
    db = sqlite3.connect(f"file:{os.path.join(out_dir, 'analysis.db')}?mode=ro", uri=True)
    db.row_factory = sqlite3.Row
    q = ",".join("?" * len(ids))
    evs = [dict(r) for r in db.execute(f"SELECT * FROM events WHERE event_id IN ({q})", ids)]
    found = {e["event_id"] for e in evs}
    missing = [i for i in ids if i not in found]
    if missing:
        raise KeyError(f"unknown event id(s): {', '.join(missing[:5])}")
    plates: Dict[str, List[dict]] = {}
    for r in db.execute(f"SELECT * FROM plates WHERE event_id IN ({q})", ids):
        plates.setdefault(r["event_id"], []).append(dict(r))
    tracks: Dict[str, List[dict]] = {}
    for r in db.execute(f"SELECT event_id, track, cls, identity, known_name, face_crop, person_crop FROM tracks "
                        f"WHERE event_id IN ({q})", ids):
        tracks.setdefault(r["event_id"], []).append(dict(r))
    db.close()
    red = Redactor(cfg, redact_plates) if redact else None
    exported_at = dt.datetime.now().astimezone().isoformat(timespec="seconds")
    case = {"case_id": case_id, "operator": operator, "notes": notes, "exported_at": exported_at,
            "redacted": bool(redact), "redact_plates": bool(redact_plates)}
    src_hash = {}
    report_events = []
    tmp = tempfile.mkdtemp(prefix="evidence_")
    try:
        for n, e in enumerate(sorted(evs, key=lambda x: (x["start_dt"] or "", x["event_id"])), 1):
            if progress:
                progress(n, len(evs), e["event_id"])
            for k in ("flags", "vlm", "snapshots", "objects", "identities", "semantic", "plates", "crossings"):
                try:
                    e[k] = json.loads(e[k]) if isinstance(e.get(k), str) else e.get(k)
                except (TypeError, json.JSONDecodeError):
                    pass
            e["flags"] = e.get("flags") or []
            d = os.path.join(tmp, "events", e["event_id"])
            os.makedirs(d, exist_ok=True)
            src = e["video_path"]
            if os.path.exists(src) and src not in src_hash:
                src_hash[src] = sha256_file(src)
            note = {}
            if not redact:                       # a package without the original says so, and why
                if not os.path.exists(src):
                    note["original_segment"] = (f"NOT INCLUDED – the recording {e['rel_path']} was not found "
                                                "(drive unplugged or moved?)")
                elif _cut_original(src, e["start_off"], e["end_off"],
                                   os.path.join(d, "original_segment" + (os.path.splitext(src)[1] or ".mp4"))):
                    note["original_segment"] = ("stream copy of the recording (not re-encoded); it may begin "
                                                "a few seconds before the event start (keyframe boundary)")
                else:
                    note["original_segment"] = "NOT INCLUDED – the recording could not be cut (see the app log)"
            images = []

            def put(path, name, d=d):
                path = resolve(out_dir, path)                # follows a moved output folder
                if not path:
                    return None
                dst = os.path.join(d, name)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if red is not None and name.lower().endswith((".jpg", ".png")):
                    img = cv2.imread(path)
                    if img is None:
                        return None
                    cv2.imwrite(dst, red.image(img), [cv2.IMWRITE_JPEG_QUALITY, 92])
                elif red is not None and name.lower().endswith(".mp4"):
                    if not red.video(path, dst):
                        return None
                else:
                    shutil.copyfile(path, dst)
                return os.path.relpath(dst, tmp).replace(os.sep, "/")

            for i, s in enumerate(e.get("snapshots") or [], 1):
                p = put(s.get("raw"), f"snapshots/snap{i:02d}_raw.jpg")
                a = put(s.get("annotated"), f"snapshots/snap{i:02d}_annotated.jpg")
                images += [x for x in (a or p,) if x]
            put(e.get("contact_sheet"), "contact_sheet.jpg")
            put(e.get("clip_path"), "clip.mp4")
            for t in tracks.get(e["event_id"], []):
                put(t["face_crop"], f"faces/track{t['track']:03d}.jpg")
                put(t["person_crop"], f"people/track{t['track']:03d}.jpg")
            pl = plates.get(e["event_id"], [])
            for p in pl:
                put(p["crop_path"], f"plates/track{p['track']:03d}_plate.jpg")
                put(p["enhanced_path"], f"plates/track{p['track']:03d}_enhanced.jpg")
            meta = {k: e.get(k) for k in ("event_id", "camera", "start_dt", "end_dt", "duration", "time_source",
                                         "rel_path", "start_off", "end_off", "kind", "level", "score", "flags",
                                         "identities", "max_persons", "n_person_tracks", "objects", "crossings")}
            meta["ai_description"] = (e.get("vlm") or {}).get("description", "") if isinstance(e.get("vlm"), dict) else ""
            meta["plates"] = [{k: p[k] for k in ("text", "confidence", "valid_format", "vehicle", "t_off", "dt",
                                                  "method", "watchlist")} for p in pl]
            meta["source_sha256"] = src_hash.get(src, "")
            meta["notes"] = note
            with open(os.path.join(d, "event.json"), "w", encoding="utf-8") as fh:
                json.dump(meta, fh, indent=1, default=str)
            report_events.append({**e, "plates": [p for p in meta["plates"] if p["text"]],
                                  "original_note": note.get("original_segment", "not included (redacted package)"),
                                   "images": [f"events/{e['event_id']}/" + x.split(f"events/{e['event_id']}/", 1)[-1]
                                              for x in images]})
        with open(os.path.join(tmp, "incident_report.html"), "w", encoding="utf-8") as fh:
            fh.write(_report_html(case, report_events, src_hash))
        files = {}
        for root, _, fs in os.walk(tmp):
            for f in fs:
                p = os.path.join(root, f)
                files[os.path.relpath(p, tmp).replace(os.sep, "/")] = sha256_file(p)
        manifest = {"tool": f"home-camera-analysis {__version__}", **case,
                    "events": [e["event_id"] for e in report_events],
                    "sources": [{"path": os.path.relpath(p, cfg.input_dir) if cfg.input_dir and os.path.isdir(cfg.input_dir)
                                 else os.path.basename(p), "sha256": h, "bytes": os.path.getsize(p)}
                                for p, h in src_hash.items()],
                    "files": files}
        mtext = json.dumps(manifest, indent=1, sort_keys=True)
        with open(os.path.join(tmp, "manifest.json"), "w", encoding="utf-8") as fh:
            fh.write(mtext)
        mhash = hashlib.sha256(mtext.encode()).hexdigest()
        with open(os.path.join(tmp, "manifest.sha256"), "w", encoding="utf-8") as fh:
            fh.write(f"{mhash}  manifest.json\n")
        os.makedirs(os.path.dirname(os.path.abspath(dest_zip)), exist_ok=True)
        part = dest_zip + ".part"
        with zipfile.ZipFile(part, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, fs in os.walk(tmp):
                for f in sorted(fs):
                    p = os.path.join(root, f)
                    z.write(p, os.path.relpath(p, tmp))
        os.replace(part, dest_zip)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    return {"zip": dest_zip, "zip_sha256": sha256_file(dest_zip), "manifest_sha256": mhash,
            "events": len(report_events), "files": len(files) + 2, "redacted": bool(redact)}


def verify_package(zip_path: str) -> dict:
    """Re-hash every file of an evidence ZIP against its manifest."""
    with zipfile.ZipFile(zip_path) as z:
        manifest_text = z.read("manifest.json").decode()
        manifest = json.loads(manifest_text)
        expect = z.read("manifest.sha256").decode().split()[0]
        bad = [n for n, h in manifest["files"].items() if hashlib.sha256(z.read(n)).hexdigest() != h]
        extra = [n for n in z.namelist() if n not in manifest["files"] and n not in ("manifest.json", "manifest.sha256")]
    ok_manifest = hashlib.sha256(manifest_text.encode()).hexdigest() == expect
    return {"ok": ok_manifest and not bad and not extra, "manifest_ok": ok_manifest, "modified": bad, "unlisted": extra,
            "files": len(manifest["files"])}

In [ ]:
%%writefile /content/cctv_analysis/sysinfo.py
"""Hardware / software check, model inventory, a short on-this-machine benchmark and the
processing-time estimate shown before a job starts.

Every estimate is a projection: it multiplies footage length by rates measured on this
computer (``benchmark``) or, before a benchmark has run, by conservative defaults. The
assumptions are returned with the numbers so the UI can show them."""
from __future__ import annotations

import importlib.util
import logging
import os
import platform
import shutil
import subprocess
import time
from typing import Dict, List, Optional

log = logging.getLogger("cctv")

# Rates used before a benchmark has run: measured with `benchmark()` (balanced preset) on the
# development machine – Intel Xeon 2.8 GHz, 4 vCPU, no GPU – on OpenCV's vtest.avi (detector,
# motion, encode) and a cloud clip with a DVR clock (OCR). See docs/PERFORMANCE.md. The detector
# rate includes pose. VLM figures are not measured here (labelled assumptions).
DEFAULT_RATES = {
    "motion_x_realtime": 9.9,         # pass-1 decode + motion analysis, × real time
    "detect_fps": 2.9,                # detector (+ pose) frames per second
    "face_s_per_frame": 0.03,         # extra face work per detection frame with people (assumption)
    "pose_s_per_frame": 0.0,          # included in detect_fps
    "ocr_s_per_read": 0.7,            # one DVR-clock read on a frame that has a clock
    "reel_x_realtime": 10.3,          # highlight-reel encode, × real time
    "plate_s_per_track": 1.5,         # measured: ~12 s plain vs ~45-110 s enhanced per 1-car clip
    "plate_deblur_s_per_track": 60.0,
    "vlm_s_per_event_transformers_cpu": 45.0,     # assumption (2B model, CPU)
    "vlm_s_per_event_transformers_gpu": 4.0,      # assumption
    "vlm_s_per_event_openai": 8.0,                # assumption (depends on the server)
    "semantic_s_per_event": 1.0,                  # assumption
}
ASSUMED_MOTION_FRACTION = 0.15        # share of footage with movement (typical shop/home DVR)
ASSUMED_EVENT_SECONDS = 30.0


def _has(mod: str) -> bool:
    try:
        return importlib.util.find_spec(mod) is not None
    except (ImportError, ValueError):
        return False


def _ffmpeg_info() -> dict:
    from . import video_io as V
    out = {"ffmpeg": bool(V.FFMPEG), "ffprobe": bool(V.FFPROBE), "version": "", "cuda_decode": False}
    if V.FFMPEG:
        try:
            r = subprocess.run([V.FFMPEG, "-hide_banner", "-version"], capture_output=True, text=True, timeout=20)
            out["version"] = (r.stdout.splitlines() or [""])[0][:80]
            r = subprocess.run([V.FFMPEG, "-hide_banner", "-hwaccels"], capture_output=True, text=True, timeout=20)
            out["cuda_decode"] = "cuda" in r.stdout
        except Exception:
            pass
    return out


def gpu_info() -> dict:
    """{'available', 'name', 'memory_gb', 'backend'} – torch first, then nvidia-smi."""
    if _has("torch"):
        try:
            import torch
            if torch.cuda.is_available():
                p = torch.cuda.get_device_properties(0)
                return {"available": True, "name": p.name, "memory_gb": round(p.total_memory / 1e9, 1),
                        "backend": "cuda"}
            if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
                return {"available": True, "name": "Apple GPU (MPS)", "memory_gb": None, "backend": "mps"}
        except Exception as e:
            log.debug("torch GPU probe failed: %s", e)
    smi = shutil.which("nvidia-smi")
    if smi:
        try:
            r = subprocess.run([smi, "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True, timeout=20)
            name, mem = r.stdout.strip().splitlines()[0].split(",")
            return {"available": True, "name": name.strip(), "memory_gb": round(float(mem) / 1024, 1),
                    "backend": "cuda (driver only – install CUDA PyTorch to use it)"}
        except Exception:
            pass
    return {"available": False, "name": "", "memory_gb": None, "backend": "cpu"}


def _nearest_existing(path: str) -> str:
    p = os.path.abspath(path or ".")
    while not os.path.exists(p) and os.path.dirname(p) != p:
        p = os.path.dirname(p)
    return p


def system_check(output_dir: str = "", model_dir: str = "") -> dict:
    import psutil
    vm = psutil.virtual_memory()
    du = shutil.disk_usage(_nearest_existing(output_dir))
    gpu = gpu_info()
    ff = _ffmpeg_info()
    from .ocr import available_engines
    deps = {"opencv": True, "ultralytics": _has("ultralytics"), "torch": _has("torch"),
            "transformers": _has("transformers"), "rapidocr": _has("rapidocr_onnxruntime"),
            "fast_plate_ocr": _has("fast_plate_ocr"), "open_image_models": _has("open_image_models"),
            "xlsxwriter": _has("xlsxwriter"), "matplotlib": _has("matplotlib")}
    ram = vm.total / 1e9
    checks: List[dict] = []

    def chk(name, ok, detail, level="error"):
        checks.append({"item": name, "ok": bool(ok), "level": "ok" if ok else level, "detail": detail})

    chk("ffmpeg", ff["ffmpeg"] and ff["ffprobe"], ff["version"] or "ffmpeg/ffprobe not found – install ffmpeg")
    chk("Object detector (YOLO)", deps["ultralytics"] and deps["torch"],
        "ultralytics + torch installed" if deps["ultralytics"] else "pip install ultralytics torch")
    chk("DVR clock reader (OCR)", any(available_engines().values()),
        ", ".join(k for k, v in available_engines().items() if v) or "pip install rapidocr-onnxruntime",
        "warning")
    chk("Number-plate models", deps["fast_plate_ocr"] and deps["open_image_models"],
        "fast-plate-ocr + open-image-models" if deps["fast_plate_ocr"] else "pip install fast-plate-ocr open-image-models",
        "warning")
    chk("Memory", ram >= 8, f"{ram:.1f} GB RAM ({vm.available / 1e9:.1f} GB free)", "warning")
    chk("Local AI descriptions (VLM on this PC)", gpu["available"] or vm.available / 1e9 >= 10,
        "GPU available" if gpu["available"] else
        "CPU only: a 2B model needs ~9 GB free RAM and ~45 s per event – consider an Ollama/LM Studio server",
        "warning")
    chk("Free disk space", du.free / 1e9 >= 5, f"{du.free / 1e9:.1f} GB free at {_nearest_existing(output_dir)}",
        "warning")
    chk("GPU", gpu["available"], gpu["name"] or "no GPU – everything runs on the CPU (slower)", "info")
    return {"os": f"{platform.system()} {platform.release()}", "python": platform.python_version(),
            "cpu": {"cores": os.cpu_count(), "model": platform.processor() or platform.machine(),
                    "freq_mhz": int(psutil.cpu_freq().max) if psutil.cpu_freq() else None},
            "ram_gb": round(ram, 1), "ram_free_gb": round(vm.available / 1e9, 1), "gpu": gpu,
            "disk_free_gb": round(du.free / 1e9, 1), "ffmpeg": ff, "dependencies": deps,
            "models": model_inventory(model_dir), "checks": checks}


def model_inventory(model_dir: str = "") -> List[dict]:
    from . import modelcache
    root = model_dir or modelcache.root()

    def present(*parts, min_bytes=100_000):
        p = os.path.join(root, *parts)
        if os.path.isdir(p):
            return any(os.path.getsize(os.path.join(dp, f)) >= min_bytes
                       for dp, _, fs in os.walk(p) for f in fs)
        return os.path.exists(p) and os.path.getsize(p) >= min_bytes

    items = [
        ("yolo26n.pt", "Object detector – fast", ("yolo", "yolo26n.pt")),
        ("yolo26s.pt", "Object detector – balanced", ("yolo", "yolo26s.pt")),
        ("yolo26m.pt", "Object detector – accurate", ("yolo", "yolo26m.pt")),
        ("yolo26n-pose.pt", "Posture (pose)", ("yolo", "yolo26n-pose.pt")),
        ("yunet", "Face detector (YuNet)", ("opencv", "face_detection_yunet_2023mar.onnx")),
        ("sface", "Face recognition (SFace)", ("opencv", "face_recognition_sface_2021dec.onnx")),
        ("plate-detector", "Number-plate detector", ("plates", "yolo-v9-t-384-license-plate-end2end")),
        ("plate-ocr", "Number-plate reader", ("plates", "cct-s-v2-global-model")),
        ("siglip2", "Scene search (SigLIP-2)", ("huggingface", "models--google--siglip2-base-patch16-224")),
        ("qwen3-vl-2b", "AI descriptions (Qwen3-VL-2B)", ("huggingface", "models--Qwen--Qwen3-VL-2B-Instruct")),
    ]
    return [{"id": i, "name": n, "installed": present(*p)} for i, n, p in items]


def download_model(model_id: str, model_dir: str = "") -> str:
    """Fetch one model into the cache (used by the app's Models page)."""
    from . import modelcache
    if model_dir:
        modelcache.init(model_dir)
    if model_id.startswith("yolo"):
        return modelcache.yolo_weights(model_id)
    if model_id in ("yunet", "sface"):
        from .faces import fetch
        return fetch(model_id)
    if model_id in ("plate-detector", "plate-ocr"):
        from .plates import FastPlateOCR, YoloPlateDetector
        (YoloPlateDetector if model_id == "plate-detector" else FastPlateOCR)()
        return model_id
    if model_id in ("siglip2", "qwen3-vl-2b"):
        from huggingface_hub import snapshot_download
        repo = {"siglip2": "google/siglip2-base-patch16-224", "qwen3-vl-2b": "Qwen/Qwen3-VL-2B-Instruct"}[model_id]
        return snapshot_download(repo, cache_dir=modelcache.hf_dir())
    raise ValueError(f"unknown model {model_id!r}")


# ---------------------------------------------------------------- benchmark

def benchmark(cfg, sample_video: str, seconds: float = 20.0) -> dict:
    """Measure the rates the estimate needs on a real sample (motion pass, detector fps, OCR,
    reel encode). Takes ~20-60 s."""
    from . import modelcache
    from . import video_io as V
    from .motion import scan_motion
    modelcache.init(cfg.model_cache_dir)
    rates = dict(DEFAULT_RATES)
    measured = []
    meta = V.probe(sample_video)
    if not meta["duration"]:
        raise ValueError("sample video unreadable")
    seg = min(seconds, meta["duration"])
    sub = dict(meta, duration=seg)
    t0 = time.time()
    scan_motion(sample_video, sub, cfg, "bench")
    dt_ = max(1e-3, time.time() - t0)
    rates["motion_x_realtime"] = round(seg / dt_, 1)
    measured.append("motion_x_realtime")
    try:
        from .detector import Detector
        det = Detector(cfg)
        frames = [f for _, f in V.iter_frames(sample_video, 2.0, cfg.detect_width, 0, min(seg, 8),
                                              src_size=(meta["width"], meta["height"]))][:12]
        if frames:
            det(frames[0])                                    # warm-up
            t0 = time.time()
            for f in frames:
                det(f)
            rates["detect_fps"] = round(len(frames) / max(1e-3, time.time() - t0), 1)
            measured.append("detect_fps")
    except Exception as e:
        log.warning("benchmark: detector unavailable (%s)", e)
    try:
        from .ocr import get_ocr
        from .timeline import read_clock
        ocr = get_ocr(cfg.ocr_engine)
        if ocr:
            f = V.read_frame_at(sample_video, 0.5, 1280, (meta["width"], meta["height"]))
            read_clock(f, ocr)
            t0 = time.time()
            read_clock(f, ocr)
            rates["ocr_s_per_read"] = round(time.time() - t0, 2)
            measured.append("ocr_s_per_read")
    except Exception as e:
        log.warning("benchmark: OCR unavailable (%s)", e)
    if V.FFMPEG:
        import tempfile

        from .reels import encode_segment
        with tempfile.TemporaryDirectory() as td:
            seg_ev = {"event_id": "bench", "video_path": sample_video, "start_off": 0.0, "end_off": min(seg, 10.0),
                      "camera": "", "level": "", "score": 0, "rel_path": "", "start_dt": None}
            t0 = time.time()
            if encode_segment(seg_ev, os.path.join(td, "b.mp4"), cfg):
                rates["reel_x_realtime"] = round(seg_ev["end_off"] / max(1e-3, time.time() - t0), 1)
                measured.append("reel_x_realtime")
    return {"rates": rates, "measured": measured, "sample": os.path.basename(sample_video)}


# ----------------------------------------------------------------- estimate

def estimate(cfg, total_seconds: float, n_videos: int, rates: Optional[Dict[str, float]] = None,
             n_untimed: int = 0, gpu: bool = False) -> dict:
    """Projected processing time for `total_seconds` of footage in `n_videos` files.
    `n_untimed` = files whose name carries no time (their DVR clock is read by OCR)."""
    r = dict(DEFAULT_RATES, **(rates or {}))
    m = ASSUMED_MOTION_FRACTION
    motion_s = total_seconds * m
    events = max(1.0, motion_s / ASSUMED_EVENT_SECONDS) if total_seconds else 0.0
    parts = {}
    parts["motion scan"] = total_seconds / max(0.1, r["motion_x_realtime"])
    det_frames = motion_s * cfg.detect_fps
    per = 1.0 / max(0.1, r["detect_fps"])
    if cfg.use_faces:
        per += r["face_s_per_frame"]
    if cfg.use_pose:
        per += r["pose_s_per_frame"]
    parts["detection"] = det_frames * per
    clock_reads = n_untimed * (cfg.osd_samples + 1) + (events * (n_untimed / max(1, n_videos))
                                                       if cfg.osd_verify_events else 0)
    if "osd" in cfg.time_sources:
        parts["reading DVR clock"] = clock_reads * r["ocr_s_per_read"]
    if cfg.use_semantic:
        parts["scene tags"] = events * r["semantic_s_per_event"]
    if cfg.use_vlm:
        share = {"all": 1.0, "people": 0.5, "flagged": 0.15}.get(cfg.vlm_scope, 0.15)
        n = min(cfg.vlm_max_events, events * share)
        key = ("vlm_s_per_event_openai" if cfg.vlm_provider == "openai" else
               "vlm_s_per_event_transformers_gpu" if gpu else "vlm_s_per_event_transformers_cpu")
        parts["AI descriptions"] = n * r[key]
    if cfg.use_plates:
        tracks = events * 0.5
        parts["number plates"] = tracks * (r["plate_deblur_s_per_track"] if cfg.plate_enhance
                                           else r["plate_s_per_track"])
    if cfg.make_highlight_reel:
        parts["highlight reels"] = motion_s / max(0.1, r["reel_x_realtime"])
    parts["reports"] = 10 + events * 0.05
    total = sum(parts.values())
    return {"total_seconds": round(total), "breakdown": {k: round(v) for k, v in parts.items()},
            "footage_seconds": round(total_seconds), "videos": n_videos, "projection": True,
            "assumptions": [f"{int(m * 100)}% of the footage contains movement (real share decides the detection time)",
                            f"about one event per {int(ASSUMED_EVENT_SECONDS)} s of movement",
                            "rates measured on this computer" if rates else
                            "rates of a 4-core CPU without GPU (run 'Measure speed' for this computer's real speed)"]}

In [ ]:
%%writefile /content/cctv_analysis/helpers.py
"""Notebook helpers: camera/zone preview, in-notebook review of the top
events (images + playable clips), free-text search and search-by-photo."""
from __future__ import annotations

import base64
import json
import os
from collections import defaultdict
from typing import Optional

import cv2
import numpy as np

from . import annotate as A
from . import timeline as T
from . import video_io as V
from .config import Config
from .motion import detect_osd_mask, poly_mask
from .ocr import get_ocr
from .rules import zones_for
from .storage import Store


def _show(img_bgr, width=900):
    try:
        from IPython.display import Image, display
        ok, buf = cv2.imencode(".jpg", img_bgr)
        display(Image(data=buf.tobytes(), width=width))
    except Exception:
        pass


def _html(s):
    try:
        from IPython.display import HTML, display
        display(HTML(s))
    except Exception:
        print(s)


def grid_overlay(img):
    h, w = img.shape[:2]
    out = img.copy()
    for i in range(1, 10):
        x, y = int(w * i / 10), int(h * i / 10)
        cv2.line(out, (x, 0), (x, h), (255, 255, 255), 1)
        cv2.line(out, (0, y), (w, y), (255, 255, 255), 1)
        cv2.putText(out, f"{i / 10:.1f}", (x + 2, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
        cv2.putText(out, f"{i / 10:.1f}", (2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
    return out


def preview_cameras(cfg: Config, per_camera: int = 1, width: int = 960):
    """List videos, then show one frame per camera with a 0.1 grid (to write zone
    coordinates), configured zones (red), ignore regions (blue) and the
    auto-detected clock mask (magenta)."""
    vids = V.find_videos(cfg)
    by_cam = defaultdict(list)
    for p in vids:
        by_cam[V.camera_name(p, cfg)].append(p)
    tot = 0.0
    print(f"{len(vids)} video(s) in {len(by_cam)} camera group(s)")
    for cam, ps in by_cam.items():
        print(f"\n=== camera '{cam}': {len(ps)} file(s)")
        for p in ps[:per_camera]:
            meta = V.probe(p)
            tot += meta["duration"]
            st, src = T.resolve_start_time(p, meta, cfg, get_ocr(cfg.ocr_engine))
            print(f"  {os.path.relpath(p, cfg.input_dir)} | {meta['width']}x{meta['height']} {meta['fps']:.1f}fps "
                  f"{V.fmt_offset(meta['duration'])} {meta['codec']} | start {st} ({src})")
            f = V.read_frame_at(p, min(5.0, meta["duration"] / 2), width, (meta["width"], meta["height"]))
            if f is None:
                print("   !! could not decode a frame")
                continue
            img = f.copy()
            z = zones_for(cfg, cam)
            if z:
                A.draw_zones(img, z)
            ign = list(cfg.ignore_regions.get("*", [])) + list(cfg.ignore_regions.get(cam, []))
            if ign:
                m = poly_mask(img.shape[:2], ign)
                img[m > 0] = (img[m > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
            if cfg.auto_mask_osd:
                osd = detect_osd_mask(p, meta, cfg.motion_width, 30)
                if osd is not None:
                    osd = cv2.resize(osd, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
                    img[osd > 0] = (img[osd > 0] * 0.5 + np.array([255, 0, 255]) * 0.5).astype(np.uint8)
            _show(grid_overlay(img), width)
        for p in ps[per_camera:]:
            tot += V.probe(p)["duration"]
    print(f"\nTotal footage: {tot / 3600:.2f} h")


def _store(cfg):
    return Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))


def _video_tag(path, width=480):
    if not path or not os.path.exists(path) or os.path.getsize(path) > 40e6:
        return ""
    with open(path, "rb") as fh:
        b = base64.b64encode(fh.read()).decode()
    return f'<video width="{width}" controls preload="none" src="data:video/mp4;base64,{b}"></video>'


def show_events(cfg: Config, where: str = "level IN ('HIGH','REVIEW')", order: str = "score DESC",
                limit: int = 10, clips: bool = True):
    """Display events (contact sheet, reasons, AI description, clip) inside the notebook."""
    st = _store(cfg)
    rows = st.query(f"SELECT * FROM events WHERE {where} ORDER BY {order} LIMIT {int(limit)}")
    if not rows:
        print("No matching events.")
    for e in rows:
        flags = json.loads(e["flags"] or "[]")
        vlm = json.loads(e["vlm"]) if e["vlm"] else None
        _html(f"<h3 style='margin-bottom:2px'>{e['level']} {e['score']:g} — {e['event_id']}</h3>"
              f"<div>🕒 {e['start_dt']} → {e['end_dt']} ({e['duration']:.0f}s) · 🎞 {e['rel_path']} "
              f"@ <b>{V.fmt_offset(e['start_off'])}</b> · 👥 {e['max_persons']} · objects {e['objects']} · "
              f"IDs {e['identities']}</div>"
              + (f"<div style='background:#eef3ff;padding:4px'>🤖 {vlm.get('description', '')} "
                 f"{'⚠️ ' + str(vlm.get('reason', '')) if vlm.get('suspicious') else ''}</div>" if vlm else "")
              + "<ul>" + "".join(f"<li><b>{f['flag']}</b> +{f['points']:g}: {f['reason']}</li>" for f in flags) + "</ul>")
        img = cv2.imread(e["contact_sheet"]) if e["contact_sheet"] else None
        if img is not None:
            _show(img, 1000)
        if clips and e["clip_path"]:
            _html(_video_tag(e["clip_path"]))
    return rows


_SEM_CACHE: dict = {}


def search_text(cfg: Config, query: str, k: int = 12, show: bool = True):
    """Free-text search over all analysed keyframes (SigLIP-2), e.g.
    'person in a red shirt', 'man with a helmet', 'someone carrying a big bag'."""
    from .detector import resolve_device
    from .semantic import Semantic, search_index
    key = (cfg.semantic_model, tuple(cfg.suspicious_prompts), tuple(cfg.normal_prompts))
    sem = _SEM_CACHE.get(key)
    if sem is None:
        sem = _SEM_CACHE[key] = Semantic(cfg, resolve_device(cfg.device))
    hits = search_index(sem, os.path.join(cfg.output_dir, "semantic_index.npz"), query, k)
    st = _store(cfg)
    results = []
    for hit, score in hits:
        eid, t = hit.split("|")
        e = st.query("SELECT * FROM events WHERE event_id=?", (eid,))
        if not e:
            continue
        e = e[0]
        snaps = json.loads(e["snapshots"] or "[]")
        snap = min(snaps, key=lambda s: abs(s["t_off"] - float(t))) if snaps else None
        results.append({"event_id": eid, "score": score, "time": snap and snap["dt"], "video": e["rel_path"],
                        "offset": V.fmt_offset(float(t)), "snapshot": snap and snap["annotated"]})
    if show:
        for r in results:
            print(f"{r['score']:.3f}  {r['event_id']}  {r['time']}  {r['video']} @ {r['offset']}")
            if r["snapshot"]:
                im = cv2.imread(r["snapshot"])
                if im is not None:
                    _show(im, 640)
    return results


def search_face(cfg: Config, image_path: str, min_sim: Optional[float] = None, show: int = 12):
    """Search-by-photo: find every appearance of the person in `image_path`."""
    from .faces import FaceEngine, match_identities
    fe = FaceEngine(cfg)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    thr = cfg.face_match_threshold - 0.05 if min_sim is None else min_sim
    matches = match_identities(fe, img, os.path.join(cfg.output_dir, "identities"), thr)
    if matches is None:
        print("No face found in the photo.")
        return []
    st = _store(cfg)
    out = []
    for pid, sim in matches:
        rows = st.query("SELECT t.event_id, t.first_dt, t.face_crop, e.rel_path, t.first_off, e.camera FROM tracks t "
                        "JOIN events e USING(event_id) WHERE t.identity=? ORDER BY t.first_dt", (pid,))
        print(f"\n{pid} similarity {sim:.3f} — {len(rows)} appearance(s)")
        for r in rows:
            print(f"   {r['first_dt']}  {r['camera']}  {r['event_id']}  {r['rel_path']} @ {V.fmt_offset(r['first_off'])}")
            out.append({**r, "identity": pid, "similarity": sim})
        if rows and rows[0]["face_crop"] and show:
            im = cv2.imread(rows[0]["face_crop"])
            if im is not None:
                _show(im, 120)
    if not out:
        print("No matching identity found.")
    return out


def summary(cfg: Config):
    st = _store(cfg)
    q = lambda s: st.query(s)[0]
    v = q("SELECT COUNT(*) n, SUM(duration) d, SUM(motion_seconds) m FROM videos WHERE status='done'")
    e = st.query("SELECT level, COUNT(*) n FROM events GROUP BY level")
    print(f"Videos: {v['n']}  footage: {(v['d'] or 0) / 3600:.2f} h  motion: {(v['m'] or 0) / 3600:.2f} h")
    print("Events by level:", {r["level"]: r["n"] for r in e})
    dup = st.query("SELECT rel_path, duplicate_of FROM videos WHERE status='duplicate'")
    if dup:
        print(f"Duplicates skipped: {len(dup)}")
        for d in dup[:20]:
            print("  ", d["rel_path"], "= copy of", d["duplicate_of"])
    nf = st.query("SELECT COUNT(*) n FROM face_alerts WHERE kind='new_face'")[0]["n"]
    ua = st.query("SELECT COUNT(*) n FROM face_alerts WHERE kind='unauthorized'")[0]["n"]
    print(f"New-face alerts (all runs): {nf}   unauthorised-person alerts: {ua}")
    bad = st.query("SELECT rel_path, error FROM videos WHERE status NOT IN ('done','duplicate')")
    if bad:
        print("Problems:")
        for b in bad:
            print("  ", b["rel_path"], (b["error"] or "")[:200])


def _q(v: str) -> str:
    return "'" + str(v).replace("'", "''") + "'"


def show_new_faces(cfg: Config, run_id: Optional[str] = None, limit: int = 60):
    """Faces never seen before (default: in the latest run), with where/when they appeared."""
    st = _store(cfg)
    if run_id is None:
        r = st.query("SELECT run_id FROM face_alerts WHERE kind='new_face' ORDER BY id DESC LIMIT 1")
        run_id = r[0]["run_id"] if r else ""
    rows = st.query("SELECT * FROM face_alerts WHERE kind='new_face' AND run_id=? ORDER BY dt LIMIT ?",
                    (run_id, int(limit)))
    if not rows:
        print("No new faces" + (f" in run {run_id}" if run_id else "") + ".")
        return rows
    print(f"{len(rows)} new face(s) in run {run_id}:")
    tiles = []
    for a in rows:
        im = cv2.imread(a["face_crop"]) if a["face_crop"] else None
        src = ""
        if im is not None:
            ok, buf = cv2.imencode(".jpg", cv2.resize(im, (112, int(112 * im.shape[0] / im.shape[1]))))
            src = "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()
        tiles.append(f"<div style='display:inline-block;width:150px;margin:4px;padding:6px;background:#fff8c4;"
                     f"border-radius:8px;text-align:center;font-size:12px;vertical-align:top'>"
                     f"{'<img src=' + chr(34) + src + chr(34) + ' width=112><br>' if src else ''}<b>{a['identity']}</b>"
                     f"<br>{a['dt'] or ''}<br>{a['camera']}<br>{a['video']} @ {V.fmt_offset(a['t_off'] or 0)}</div>")
    _html("".join(tiles))
    return rows


def show_highlights(cfg: Config, day: Optional[str] = None, play: bool = True):
    """List the daily highlight reels (or one day's reel with its timing table)."""
    st = _store(cfg)
    if day is None:
        rows = st.query("SELECT day, COUNT(*) n, MAX(reel_end) d, reel_path FROM reel_segments GROUP BY day ORDER BY day DESC")
        for r in rows:
            print(f"{r['day']}: {r['n']:4d} movement(s), {V.fmt_offset(r['d'] or 0)}  ->  {r['reel_path']}")
        if not rows:
            print("No highlight reels yet.")
        return rows
    rows = st.query("SELECT r.*, e.camera, e.start_dt, e.level, e.score, e.rel_path, e.start_off FROM reel_segments r "
                    "JOIN events e USING(event_id) WHERE r.day=? ORDER BY r.idx", (day,))
    if not rows:
        print("No reel for", day)
        return rows
    _html("<table><tr><th>#</th><th>Reel time</th><th>Camera</th><th>Clock</th><th>Level</th><th>Video @ offset</th></tr>" +
          "".join(f"<tr><td>{r['idx']}</td><td>{V.fmt_offset(r['reel_start'])}</td><td>{r['camera']}</td>"
                  f"<td>{(r['start_dt'] or '')[11:]}</td><td>{r['level']} {r['score']:g}</td>"
                  f"<td>{r['rel_path']} @ {V.fmt_offset(r['start_off'])}</td></tr>" for r in rows) + "</table>")
    if play:
        _html(_video_tag(rows[0]["reel_path"], 720) or f"Reel: {rows[0]['reel_path']} (too large to embed – open it from Drive)")
    return rows


def review_dashboard(cfg: Config):
    """Interactive filter panel (date range, camera, level, person ID, text, new faces) for the notebook."""
    try:
        import ipywidgets as W
        from IPython.display import clear_output, display
    except Exception:
        print("ipywidgets not available – use show_events(cfg, where=...) instead.")
        return
    st = _store(cfg)
    cams = [r["camera"] for r in st.query("SELECT DISTINCT camera FROM events ORDER BY camera") if r["camera"]]
    days = [r["d"] for r in st.query("SELECT DISTINCT substr(start_dt,1,10) d FROM events WHERE start_dt IS NOT NULL ORDER BY d")]
    import datetime as _dt
    d1 = W.DatePicker(description="From", value=_dt.date.fromisoformat(days[0]) if days else None)
    d2 = W.DatePicker(description="To", value=_dt.date.fromisoformat(days[-1]) if days else None)
    cam = W.Dropdown(options=["(all)"] + cams, description="Camera")
    lvl = W.SelectMultiple(options=["HIGH", "REVIEW", "LOW", "NORMAL"], value=("HIGH", "REVIEW"), description="Level", rows=4)
    pid = W.Text(description="Person ID", placeholder="P0003")
    txt = W.Text(description="Contains", placeholder="flag / object / AI text, e.g. running")
    ppl = W.Checkbox(description="people only", value=False)
    nwf = W.Checkbox(description="new faces only", value=False)
    srt = W.Dropdown(options=[("score", "score DESC"), ("time", "start_dt")], description="Sort")
    lim = W.IntSlider(description="Max", value=10, min=1, max=100)
    clips = W.Checkbox(description="embed clips", value=True)
    btn = W.Button(description="Show events", button_style="primary")
    out = W.Output()

    def go(_=None):
        cond = ["1=1"]
        if d1.value:
            cond.append(f"substr(start_dt,1,10) >= {_q(d1.value.isoformat())}")
        if d2.value:
            cond.append(f"substr(start_dt,1,10) <= {_q(d2.value.isoformat())}")
        if cam.value != "(all)":
            cond.append(f"camera = {_q(cam.value)}")
        if lvl.value:
            cond.append("level IN (" + ",".join(_q(v) for v in lvl.value) + ")")
        if pid.value.strip():
            cond.append(f"identities LIKE {_q('%' + pid.value.strip().upper() + '%')}")
        if txt.value.strip():
            t = _q("%" + txt.value.strip() + "%")
            cond.append(f"(flags LIKE {t} OR objects LIKE {t} OR vlm LIKE {t} OR semantic LIKE {t})")
        if ppl.value:
            cond.append("n_person_tracks > 0")
        if nwf.value:
            cond.append("event_id IN (SELECT event_id FROM face_alerts WHERE kind='new_face')")
        with out:
            clear_output()
            n = st.query("SELECT COUNT(*) n FROM events WHERE " + " AND ".join(cond))[0]["n"]
            print(f"{n} matching event(s); showing up to {lim.value}")
            show_events(cfg, where=" AND ".join(cond), order=srt.value, limit=lim.value, clips=clips.value)

    btn.on_click(go)
    display(W.VBox([W.HBox([d1, d2, cam]), W.HBox([lvl, W.VBox([pid, txt, W.HBox([ppl, nwf])])]),
                    W.HBox([srt, lim, clips, btn]), out]))

In [ ]:
%%writefile /content/cctv_analysis/__main__.py
"""Command line:  python -m cctv_analysis --input /videos --output /results [--config cfg.json] [key=value ...]"""
import argparse
import json

from .config import Config
from .pipeline import run


def main():
    ap = argparse.ArgumentParser(description="CCTV footage analysis")
    ap.add_argument("--input", required=False)
    ap.add_argument("--output", required=False)
    ap.add_argument("--work", default=None)
    ap.add_argument("--config", default=None, help="JSON file with Config fields")
    ap.add_argument("--preset", choices=["fast", "balanced", "accurate"], help="speed preset (applied before overrides)")
    ap.add_argument("--reports-only", action="store_true", help="rebuild reports from the database")
    ap.add_argument("--retime", action="store_true",
                    help="re-read recording times of analysed videos, fix all timestamps, rebuild reels/reports")
    ap.add_argument("overrides", nargs="*", help="field=value (value parsed as JSON when possible)")
    a = ap.parse_args()
    from .config import PRESETS
    d = dict(PRESETS[a.preset]) if a.preset else {}
    if a.config:
        with open(a.config) as fh:
            d.update(json.load(fh))
    for kv in a.overrides:
        k, v = kv.split("=", 1)
        try:
            d[k] = json.loads(v)
        except json.JSONDecodeError:
            d[k] = v
    if a.input:
        d["input_dir"] = a.input
    if a.output:
        d["output_dir"] = a.output
    if a.work:
        d["work_dir"] = a.work
    cfg = Config.from_dict(d)
    if a.retime:
        from .pipeline import setup_logging
        from .retime import retime
        setup_logging(cfg.output_dir)
        print(retime(cfg))
        return
    if a.reports_only:
        from .pipeline import Analyzer
        from .report import build_reports
        an = Analyzer(cfg)
        print(build_reports(an.store, cfg))
        an.store.commit()
        return
    print(run(cfg))


if __name__ == "__main__":
    main()

In [ ]:
import sys, importlib, os
os.makedirs('/content/cctv_analysis', exist_ok=True)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
for m in [k for k in list(sys.modules) if k.startswith('cctv_analysis')]:
    del sys.modules[m]          # pick up edits when re-running
import cctv_analysis
print("cctv_analysis", cctv_analysis.__version__, "ready")

## Step 3 · Settings
Recommended defaults are already filled in. The only must-change items are the two folders.

* **INPUT_DIR** – the Drive folder your DVR/NVR syncs to (sub-folders are scanned; a sub-folder name or `ch01`/`cam2` in the file name becomes the camera name).
* **SPEED_PRESET** – `balanced` is recommended on a T4. `fast` ≈ 2–3× quicker (lighter models, no pose); `accurate` for small faces / far cameras.
* **FOLDER_KEYWORDS** *(optional)* – only scan folders whose name matches, comma-separated, e.g. `6-oct, 7-oct` or `ch01`. Dates match in any common style (`6-oct` also matches `06Oct2026`, `2026-10-06`, `Oct 6`; it never matches `16-oct`). Blank = all folders.
* **Shop hours** *(optional)* – blank = open 24 h (nothing is flagged just for the time of day). Fill in e.g. `09:00` / `21:30` to flag activity outside them as *after hours*.
* **AUTHORIZED_ONLY_CAMERAS** *(optional)* – comma-separated cameras (e.g. `ch03`) where only known faces are allowed; anyone else raises an alert.
* **MODEL_CACHE_DIR** – AI models are saved here on first use and reused by every later run.
* **KNOWN_FACES_DIR** *(optional)* – `known_faces/Owner/*.jpg`, `known_faces/Staff_Ravi/*.jpg` … these people get named instead of P-numbers and lower the score of their events.

In [ ]:
#@title Main settings { run: "auto" }
INPUT_DIR = "/content/drive/MyDrive/CCTV"            #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/CCTV_Analysis"  #@param {type:"string"}
KNOWN_FACES_DIR = "/content/drive/MyDrive/CCTV_known_faces"  #@param {type:"string"}
SPEED_PRESET = "balanced"  #@param ["fast", "balanced", "accurate"]
TIMEZONE = "Asia/Kolkata"  #@param {type:"string"}
FOLDER_KEYWORDS = ""  #@param {type:"string"}
OPEN_HOUR = ""  #@param {type:"string"}
CLOSE_HOUR = ""  #@param {type:"string"}
AUTHORIZED_ONLY_CAMERAS = ""  #@param {type:"string"}
MODEL_CACHE_DIR = "/content/drive/MyDrive/CCTV_models"  #@param {type:"string"}
DAILY_HIGHLIGHT_REELS = True  #@param {type:"boolean"}
REEL_CONTENT = "all"  #@param ["all", "people", "flagged"]
WEEK_STARTS_ON = "sunday"  #@param ["sunday", "monday"]
MOTION_SENSITIVITY = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
USE_FACES = True  #@param {type:"boolean"}
UNIQUE_FACES = True  #@param {type:"boolean"}
READ_NUMBER_PLATES = False  #@param {type:"boolean"}
ENHANCE_HARD_PLATES = False  #@param {type:"boolean"}
PLATE_WATCHLIST = ""  #@param {type:"string"}
TEXT_ON_HIGHLIGHT_VIDEOS = False  #@param {type:"boolean"}
USE_AI_DESCRIPTIONS = True  #@param {type:"boolean"}
AI_MODEL_RUNS_IN = "transformers"  #@param ["transformers", "openai"]
AI_SERVER_URL = ""  #@param {type:"string"}
# AI_SERVER_URL: an OpenAI-compatible server (Ollama http://HOST:11434/v1, LM Studio, vLLM) when AI_MODEL_RUNS_IN = "openai"
AI_DESCRIBE = "flagged"  #@param ["flagged", "people", "all"]
SAVE_CLIPS = "people"  #@param ["all", "people", "flagged", "none"]
ONLY_NEWER_THAN_DAYS = 0  #@param {type:"number"}
MAX_VIDEOS = 0  #@param {type:"integer"}
REPROCESS_ALL = False  #@param {type:"boolean"}
STOP_AFTER_MINUTES = 0  #@param {type:"number"}
# STOP_AFTER_MINUTES: e.g. 600 on free Colab so reports are written before the session limit (0 = no limit)

In [ ]:
#@title Advanced settings (zones, ignore areas, file naming) – optional
# Coordinates are fractions of the frame (0..1); use the grid shown in Step 4 to read them.
# "*" applies to every camera; use the camera name (e.g. "ch01") for one camera.
ZONES = {
    # "*":    {"cash_counter": [[0.60, 0.45], [1.00, 0.45], [1.00, 1.00], [0.60, 1.00]]},
    # "ch02": {"stock_room_door": [[0.05, 0.10], [0.25, 0.10], [0.25, 0.80], [0.05, 0.80]]},
}
IGNORE_REGIONS = {
    # "ch01": [[[0.0, 0.0], [0.35, 0.0], [0.35, 0.07], [0.0, 0.07]]],   # e.g. a TV screen / fan / road
}
CLOSED_WEEKDAYS = []          # e.g. [6] = closed on Sunday (0=Mon … 6=Sun)
FILENAME_TIME_PATTERNS = []   # extra regexes with named groups Y m d H M S if your file names are unusual
TIME_OVERRIDES = {}           # {"file.mp4": "2026-10-11 09:30:00"} for files without any time info
EXTRA = {}                    # any other Config field, e.g. {"loiter_seconds": 120, "crowd_size": 5}

In [ ]:
#@title Build configuration
from cctv_analysis.config import Config, PRESETS
cfg = Config(
    input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, work_dir="/content/cctv_work",
    known_faces_dir=KNOWN_FACES_DIR if os.path.isdir(KNOWN_FACES_DIR) else "",
    timezone=TIMEZONE, open_hour=OPEN_HOUR, close_hour=CLOSE_HOUR, closed_weekdays=CLOSED_WEEKDAYS,
    motion_sensitivity=MOTION_SENSITIVITY, use_faces=USE_FACES, face_identities=UNIQUE_FACES,
    use_plates=READ_NUMBER_PLATES, plate_enhance=ENHANCE_HARD_PLATES,
    plate_watchlist=[k.strip() for k in PLATE_WATCHLIST.split(",") if k.strip()], reel_overlay=TEXT_ON_HIGHLIGHT_VIDEOS,
    use_vlm=USE_AI_DESCRIPTIONS, vlm_scope=AI_DESCRIBE, vlm_provider=AI_MODEL_RUNS_IN, vlm_base_url=AI_SERVER_URL,
    save_clips=SAVE_CLIPS != "none", clip_scope=SAVE_CLIPS if SAVE_CLIPS != "none" else "flagged",
    only_newer_than_days=ONLY_NEWER_THAN_DAYS, max_videos=MAX_VIDEOS, reprocess=REPROCESS_ALL,
    max_runtime_minutes=STOP_AFTER_MINUTES,
    folder_keywords=[k.strip() for k in FOLDER_KEYWORDS.split(",") if k.strip()],
    authorized_only_cameras=[k.strip() for k in AUTHORIZED_ONLY_CAMERAS.split(",") if k.strip()],
    model_cache_dir=MODEL_CACHE_DIR, make_highlight_reel=DAILY_HIGHLIGHT_REELS, reel_scope=REEL_CONTENT,
    week_start=WEEK_STARTS_ON,
    zones=ZONES, ignore_regions=IGNORE_REGIONS, filename_time_patterns=FILENAME_TIME_PATTERNS,
    time_overrides=TIME_OVERRIDES, **PRESETS[SPEED_PRESET])
for k, v in EXTRA.items():
    setattr(cfg, k, v)
assert os.path.isdir(cfg.input_dir), f"INPUT_DIR not found: {cfg.input_dir}"
assert not cfg.validate(), cfg.validate()
print("Config ready →", cfg.output_dir)

## Step 4 · Preview (recommended once)
Shows every camera found, how the recording start time was worked out, and one frame per camera with a **0.1 grid** (to write zone coordinates), your **zones** (red), **ignored areas** (blue) and the **auto-detected DVR clock** (magenta, ignored for motion).
Times come from the file name, else from the **DVR clock printed on the picture** (read by OCR and cross-checked at several points of each file and against a date in the folder name such as `31Oct26`). If a start time says `approx`, add a pattern or override in *Advanced settings*.

In [ ]:
#@title Preview cameras
from cctv_analysis.helpers import preview_cameras
preview_cameras(cfg)

## Step 5 · Run the analysis
Progress is printed per video and saved after every file, so if Colab disconnects just run this cell again — it continues where it stopped.
Typical speed on a T4 (`balanced`): idle footage ≈ 20–60× real-time, busy footage ≈ 2–6× real-time.

In [ ]:
#@title ▶️ Analyse all videos
from cctv_analysis.pipeline import Analyzer
analyzer = Analyzer(cfg)
stats = analyzer.run(build_reports=True)
print(stats)

## Step 6 · Review results

In [ ]:
#@title Summary + top suspicious events (snapshots, reasons, AI description, playable clip)
from cctv_analysis.helpers import summary, show_events
summary(cfg)
TOP_N = 10  #@param {type:"integer"}
_ = show_events(cfg, where="level IN ('HIGH','REVIEW')", order="score DESC", limit=TOP_N)

In [ ]:
#@title 🆕 New faces from the latest run (never seen before)
from cctv_analysis.helpers import show_new_faces
_ = show_new_faces(cfg)

In [ ]:
#@title 🎛️ Review dashboard – filter by date range, camera, level, person, text
from cctv_analysis.helpers import review_dashboard
review_dashboard(cfg)

In [ ]:
#@title 🎬 Daily highlight reels (leave DAY blank to list all days)
DAY = ""  #@param {type:"string"}
from cctv_analysis.helpers import show_highlights
_ = show_highlights(cfg, DAY or None)

In [ ]:
#@title 🕒 Fix recording times of folders analysed with an older version (no re-analysis)
# Re-reads the DVR clock of every analysed video, corrects all times and rebuilds highlight videos/reports.
RUN_RETIME = False  #@param {type:"boolean"}
if RUN_RETIME:
    from cctv_analysis.retime import retime
    print(retime(cfg))

In [ ]:
#@title Browse any events (SQL filter)
# Examples:  "after_hours=1"   "camera='ch01' AND start_dt LIKE '2026-10-11%'"   "identities LIKE '%P0003%'"
#            "objects LIKE '%handbag%'"   "start_dt BETWEEN '2026-10-11 20:00' AND '2026-10-11 23:59'"
WHERE = "n_person_tracks > 0"  #@param {type:"string"}
ORDER = "start_dt"  #@param ["start_dt", "score DESC", "duration DESC", "max_persons DESC"]
LIMIT = 10  #@param {type:"integer"}
_ = show_events(cfg, where=WHERE, order=ORDER, limit=LIMIT)

In [ ]:
#@title 🔎 Free-text search over every snapshot
QUERY = "a person wearing a helmet"  #@param {type:"string"}
from cctv_analysis.helpers import search_text
_ = search_text(cfg, QUERY, k=8)

In [ ]:
#@title 🧑 Search by photo – find every appearance of a person
PHOTO = "/content/drive/MyDrive/suspect.jpg"  #@param {type:"string"}
from cctv_analysis.helpers import search_face
_ = search_face(cfg, PHOTO)

In [ ]:
#@title Show outputs (charts, files)
from IPython.display import Image, display
for p in ["charts/timeline.png", "charts/people_by_hour.png"]:
    f = os.path.join(cfg.output_dir, p)
    if os.path.exists(f):
        display(Image(f, width=1000))
print("Excel (all)  :", os.path.join(cfg.output_dir, "CCTV_Report.xlsx"))
print("Weekly Excel :", os.path.join(cfg.output_dir, "reports"))
print("Unique faces :", os.path.join(cfg.output_dir, "Unique_Faces.xlsx"))
print("HTML report  :", os.path.join(cfg.output_dir, "report.html"), "(download the output folder to open it with images)")
print("Highlights   :", os.path.join(cfg.output_dir, "highlights"))
print("Event folders:", os.path.join(cfg.output_dir, "events"))

In [ ]:
#@title (Optional) Rebuild reports only / download a zip of the reports
from cctv_analysis.report import build_reports
from cctv_analysis.storage import Store
st = Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))
print(build_reports(st, cfg)); st.commit()
MAKE_ZIP = False  #@param {type:"boolean"}
if MAKE_ZIP:
    import shutil
    z = shutil.make_archive("/content/cctv_report", "zip", cfg.output_dir)
    from google.colab import files; files.download(z)

## Tips & troubleshooting
* **Too many events?** Lower `MOTION_SENSITIVITY`, add `IGNORE_REGIONS` for TVs/fans/roads, or set `EXTRA = {"require_person_for_event": True}`.
* **Missed movement?** Raise `MOTION_SENSITIVITY` (0.6–0.7) or `EXTRA = {"min_motion_area": 0.0008}`.
* **Wrong clock time?** Check Step 4. Supported names include Hikvision `ch01_20261011143000`, Dahua `2026-10-11/…/14.30.00-14.45.00.dav`, `NVR_ch2_main_20261011093015_…`, `11-10-2026_08-00-00`. Otherwise the burnt-in DVR clock is read by OCR (RapidOCR); container metadata is used only when it agrees with the date in the folder name; file time is the last resort (marked approx). Already analysed with an older version? Run the *Fix recording times* cell.
* **Same person gets several IDs?** Lower `EXTRA = {"face_match_threshold": 0.38}`; **different people merged?** raise it to 0.48.
* **Only some days?** `FOLDER_KEYWORDS = "6-oct, 7-oct"`. A folder that doesn't match is listed in the log so you can see why.
* **Duplicates:** a file whose content was already analysed (same video copied/renamed) is skipped and listed under *Videos* as `duplicate`.
* **Faster runs:** `SPEED_PRESET="fast"`, `AI_DESCRIBE="flagged"`, `SAVE_CLIPS="flagged"`.
* **`.dav` files** (Dahua) are read natively by ffmpeg; if one fails, export from the DVR as MP4.
* The score is a triage aid: always confirm by watching the cited clip / original video at the given offset.
* Face recognition: use only in line with local privacy law and inform customers that CCTV is in use.